# Radar Individual

Geração otimizada do Radar Financeiro individual em HTML autônomo.


In [ ]:
from traceback import format_exc

try:
    # Gerenciador corporativo local para criação da sessão Spark com suporte a DB2 e Hive
    from src.utils.gerenciador_local_v2 import GerenciadorLocal

    gerenciador_local = GerenciadorLocal(
        nome_sessao='radar-financeiro-v3-dashboard',
        exibir_configuracao=False,
        ativar_logs=True,
    )
    spark = gerenciador_local.criar_sessao_spark(db2=True)
    print('[V3_DASHBOARD] Sessão Spark inicializada com sucesso pelo padrão corporativo.')
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise

### Utilitários Corporativos
Carregamento dos gerenciadores corporativos no kernel local.


In [ ]:
# Carrega conectores e utilitários de ambiente corporativo
%run ./src/utils/gerenciador_spark_v2.ipynb
%run ./src/utils/gerenciador_db2_spark_v2.ipynb

## Parâmetros

Informe o cliente, CPF e a quantidade de ciclos.


In [ ]:
%%spark

import calendar
import datetime
import html
import json
import os
import re
import time
from decimal import Decimal, ROUND_HALF_UP
from datetime import timedelta
from functools import reduce

from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, IntegerType, LongType, ShortType,
    StringType, DateType, TimestampType, DecimalType, ArrayType
)
from pyspark.storagelevel import StorageLevel

# --- Nomes das Tabelas Físicas nas Fontes Corporativas ---
FONTE_TRAN = 'DB2GFP.TRAN_RLZD_INST_PCT'   # Movimentações e transações realizadas
FONTE_CICLO = 'DB2GFP.CT_GRDR_FNCO'        # Ciclo e dia de fechamento do balanço da conta
FONTE_RENDA = 'DB2DFE.REN_AVLD_PF'         # Renda presumida e avaliada da pessoa física (Hive)
FONTE_PERFIL = 'DB2D1D.DVS_GRDR_FNCO_PF'   # Perfil financeiro do cliente (Macro/Micro)
VIEW_RESULTADO = 'vw_radar_financeiro_cliente_mvp' # Nome contratual da view temporária publicada

# --- Parâmetros Técnicos de Execução e Limites ---
FETCHSIZE = 10_000                         # Tamanho de lote para leitura DB2
QUERY_TIMEOUT_SECONDS = 900                # Timeout de 15 minutos para consultas DB2
DIAS_CONTEXTO_RECONCILIACAO = 5            # Janela de contexto temporal (±5 dias corridos)
LIMITE_PAYLOAD_BYTES = 2 * 1024 * 1024     # Limite de segurança de 2 MiB para payload HTML

# --- Determinação das Datas da Janela de Formação do Público ---
# DATA_EXECUCAO é obtida da variável de ambiente HOJE (ou data atual)
DATA_EXECUCAO = datetime.date.fromisoformat(str(obter_variavel_ambiente('HOJE'))[:10])

def recuar_um_mes_calendario(data):
    """Calcula o mesmo dia no mês anterior, ajustando para o último dia caso o mês anterior seja mais curto."""
    total = data.year * 12 + data.month - 2
    ano, mes_zero = divmod(total, 12)
    mes = mes_zero + 1
    return datetime.date(ano, mes, min(data.day, calendar.monthrange(ano, mes)[1]))

# Janela de formação do público: 1 mês calendário fechado anterior a DATA_EXECUCAO
DATA_INICIAL_PUBLICO = recuar_um_mes_calendario(DATA_EXECUCAO)
DATA_FINAL_EXCLUSIVA_PUBLICO = DATA_EXECUCAO
DT_MES_EXEA = DATA_EXECUCAO.replace(day=1)

# Inicializa conector JDBC DB2 corporativo
conector_db2 = criar_conector_db2_spark(env=dict(os.environ))
inicio_execucao = time.perf_counter()

print(f'[RADAR_INDIVIDUAL] DATA_EXECUCAO={DATA_EXECUCAO}')
print(f'[RADAR_INDIVIDUAL] JANELA_PUBLICO={DATA_INICIAL_PUBLICO} <= TS_INCL_TRAN < {DATA_FINAL_EXCLUSIVA_PUBLICO}')

# Parâmetros editáveis
CD_CLI = None
CPF = None
periodo = 1

def normalizar_periodo(valor):
    if isinstance(valor, bool) or not isinstance(valor, int):
        raise TypeError('periodo deve ser um número inteiro entre 1 e 6.')
    if not 1 <= valor <= 6:
        raise ValueError('periodo deve estar entre 1 e 6 ciclos fechados.')
    return valor

periodo = normalizar_periodo(periodo)
print(f'[RADAR_INDIVIDUAL] Quantidade de ciclos fechados: {periodo}')

# Seleção do cliente: código informado, CPF ou amostra aleatória.
if CD_CLI is None:
    if CPF is not None:
        sql_selecao_cliente = f"""
SELECT CD_CLI
FROM {FONTE_TRAN}
WHERE NR_CPF_CNPJ_TITR = {CPF}
  AND CD_EST_TRAN_INST = 0
  AND CD_TIP_PSS = 1
  AND TIMESTAMP(TS_INCL_TRAN) >= TIMESTAMP('{DATA_INICIAL_PUBLICO.isoformat()} 00:00:00')
  AND TIMESTAMP(TS_INCL_TRAN) < TIMESTAMP('{DATA_FINAL_EXCLUSIVA_PUBLICO.isoformat()} 00:00:00')
ORDER BY TS_INCL_TRAN DESC
FETCH FIRST 1 ROW ONLY
"""
        CD_CLI = conector_db2.sql(
            sql_selecao_cliente,
            fetchsize=FETCHSIZE,
            query_timeout=QUERY_TIMEOUT_SECONDS,
        ).first()['CD_CLI']
        print(f'[RADAR_INDIVIDUAL] CPF={CPF} -> CD_CLI={CD_CLI}')
    else:
        from pyspark.sql.functions import rand
        sql_selecao_cliente = f"""
SELECT DISTINCT CD_CLI
FROM {FONTE_TRAN}
WHERE CD_CLI IS NOT NULL
  AND CD_EST_TRAN_INST = 0
  AND CD_TIP_PSS = 1
  AND TIMESTAMP(TS_INCL_TRAN) >= TIMESTAMP('{DATA_INICIAL_PUBLICO.isoformat()} 00:00:00')
  AND TIMESTAMP(TS_INCL_TRAN) < TIMESTAMP('{DATA_FINAL_EXCLUSIVA_PUBLICO.isoformat()} 00:00:00')
"""
        CD_CLI = conector_db2.sql(
            sql_selecao_cliente,
            fetchsize=FETCHSIZE,
            query_timeout=QUERY_TIMEOUT_SECONDS,
        ).orderBy(rand()).first()['CD_CLI']
        print(f'[RADAR_INDIVIDUAL] CD_CLI aleatório selecionado: {CD_CLI}')
# Normalização dos parâmetros antes de interpolá-los nas consultas corporativas
if CD_CLI is None:
    raise RuntimeError('Informe um único CD_CLI inteiro nesta célula.')
if isinstance(CD_CLI, bool):
    raise TypeError('CD_CLI deve ser inteiro, não booleano.')
texto_cd_cli = str(CD_CLI).strip()
if not re.fullmatch(r'[+-]?[0-9]+', texto_cd_cli):
    raise TypeError('CD_CLI deve possuir representação inteira exata.')
CD_CLI = int(texto_cd_cli)
if not (-2147483648 <= CD_CLI <= 2147483647):
    raise ValueError('CD_CLI não cabe no tipo físico INT (INT32).')

print(f'[RADAR_INDIVIDUAL] Cliente selecionado: {CD_CLI}')


## Processamento e geração do dashboard


In [ ]:
%%spark

# Limpeza preventiva de todas as views temporárias do Radar no catálogo Spark
# Garante que execuções anteriores não deixem resíduos que possam afetar o processamento atual
def limpar_views_temporarias_radar():
    prefixos = [
        'vw_q1_', 'vw_q2_', 'vw_q3_', 'vw_q4_', 'vw_q5_', 'vw_mov_',
        'vw_pares_', 'vw_ids_', 'vw_cliente_', 'vw_ciclo_', 'vw_renda_',
        'vw_perfil_', 'vw_conta_', 'vw_agregacoes_', 'vw_orcamento_',
        'vw_percentuais_', 'vw_pontuacoes_', 'vw_tema_', 'vw_resultado_', 'vw_dashboard_',
        VIEW_RESULTADO
    ]
    for obj in spark.catalog.listTables():
        if obj.isTemporary:
            for p in prefixos:
                if obj.name.startswith(p) or obj.name == p:
                    spark.catalog.dropTempView(obj.name)
                    break

limpar_views_temporarias_radar()
print('[RADAR_INDIVIDUAL] Catálogo limpo de views anteriores.')

# Definição dos 12 atributos e das 70 linhas contratuais de mapeamento de categorias
# Chave de casamento no Spark SQL: (CD_CATEGORIA, TIPO)
COLUNAS_CATEGORIAS = [
    'TIPO', 'CD_GRUPO', 'TX_GRUPO', 'CD_CATEGORIA', 'TX_CATEGORIA',
    'CD_IR', 'TX_IR', 'CD_CLASS_RADAR', 'TX_CLASS_RADAR',
    'IN_AGRO', 'IN_PARTICIPA_CALCULO', 'IN_PARTICIPA_ORCAMENTO'
]

LINHAS_CATEGORIAS = [
    (None, 0, 'Sem categoria', 0, 'Sem categoria', 0, 'Não pertence', 0, 'Outras Entradas', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 1, 'Salário', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 2, 'Vale Alimentação', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 3, 'Restituição de IR', 0, 'Não pertence', 2, 'Estorno', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 4, 'Bonificação', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 5, 'Outros Rendimentos', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 6, 'Água', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 7, 'Eletricidade e Gás', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 9, 'Compra de Imóvel', 2, 'Bens e direitos', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 10, 'Aluguel e Condomínio', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 11, 'Móveis e Utensílios', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 12, 'Serviços e Manutenção', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 13, 'Empregados', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 14, 'Animais e Pets', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 15, 'Educação Superior', 1, 'Pagamentos efetuados', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 16, 'Colégio', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 17, 'Idiomas', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 18, 'Publicações e Papelaria', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 20, 'Outros Gastos, Educação', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 21, 'Viagens e Lazer', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 22, 'Esportes e Academia', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 25, 'Cultura e Entretenimento', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 27, 'Plano de Saúde', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 28, 'Serviços de Saúde', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 29, 'Dentista', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 30, 'Farmácias e Drogarias', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 6, 'Alimentação', 32, 'Feira e Supermercado', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 6, 'Alimentação', 35, 'Bar, Rest. e Padaria', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 36, 'Compra de Veículo', 2, 'Bens e direitos', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 37, 'Combustível', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 38, 'Estacionamento e Pedágio', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 39, 'Seguro de Veículo', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 40, 'Serviços e Manutenção', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 41, 'Transporte Urbano e Apps', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 42, 'Vestuário e Acessórios', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 43, 'Cuidado Pessoal e Beleza', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 44, 'Compras Diversas', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 45, 'Pensão Alimentícia', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 46, 'Seguros e Previdência', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 47, 'Doação', 4, 'Doações efetuadas', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 48, 'Gasto com Familiares', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 49, 'Presentes', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 9, 'Comunicação', 51, 'Telefonia e Internet', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 9, 'Comunicação', 53, 'Assinatura TV e Streaming', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 54, 'IPTU', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 55, 'IPVA e Gastos Detran', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 56, 'Imposto de Renda', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 57, 'ISS(Imposto sobre Serviços)', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 58, 'GPS(Guia de Previdência Social)', 0, 'Não pertence', 8, 'Futuro', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 59, 'Serviços Financeiros', 0, 'Não pertence', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 60, 'Serviços Diversos', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 61, 'Jogos e Loterias', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    (None, 0, 'Sem categoria', 83, 'Sem Categoria', 0, 'Não pertence', 0, 'Outras Entradas', 'N', 'S', 'S'),
    ('D', 12, 'Fatura', 111, 'Cartão de Crédito', 0, 'Não pertence', 9, 'Obrigações', 'N', 'N', 'N'),
    ('D', 11, 'Outros', 279, 'Gastos Diversos', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('C', 14, 'Agro', 300, 'Receitas Agro', 0, 'Não pertence', 1, 'Renda', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 310, 'Criações', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 330, 'Cultivos', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 350, 'Insumos', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 370, 'Apoio Produtivo', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 10, 'Tarifas e impostos', 3787, 'IOF', 0, 'Não pertence', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 3788, 'Encargos e Tarifas', 0, 'Não pertence', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 3790, 'Seguro Residencial', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 4417, 'Empréstimos e Prestações', 3, 'Dívidas e ônus reais', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39434, 'Cheque', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39435, 'Saque', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39436, 'Transferência', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39437, 'Boletos Diversos', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 13, 'Investimentos', 448977, 'Aplicação', 0, 'Não pertence', 8, 'Futuro', 'N', 'S', 'N'),
    ('C', 13, 'Investimentos', 448978, 'Resgate de Investimentos', 0, 'Não pertence', 3, 'Resgate', 'N', 'S', 'N'),
]

schema_categorias = StructType([
    StructField('TIPO', StringType(), True),
    StructField('CD_GRUPO', IntegerType(), False),
    StructField('TX_GRUPO', StringType(), False),
    StructField('CD_CATEGORIA', IntegerType(), False),
    StructField('TX_CATEGORIA', StringType(), False),
    StructField('CD_IR', IntegerType(), False),
    StructField('TX_IR', StringType(), False),
    StructField('CD_CLASS_RADAR', IntegerType(), False),
    StructField('TX_CLASS_RADAR', StringType(), False),
    StructField('IN_AGRO', StringType(), False),
    StructField('IN_PARTICIPA_CALCULO', StringType(), False),
    StructField('IN_PARTICIPA_ORCAMENTO', StringType(), False),
])

# Criação e registro da view temporária SQL
df_categorias = spark.createDataFrame(LINHAS_CATEGORIAS, schema_categorias)
df_categorias.createOrReplaceTempView('vw_categorias')

print('[RADAR_INDIVIDUAL] Mapa de categorias registrado.')

inicio_q1 = time.perf_counter()

# Consulta SQL enviada diretamente ao DB2
# Objetivo: identificar se o cliente é elegível na janela de formação e extrair titularidade/contas
sql_q1_db2 = f"""
SELECT
    CD_CLI,
    TS_INCL_TRAN,
    NR_CPF_CNPJ_TITR,
    NR_AG_TITR,
    CD_CT_TITR,
    NR_MCA_PCT_OPB,
    CD_PRD
FROM {FONTE_TRAN}
WHERE CD_CLI = {CD_CLI}
  AND CD_EST_TRAN_INST = 0
  AND CD_TIP_PSS = 1
  AND TIMESTAMP(TS_INCL_TRAN) >= TIMESTAMP('{DATA_INICIAL_PUBLICO.isoformat()} 00:00:00')
  AND TIMESTAMP(TS_INCL_TRAN) < TIMESTAMP('{DATA_FINAL_EXCLUSIVA_PUBLICO.isoformat()} 00:00:00')
"""

df_q1_raw = conector_db2.sql(sql_q1_db2, fetchsize=FETCHSIZE, query_timeout=QUERY_TIMEOUT_SECONDS)
df_q1_raw = df_q1_raw.persist(StorageLevel.MEMORY_AND_DISK)
df_q1_raw.createOrReplaceTempView('vw_q1_cliente_raw')

# 1. Derivação do Timestamp de Referência e Unicidade do CPF
# - TS_INCL_TRAN_REF: maior timestamp de inclusão de transação no período de formação
# - FL_CPF_UNICO = 'S' se e somente se houver exatamente 1 CPF distinto associado às transações
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_cliente_derivado AS
SELECT
    CD_CLI,
    MAX(TS_INCL_TRAN) AS TS_INCL_TRAN_REF,
    CASE WHEN COUNT(DISTINCT NR_CPF_CNPJ_TITR) = 1 THEN 'S' ELSE 'N' END AS FL_CPF_UNICO,
    CASE WHEN COUNT(DISTINCT NR_CPF_CNPJ_TITR) = 1 THEN CAST(MAX(NR_CPF_CNPJ_TITR) AS DECIMAL(14,0)) ELSE NULL END AS CD_CPF
FROM vw_q1_cliente_raw
GROUP BY CD_CLI
""")

# 2. Identificação das Contas Correntes Elegíveis do Cliente
# - NR_MCA_PCT_OPB = 999999999 (pacote operacional de conta corrente padrão)
# - CD_PRD = 6 (código de produto correspondente a conta corrente ativa)
# - Agência e conta preenchidas e não vazias
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_contas_elegiveis_q1 AS
SELECT
    NR_AG_TITR,
    CD_CT_TITR,
    COUNT(1) AS QT_OCORRENCIAS
FROM vw_q1_cliente_raw
WHERE NR_MCA_PCT_OPB = 999999999
  AND CD_PRD = 6
  AND NR_AG_TITR IS NOT NULL
  AND CD_CT_TITR IS NOT NULL
  AND TRIM(CAST(CD_CT_TITR AS STRING)) != ''
GROUP BY NR_AG_TITR, CD_CT_TITR
""")

# 3. Resumo da Conta Elegível
# - FL_CONTA_ELEGIVEL_UNICA = 'S' se houver exatamente 1 conta corrente distinta elegível
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_conta_elegivel_resumo AS
SELECT
    CASE WHEN COUNT(1) = 1 THEN 'S' ELSE 'N' END AS FL_CONTA_ELEGIVEL_UNICA,
    CASE WHEN COUNT(1) = 1 THEN MAX(NR_AG_TITR) ELSE NULL END AS NR_AG_TITR,
    CASE WHEN COUNT(1) = 1 THEN MAX(CD_CT_TITR) ELSE NULL END AS CD_CT_TITR
FROM vw_contas_elegiveis_q1
""")

def criar_view_conta_normalizada(view_origem, view_destino):
    """Normaliza a chave da conta aplicando TRIM antes de validar e converter."""
    spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW {view_destino} AS
    WITH conta_textual AS (
        SELECT
            FL_CONTA_ELEGIVEL_UNICA,
            NR_AG_TITR,
            CD_CT_TITR,
            TRIM(CAST(NR_AG_TITR AS STRING)) AS NR_AG_TITR_TXT,
            TRIM(CAST(CD_CT_TITR AS STRING)) AS CD_CT_TITR_TXT
        FROM {view_origem}
    ), conta_significativa AS (
        SELECT
            *,
            LTRIM('0', CD_CT_TITR_TXT) AS NR_CC_SIGNIFICATIVA
        FROM conta_textual
    )
    SELECT
        FL_CONTA_ELEGIVEL_UNICA,
        NR_AG_TITR,
        CD_CT_TITR,
        CASE
            WHEN FL_CONTA_ELEGIVEL_UNICA = 'S'
             AND NR_AG_TITR_TXT RLIKE '^[0-9]+$'
             AND CD_CT_TITR_TXT RLIKE '^[0-9]+$'
             AND CAST(NR_AG_TITR_TXT AS BIGINT) BETWEEN -2147483648 AND 2147483647
             AND LENGTH(NR_CC_SIGNIFICATIVA) <= 11
            THEN CAST(NR_AG_TITR_TXT AS INT)
            ELSE NULL
        END AS CD_UOR_CC_NORM,
        CASE
            WHEN FL_CONTA_ELEGIVEL_UNICA = 'S'
             AND NR_AG_TITR_TXT RLIKE '^[0-9]+$'
             AND CD_CT_TITR_TXT RLIKE '^[0-9]+$'
             AND CAST(NR_AG_TITR_TXT AS BIGINT) BETWEEN -2147483648 AND 2147483647
             AND LENGTH(NR_CC_SIGNIFICATIVA) <= 11
            THEN CAST(
                CASE
                    WHEN NR_CC_SIGNIFICATIVA = '' THEN '0'
                    ELSE NR_CC_SIGNIFICATIVA
                END AS DECIMAL(11,0)
            )
            ELSE NULL
        END AS NR_CC_NORM
    FROM conta_significativa
    """)


criar_view_conta_normalizada(
    'vw_conta_elegivel_resumo',
    'vw_conta_normalizada',
)

estado_cliente = spark.sql("""
SELECT c.*, n.*
FROM vw_cliente_derivado c
CROSS JOIN vw_conta_normalizada n
""").first()
if estado_cliente is None:
    raise RuntimeError(
        f'O cliente {CD_CLI} não pertence à janela de formação do público '
        f'({DATA_INICIAL_PUBLICO} a {DATA_FINAL_EXCLUSIVA_PUBLICO}).'
    )

cta_norm_row = estado_cliente
res_cta = estado_cliente
cpf_row = estado_cliente
tem_conta_norm = (
    cta_norm_row['CD_UOR_CC_NORM'] is not None
    and cta_norm_row['NR_CC_NORM'] is not None
)
df_q1_raw.unpersist()
tempo_q1 = time.perf_counter() - inicio_q1
print(
    f'[RADAR_INDIVIDUAL] Q1 concluída em {tempo_q1:.3f}s; '
    f'CPF_UNICO={estado_cliente["FL_CPF_UNICO"]}; '
    f'CONTA_UNICA={estado_cliente["FL_CONTA_ELEGIVEL_UNICA"]}.'
)
print(
    f'[RADAR_INDIVIDUAL] Conta normalizada: '
    f'UOR={cta_norm_row["CD_UOR_CC_NORM"]}, NR_CC={cta_norm_row["NR_CC_NORM"]}.'
)

inicio_q2 = time.perf_counter()

# Executa Q2 no DB2 somente se houver uma conta corrente única devidamente normalizada
if not tem_conta_norm:
    schema_ciclo_raw = StructType([
        StructField('CD_UOR_CC', IntegerType(), True),
        StructField('NR_CC', DecimalType(11, 0), True),
        StructField('DD_INC_MM_CLC_BLC', ShortType(), True),
        StructField('TS_ULT_EXEA_PSQ', TimestampType(), True),
    ])
    df_q2_raw = spark.createDataFrame([], schema_ciclo_raw)
    df_q2_raw.createOrReplaceTempView('vw_q2_ciclo_raw')
    print('[RADAR_INDIVIDUAL] Q2: SKIPPED (conta normalizada indisponível).')
else:
    uor_val = cta_norm_row['CD_UOR_CC_NORM']
    nr_val = cta_norm_row['NR_CC_NORM']
    sql_q2_db2 = f"""
SELECT
    CD_UOR_CC,
    NR_CC,
    DD_INC_MM_CLC_BLC,
    TS_ULT_EXEA_PSQ
FROM {FONTE_CICLO}
WHERE CD_UOR_CC = {uor_val}
  AND NR_CC = {nr_val}
"""
    df_q2_raw = conector_db2.sql(sql_q2_db2, fetchsize=FETCHSIZE, query_timeout=QUERY_TIMEOUT_SECONDS)
    df_q2_raw.createOrReplaceTempView('vw_q2_ciclo_raw')
    print('[RADAR_INDIVIDUAL] Q2 consultada no DB2.')

# 1. Seleção do Ciclo Mais Recente via Window Function em SQL
# Desempate estrito por TS_ULT_EXEA_PSQ DESC
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_ciclo_selecionado AS
WITH rankeado AS (
    SELECT
        CD_UOR_CC,
        NR_CC,
        DD_INC_MM_CLC_BLC,
        TS_ULT_EXEA_PSQ,
        ROW_NUMBER() OVER (PARTITION BY CD_UOR_CC, NR_CC ORDER BY TS_ULT_EXEA_PSQ DESC) AS RN
    FROM vw_q2_ciclo_raw
)
SELECT
    TS_ULT_EXEA_PSQ AS TS_DD_INC_MM_CLC_BLC_REF,
    CAST(DD_INC_MM_CLC_BLC AS SMALLINT) AS DD_INC_MM_CLC_BLC
FROM rankeado
WHERE RN = 1
""")

# 2. Resolução do Dia de Ciclo com Fallback
# - Se o cliente não possui conta única: DD_INC_MM_CLC_BLC_FALLBACK = NULL
# - Se possui conta única mas não há registro na fonte Q2: assume Fallback = 1
# - Se há registro na fonte: assume o valor do dia retornado
row_cli_ts = estado_cliente['TS_INCL_TRAN_REF']
row_ciclo = None if not tem_conta_norm else spark.sql('SELECT * FROM vw_ciclo_selecionado').first()

dd_ciclo_val = row_ciclo['DD_INC_MM_CLC_BLC'] if row_ciclo else None
if not tem_conta_norm:
    dd_fallback_val = None
elif dd_ciclo_val is None:
    dd_fallback_val = 1
else:
    dd_fallback_val = int(dd_ciclo_val)

# 3. Cálculo da Janela Financeira Fechada [DT_REF_INI, DT_REF_FIM]
# DT_REF_FIM permanece no último dia anterior ao ciclo aberto atual.
# DT_REF_INI recua a quantidade de ciclos fechados definida em periodo.
def calcular_inicio_periodo_fechado(inicio_ciclo_aberto, dia_ciclo, quantidade_ciclos):
    mes_alvo_total = inicio_ciclo_aberto.year * 12 + inicio_ciclo_aberto.month - 1 - quantidade_ciclos
    ano_alvo, mes_alvo_zero = divmod(mes_alvo_total, 12)
    mes_alvo = mes_alvo_zero + 1
    dia_alvo = min(dia_ciclo, calendar.monthrange(ano_alvo, mes_alvo)[1])
    return datetime.date(ano_alvo, mes_alvo, dia_alvo)

if dd_fallback_val is None:
    dt_ref_ini_val = None
    dt_ref_fim_val = None
else:
    if not (1 <= dd_fallback_val <= 31):
        raise RuntimeError(f'Dia de ciclo fora do domínio 1..31: {dd_fallback_val}.')
    ts_ref = row_cli_ts
    dia_mes_ref = min(dd_fallback_val, calendar.monthrange(ts_ref.year, ts_ref.month)[1])
    candidato = datetime.datetime(ts_ref.year, ts_ref.month, dia_mes_ref)
    if ts_ref >= candidato:
        inicio_aberto = candidato.date()
    else:
        total = ts_ref.year * 12 + ts_ref.month - 2
        ano_ant, mes_ant_zero = divmod(total, 12)
        mes_ant = mes_ant_zero + 1
        inicio_aberto = datetime.date(ano_ant, mes_ant, min(dd_fallback_val, calendar.monthrange(ano_ant, mes_ant)[1]))
    dt_ref_fim_val = inicio_aberto - timedelta(days=1)
    dt_ref_ini_val = calcular_inicio_periodo_fechado(inicio_aberto, dd_fallback_val, periodo)

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW vw_ciclo_janela AS
SELECT
    CAST('{row_ciclo['TS_DD_INC_MM_CLC_BLC_REF']}' AS TIMESTAMP) AS TS_DD_INC_MM_CLC_BLC_REF,
    CAST({('NULL' if dd_ciclo_val is None else dd_ciclo_val)} AS SMALLINT) AS DD_INC_MM_CLC_BLC,
    CAST({('NULL' if dd_fallback_val is None else dd_fallback_val)} AS SMALLINT) AS DD_INC_MM_CLC_BLC_FALLBACK,
    CAST({('NULL' if dt_ref_ini_val is None else f"'{dt_ref_ini_val.isoformat()}'")} AS DATE) AS DT_REF_INI,
    CAST({('NULL' if dt_ref_fim_val is None else f"'{dt_ref_fim_val.isoformat()}'")} AS DATE) AS DT_REF_FIM
""" if row_ciclo else f"""
CREATE OR REPLACE TEMPORARY VIEW vw_ciclo_janela AS
SELECT
    CAST(NULL AS TIMESTAMP) AS TS_DD_INC_MM_CLC_BLC_REF,
    CAST(NULL AS SMALLINT) AS DD_INC_MM_CLC_BLC,
    CAST({('NULL' if dd_fallback_val is None else dd_fallback_val)} AS SMALLINT) AS DD_INC_MM_CLC_BLC_FALLBACK,
    CAST({('NULL' if dt_ref_ini_val is None else f"'{dt_ref_ini_val.isoformat()}'")} AS DATE) AS DT_REF_INI,
    CAST({('NULL' if dt_ref_fim_val is None else f"'{dt_ref_fim_val.isoformat()}'")} AS DATE) AS DT_REF_FIM
""")

tempo_q2 = time.perf_counter() - inicio_q2
print(
    f'[RADAR_INDIVIDUAL] Q2 concluída em {tempo_q2:.3f}s; '
    f'janela={dt_ref_ini_val} .. {dt_ref_fim_val}.'
)

inicio_q3 = time.perf_counter()

# Executa Q3 no Hive somente se o CPF for estritamente único
if cpf_row['FL_CPF_UNICO'] != 'S' or cpf_row['CD_CPF'] is None:
    schema_q3 = StructType([
        StructField('NR_CPF', DecimalType(11, 0), True),
        StructField('DT_INCL_REN_AVLD', DateType(), True),
        StructField('VL_REN', DecimalType(17, 2), True),
    ])
    df_q3_raw = spark.createDataFrame([], schema_q3)
    df_q3_raw.createOrReplaceTempView('vw_q3_renda_raw')
    print('[RADAR_INDIVIDUAL] Q3: SKIPPED (CPF único indisponível).')
else:
    cpf_num = int(cpf_row['CD_CPF'])
    sql_q3_hive = f"""
SELECT
    NR_CPF_BASE_SRF AS NR_CPF,
    DT_INCL_REN_AVLD,
    VL_REN
FROM {FONTE_RENDA}
WHERE NR_CPF_BASE_SRF = {cpf_num}
"""
    df_q3_raw = spark.sql(sql_q3_hive)
    df_q3_raw.createOrReplaceTempView('vw_q3_renda_raw')
    print('[RADAR_INDIVIDUAL] Q3 consultada no Hive.')

# Seleção da renda mais recente associada ao CPF único
# Ordenação estrita por DT_INCL_REN_AVLD DESC
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW vw_renda_derivada AS
WITH rankeado AS (
    SELECT
        CAST(DT_INCL_REN_AVLD AS DATE) AS DT_REN_PRES_REF,
        CAST(VL_REN * {periodo} AS DECIMAL(17,2)) AS VL_REN_PRES,
        ROW_NUMBER() OVER (PARTITION BY NR_CPF ORDER BY DT_INCL_REN_AVLD DESC) AS RN
    FROM vw_q3_renda_raw
)
SELECT
    DT_REN_PRES_REF,
    VL_REN_PRES
FROM rankeado
WHERE RN = 1
""")

res_renda = spark.sql('SELECT * FROM vw_renda_derivada').first()
if res_renda is None:
    spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW vw_renda_derivada AS
    SELECT CAST(NULL AS DATE) AS DT_REN_PRES_REF, CAST(NULL AS DECIMAL(17,2)) AS VL_REN_PRES
    """)
    res_renda = Row(DT_REN_PRES_REF=None, VL_REN_PRES=None)

tempo_q3 = time.perf_counter() - inicio_q3
print(f'[RADAR_INDIVIDUAL] Q3 concluída em {tempo_q3:.3f}s; renda_ref={res_renda["DT_REN_PRES_REF"]}.')

inicio_q4 = time.perf_counter()

# Consulta SQL no DB2 para o perfil do cliente
# Filtro de corte temporal: DT_REF <= DATA_EXECUCAO (garante estabilidade temporal)
sql_q4_db2 = f"""
SELECT
    CD_CLI,
    DT_REF,
    CD_MAC_PRFL_CLI,
    NM_MAC_PRFL_CLI,
    CD_MIC_PRFL_CLI,
    NM_MIC_PRFL_CLI
FROM {FONTE_PERFIL}
WHERE CD_CLI = {CD_CLI}
  AND DT_REF <= DATE('{DATA_EXECUCAO.isoformat()}')
"""

df_q4_raw = conector_db2.sql(sql_q4_db2, fetchsize=FETCHSIZE, query_timeout=QUERY_TIMEOUT_SECONDS)
df_q4_raw.createOrReplaceTempView('vw_q4_perfil_raw')

# Seleção do perfil do cliente na data mais recente elegível (MAX(DT_REF) <= DATA_EXECUCAO)
# Validação: se houver mais de um perfil na mesma data máxima, bloqueia por ambiguidade
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW vw_perfil_derivado AS
WITH max_data AS (
    SELECT MAX(DT_REF) AS MAX_DT_REF FROM vw_q4_perfil_raw
),
elegiveis AS (
    SELECT p.*
    FROM vw_q4_perfil_raw p
    INNER JOIN max_data m ON p.DT_REF = m.MAX_DT_REF
)
SELECT
    DT_REF AS DT_REF_PRFL,
    CAST(CD_MAC_PRFL_CLI AS INT) AS CD_MAC_PRFL_CLI,
    NM_MAC_PRFL_CLI,
    CAST(CD_MIC_PRFL_CLI AS INT) AS CD_MIC_PRFL_CLI,
    NM_MIC_PRFL_CLI
FROM elegiveis
""")

linhas_perfil = spark.sql('SELECT * FROM vw_perfil_derivado').collect()
if len(linhas_perfil) > 1:
    raise RuntimeError('Q4 retornou mais de uma linha na maior DT_REF elegível.')
if len(linhas_perfil) == 1 and linhas_perfil[0]['DT_REF_PRFL'] > DATA_EXECUCAO:
    raise RuntimeError(f'DT_REF_PRFL posterior à DATA_EXECUCAO: {linhas_perfil[0]["DT_REF_PRFL"]} > {DATA_EXECUCAO}.')

if len(linhas_perfil) == 0:
    spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW vw_perfil_derivado AS
    SELECT
        CAST(NULL AS DATE) AS DT_REF_PRFL,
        CAST(NULL AS INT) AS CD_MAC_PRFL_CLI,
        CAST(NULL AS STRING) AS NM_MAC_PRFL_CLI,
        CAST(NULL AS INT) AS CD_MIC_PRFL_CLI,
        CAST(NULL AS STRING) AS NM_MIC_PRFL_CLI
    """)
    res_prfl = Row(
        DT_REF_PRFL=None, CD_MAC_PRFL_CLI=None, NM_MAC_PRFL_CLI=None,
        CD_MIC_PRFL_CLI=None, NM_MIC_PRFL_CLI=None,
    )
else:
    res_prfl = linhas_perfil[0]

tempo_q4 = time.perf_counter() - inicio_q4
print(f'[RADAR_INDIVIDUAL] Q4 concluída em {tempo_q4:.3f}s; perfil={res_prfl["CD_MAC_PRFL_CLI"]}.')

inicio_q5 = time.perf_counter()

dt_ini_j = dt_ref_ini_val
dt_fim_j = dt_ref_fim_val

# Projeção funcional de sete colunas consumida pelo motor financeiro.
COLUNAS_Q5 = [
    'NR_TRAN_INST_PCT',
    'CD_CLI',
    'DT_TRAN',
    'CD_NTZ_CTB_TRAN',
    'CD_CTGR_TRAN_OGNL',
    'CD_TIP_MOE_CRR',
    'VL_TRAN',
]

SCHEMA_Q5_FUNCIONAL = StructType([
    StructField('NR_TRAN_INST_PCT', LongType(), True),
    StructField('CD_CLI', IntegerType(), True),
    StructField('DT_TRAN', DateType(), True),
    StructField('CD_NTZ_CTB_TRAN', StringType(), True),
    StructField('CD_CTGR_TRAN_OGNL', IntegerType(), True),
    StructField('CD_TIP_MOE_CRR', StringType(), True),
    StructField('VL_TRAN', DecimalType(15, 2), True),
])

# As duas colunas adicionais existem somente para a apresentação.
if dt_ini_j is None:
    schema_q5_apresentacao = StructType(
        list(SCHEMA_Q5_FUNCIONAL.fields) + [
            StructField('TX_DCR_TRAN_OGNL', StringType(), True),
            StructField('NR_MCA_PCT_OPB', StringType(), True),
        ]
    )
    df_q5_contexto_apresentacao = spark.createDataFrame([], schema_q5_apresentacao)
    qt_q5 = 0
    print('[RADAR_INDIVIDUAL] Q5: SKIPPED (janela financeira indisponível).')
else:
    dt_q5_ini = (dt_ini_j - timedelta(days=DIAS_CONTEXTO_RECONCILIACAO)).isoformat()
    dt_q5_fim = (dt_fim_j + timedelta(days=DIAS_CONTEXTO_RECONCILIACAO)).isoformat()
    sql_q5_db2 = f"""
SELECT
    NR_TRAN_INST_PCT,
    CD_CLI,
    DT_TRAN,
    CD_NTZ_CTB_TRAN,
    CD_CTGR_TRAN_OGNL,
    CD_TIP_MOE_CRR,
    VL_TRAN,
    TX_DCR_TRAN_OGNL,
    NR_MCA_PCT_OPB
FROM {FONTE_TRAN}
WHERE CD_CLI = {CD_CLI}
  AND CD_EST_TRAN_INST = 0
  AND DT_TRAN >= DATE('{dt_q5_ini}')
  AND DT_TRAN <= DATE('{dt_q5_fim}')
  AND (
      CD_NTZ_CTB_TRAN = 'C'
      OR (CD_NTZ_CTB_TRAN = 'D' AND IN_VSLO_CSM = 'S')
  )
"""
    df_q5_contexto_apresentacao = conector_db2.sql(
        sql_q5_db2,
        fetchsize=FETCHSIZE,
        query_timeout=QUERY_TIMEOUT_SECONDS,
    ).persist(StorageLevel.MEMORY_AND_DISK)
    qt_q5 = df_q5_contexto_apresentacao.count()
    print(f'[RADAR_INDIVIDUAL] Q5 executada no DB2: {qt_q5} registros.')

df_q5_contexto_apresentacao.createOrReplaceTempView('vw_q5_mov_contexto_apresentacao')

# O motor recebe somente a projeção funcional.
df_q5_contexto = df_q5_contexto_apresentacao.select(*COLUNAS_Q5)
df_q5_contexto.createOrReplaceTempView('vw_q5_mov_contexto')

tempo_q5 = time.perf_counter() - inicio_q5
print(f'[RADAR_INDIVIDUAL] Q5 concluída em {tempo_q5:.3f}s; linhas={qt_q5}.')

# 1. Marcação da Flag IN_JANELA
# - 'S': transação realizada dentro da janela oficial [DT_REF_INI, DT_REF_FIM]
# - 'N': transação realizada na borda temporal externa (contexto ±5 dias)
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW vw_mov_marcado AS
SELECT
    NR_TRAN_INST_PCT,
    CD_CLI,
    DT_TRAN,
    CD_NTZ_CTB_TRAN,
    CD_CTGR_TRAN_OGNL,
    CD_TIP_MOE_CRR,
    VL_TRAN,
    CASE 
        WHEN DT_TRAN >= DATE('{dt_ini_j.isoformat() if dt_ini_j else '1900-01-01'}') 
         AND DT_TRAN <= DATE('{dt_fim_j.isoformat() if dt_fim_j else '1900-01-01'}') 
        THEN 'S' 
        ELSE 'N' 
    END AS IN_JANELA
FROM vw_q5_mov_contexto
""")

# 2. Universo Oficial Inicial (vw_mov_raw)
# Contém todas as transações oficiais antes de qualquer anulação por reconciliação
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_mov_raw AS
SELECT * FROM vw_mov_marcado WHERE IN_JANELA = 'S'
""")

# 1. Identificação dos Pares Exatos Candidatos
# Anula créditos e débitos idênticos na mesma data: número de pares = LEAST(QT_C, QT_D)
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_pares_exatos_candidatos AS
SELECT
    CD_CLI,
    DT_TRAN,
    VL_TRAN,
    CD_TIP_MOE_CRR,
    IN_JANELA,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 1 ELSE 0 END) AS QT_C,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' THEN 1 ELSE 0 END) AS QT_D,
    LEAST(
        SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 1 ELSE 0 END),
        SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' THEN 1 ELSE 0 END)
    ) AS QT_PARES_EXATOS
FROM vw_mov_marcado
WHERE NR_TRAN_INST_PCT IS NOT NULL
  AND CD_NTZ_CTB_TRAN IN ('C', 'D')
  AND DT_TRAN IS NOT NULL
  AND VL_TRAN IS NOT NULL
  AND CD_TIP_MOE_CRR IS NOT NULL
GROUP BY CD_CLI, DT_TRAN, VL_TRAN, CD_TIP_MOE_CRR, IN_JANELA
HAVING QT_PARES_EXATOS > 0
""")

# 2. Seleção dos IDs Consumidos por Par Exato
# Consome os primeiros IDs em ordem crescente (NR_TRAN_INST_PCT ASC) para cada natureza
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_ids_consumidos_exatos AS
WITH rankeado AS (
    SELECT
        m.NR_TRAN_INST_PCT,
        m.IN_JANELA,
        m.CD_NTZ_CTB_TRAN,
        p.QT_PARES_EXATOS,
        ROW_NUMBER() OVER (
            PARTITION BY m.CD_CLI, m.DT_TRAN, m.VL_TRAN, m.CD_TIP_MOE_CRR, m.IN_JANELA, m.CD_NTZ_CTB_TRAN
            ORDER BY m.NR_TRAN_INST_PCT ASC
        ) AS RN
    FROM vw_mov_marcado m
    INNER JOIN vw_pares_exatos_candidatos p
       ON m.CD_CLI = p.CD_CLI
      AND m.DT_TRAN = p.DT_TRAN
      AND m.VL_TRAN = p.VL_TRAN
      AND m.CD_TIP_MOE_CRR = p.CD_TIP_MOE_CRR
      AND m.IN_JANELA = p.IN_JANELA
    WHERE m.CD_NTZ_CTB_TRAN IN ('C', 'D')
)
SELECT
    NR_TRAN_INST_PCT,
    CASE WHEN IN_JANELA = 'S' THEN 'EXATO_OFICIAL' ELSE 'EXATO_CONTEXTO' END AS TIPO_CONSUMO
FROM rankeado
WHERE RN <= QT_PARES_EXATOS
""")

# 3. Universo Residual
# Transações não consumidas por par exato que avançam para a reconciliação de bordas temporais
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_mov_contexto_residual AS
SELECT m.*
FROM vw_mov_marcado m
LEFT ANTI JOIN vw_ids_consumidos_exatos e ON m.NR_TRAN_INST_PCT = e.NR_TRAN_INST_PCT
""")

# UDF SQL para o matching determinístico de bordas temporais (±5 dias)
# Critérios de Otimização Contratuais:
# 1. Maximizar quantidade total de pares (cardinalidade)
# 2. Minimizar distância absoluta total em dias (proximidade)
# 3. Desempatar deterministicamente por IDs ascendentes
SCHEMA_PAR_BORDA = ArrayType(StructType([
    StructField('NR_TRAN_DENTRO', LongType(), False),
    StructField('NR_TRAN_FORA', LongType(), False),
    StructField('DT_TRAN_DENTRO', DateType(), False),
    StructField('DT_TRAN_FORA', DateType(), False),
    StructField('DIF_DIAS', IntegerType(), False),
]))

def parear_listas_residuais_sql_impl(lista_dentro, lista_fora):
    dentro = sorted(
        [(r['DT_TRAN'], int(r['NR_TRAN_INST_PCT'])) for r in (lista_dentro or [])],
        key=lambda item: (item[0], item[1])
    )
    fora = sorted(
        [(r['DT_TRAN'], int(r['NR_TRAN_INST_PCT'])) for r in (lista_fora or [])],
        key=lambda item: (item[0], item[1])
    )
    n = len(dentro)
    m = len(fora)
    vazio = (0, 0, ())
    dp = [[vazio for _ in range(m + 1)] for _ in range(n + 1)]

    def chave_solucao(solucao):
        quantidade, custo, pares = solucao
        assinatura_ids = tuple((par[0], par[1]) for par in pares)
        return (-quantidade, custo, assinatura_ids)

    for i in range(n - 1, -1, -1):
        for j in range(m - 1, -1, -1):
            candidatos = [dp[i + 1][j], dp[i][j + 1]]
            dt_dentro, id_dentro = dentro[i]
            dt_fora, id_fora = fora[j]
            dif_dias = abs((dt_dentro - dt_fora).days)
            if 1 <= dif_dias <= DIAS_CONTEXTO_RECONCILIACAO:
                quantidade, custo, pares = dp[i + 1][j + 1]
                par = (id_dentro, id_fora, dt_dentro, dt_fora, dif_dias)
                candidatos.append((quantidade + 1, custo + dif_dias, (par,) + pares))
            dp[i][j] = min(candidatos, key=chave_solucao)

    return list(dp[0][0][2])

spark.udf.register('parear_borda_udf', parear_listas_residuais_sql_impl, SCHEMA_PAR_BORDA)

# 1. Agrupamento das listas de dentro e fora da janela com valores e moedas iguais
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_pares_borda_calculados AS
WITH dentro_agg AS (
    SELECT
        CD_CLI,
        VL_TRAN,
        CD_TIP_MOE_CRR,
        CD_NTZ_CTB_TRAN AS NTZ_DENTRO,
        COLLECT_LIST(NAMED_STRUCT('DT_TRAN', DT_TRAN, 'NR_TRAN_INST_PCT', NR_TRAN_INST_PCT)) AS LISTA_DENTRO
    FROM vw_mov_contexto_residual
    WHERE IN_JANELA = 'S'
      AND CD_NTZ_CTB_TRAN IN ('C', 'D')
    GROUP BY CD_CLI, VL_TRAN, CD_TIP_MOE_CRR, CD_NTZ_CTB_TRAN
),
fora_agg AS (
    SELECT
        CD_CLI,
        VL_TRAN,
        CD_TIP_MOE_CRR,
        CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 'D' ELSE 'C' END AS NTZ_DENTRO,
        COLLECT_LIST(NAMED_STRUCT('DT_TRAN', DT_TRAN, 'NR_TRAN_INST_PCT', NR_TRAN_INST_PCT)) AS LISTA_FORA
    FROM vw_mov_contexto_residual
    WHERE IN_JANELA = 'N'
      AND CD_NTZ_CTB_TRAN IN ('C', 'D')
    GROUP BY CD_CLI, VL_TRAN, CD_TIP_MOE_CRR, CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 'D' ELSE 'C' END
),
pares_array AS (
    SELECT
        d.CD_CLI,
        d.VL_TRAN,
        d.CD_TIP_MOE_CRR,
        EXPLODE(parear_borda_udf(d.LISTA_DENTRO, f.LISTA_FORA)) AS PAR
    FROM dentro_agg d
    INNER JOIN fora_agg f
       ON d.CD_CLI = f.CD_CLI
      AND d.VL_TRAN = f.VL_TRAN
      AND d.CD_TIP_MOE_CRR = f.CD_TIP_MOE_CRR
      AND d.NTZ_DENTRO = f.NTZ_DENTRO
)
SELECT
    PAR.NR_TRAN_DENTRO AS NR_TRAN_DENTRO,
    PAR.NR_TRAN_FORA AS NR_TRAN_FORA,
    PAR.DT_TRAN_DENTRO AS DT_TRAN_DENTRO,
    PAR.DT_TRAN_FORA AS DT_TRAN_FORA,
    PAR.DIF_DIAS AS DIF_DIAS
FROM pares_array
""")

# 2. Identificação dos IDs Consumidos por Borda Temporal
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_ids_consumidos_borda AS
SELECT NR_TRAN_DENTRO AS NR_TRAN_INST_PCT, 'BORDA_OFICIAL' AS TIPO_CONSUMO FROM vw_pares_borda_calculados
UNION ALL
SELECT NR_TRAN_FORA AS NR_TRAN_INST_PCT, 'BORDA_CONTEXTO' AS TIPO_CONSUMO FROM vw_pares_borda_calculados
""")

# União de todos os IDs consumidos (Pares Exatos + Bordas)
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_ids_consumidos_todos AS
SELECT * FROM vw_ids_consumidos_exatos
UNION ALL
SELECT * FROM vw_ids_consumidos_borda
""")

# IDs oficiais que devem ser expurgados da janela oficial
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_ids_removidos_oficiais AS
SELECT * FROM vw_ids_consumidos_todos WHERE TIPO_CONSUMO IN ('EXATO_OFICIAL', 'BORDA_OFICIAL')
""")

# Construção do Universo Efetivo (vw_mov_efetivo):
# Remove as transações anuladas via LEFT ANTI JOIN
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_mov_efetivo AS
SELECT
    m.CD_CLI,
    m.DT_TRAN,
    m.CD_NTZ_CTB_TRAN,
    m.CD_CTGR_TRAN_OGNL,
    m.CD_TIP_MOE_CRR,
    m.VL_TRAN
FROM vw_mov_raw m
LEFT ANTI JOIN vw_ids_removidos_oficiais r ON m.NR_TRAN_INST_PCT = r.NR_TRAN_INST_PCT
""")

# 1. Classificação das Transações Efetivas com o Mapa CATEGORIAS
# LEFT JOIN pela chave composta (CD_CTGR_TRAN_OGNL == CD_CATEGORIA AND CD_NTZ_CTB_TRAN == TIPO)
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_mov_classificado AS
SELECT
    m.CD_CLI,
    m.DT_TRAN,
    m.CD_NTZ_CTB_TRAN,
    m.CD_CTGR_TRAN_OGNL,
    m.CD_TIP_MOE_CRR,
    m.VL_TRAN,
    c.CD_GRUPO,
    c.TX_GRUPO,
    COALESCE(c.TX_CATEGORIA, 'Sem Categoria') AS TX_CATEGORIA,
    c.CD_IR,
    c.TX_IR,
    COALESCE(c.CD_CLASS_RADAR, 0) AS CD_CLASS_RADAR,
    COALESCE(c.TX_CLASS_RADAR, 'Outras Entradas') AS TX_CLASS_RADAR,
    COALESCE(c.IN_AGRO, 'N') AS IN_AGRO,
    COALESCE(c.IN_PARTICIPA_CALCULO, 'N') AS IN_PARTICIPA_CALCULO,
    COALESCE(c.IN_PARTICIPA_ORCAMENTO, 'N') AS IN_PARTICIPA_ORCAMENTO
FROM vw_mov_efetivo m
LEFT JOIN vw_categorias c
  ON m.CD_CTGR_TRAN_OGNL = c.CD_CATEGORIA
 AND m.CD_NTZ_CTB_TRAN = c.TIPO
""")

# 2. Universo em Moeda Corrente Nacional (BRL)
# Filtra apenas transações em BRL para alimentação das métricas financeiras e orçamentárias
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_mov_brl AS
SELECT * FROM vw_mov_classificado WHERE CD_TIP_MOE_CRR = 'BRL'
""")

# 3. Derivação das Flags de Moeda e Agro
# - FL_SOMENTE_BRL: 'S' se todas as transações efetivas forem BRL, 'N' caso haja moedas estrangeiras
# - FL_TEM_MOV_AGRO: 'S' se houver pelo menos uma transação com indicador agropecuário
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_flags_moeda_agro AS
SELECT
    CASE 
        WHEN (SELECT COUNT(1) FROM vw_mov_classificado) = 0 THEN NULL
        WHEN (SELECT COUNT(DISTINCT CD_TIP_MOE_CRR) FROM vw_mov_classificado) = 1 
         AND (SELECT MAX(CD_TIP_MOE_CRR) FROM vw_mov_classificado) = 'BRL' THEN 'S'
        ELSE 'N'
    END AS FL_SOMENTE_BRL,
    CASE 
        WHEN (SELECT COUNT(1) FROM vw_mov_brl) = 0 THEN NULL
        WHEN (SELECT COUNT(1) FROM vw_mov_brl WHERE IN_AGRO = 'S') > 0 THEN 'S'
        ELSE 'N'
    END AS FL_TEM_MOV_AGRO
""")

# Agregações Financeiras em SQL:
# - Entradas Temáticas (0 a 4): Renda (1), Estorno (2), Resgate (3), Outras (0), Crédito (4) com IN_PARTICIPA_CALCULO = 'S'
# - Saídas Temáticas (5 a 9): Indeterminado (5), Essenciais (6), Não Essenciais (7), Futuro (8), Obrigações (9) com IN_PARTICIPA_CALCULO = 'S'
# - Totais Orçamentários: VL_ENT_TOTAL e VL_SAI_TOTAL considerando apenas transações com IN_PARTICIPA_ORCAMENTO = 'S'
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_agregacoes_financeiras AS
SELECT
    -- Quantidades
    COUNT(1) AS QT_TRANS_TOTAL,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 1 ELSE 0 END) AS QT_TRANS_ENT,
    SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' THEN 1 ELSE 0 END) AS QT_TRANS_SAI,
    -- Entradas Temáticas (Classes 0 a 4 com IN_PARTICIPA_CALCULO = 'S')
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' AND CD_CLASS_RADAR = 1 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_REN,
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' AND CD_CLASS_RADAR = 2 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_EST,
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' AND CD_CLASS_RADAR = 3 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_RESG,
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' AND CD_CLASS_RADAR = 0 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_OUT,
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' AND CD_CLASS_RADAR = 4 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_CRED,
    -- Saídas Temáticas (Classes 5 a 9 com IN_PARTICIPA_CALCULO = 'S')
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' AND CD_CLASS_RADAR = 5 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_IND,
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' AND CD_CLASS_RADAR = 6 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_ESS,
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' AND CD_CLASS_RADAR = 7 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_NAO_ESS,
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' AND CD_CLASS_RADAR = 8 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_FUT,
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' AND CD_CLASS_RADAR = 9 AND IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_OBR,
    -- Totais Orçamentários (IN_PARTICIPA_ORCAMENTO = 'S')
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' AND IN_PARTICIPA_ORCAMENTO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_TOTAL,
    CAST(COALESCE(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' AND IN_PARTICIPA_ORCAMENTO = 'S' THEN VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_TOTAL
FROM vw_mov_brl
""")

# 1. Cálculo do Saldo Orçamentário e Razão Saídas / Entradas
# - VL_RES_ORC = VL_ENT_TOTAL - VL_SAI_TOTAL
# - PC_SAI_ENT = VL_SAI_TOTAL / VL_ENT_TOTAL (arredondado em 6 casas decimais)
# 2. Enquadramento nas 5 Faixas Orçamentárias Contratuais:
# - Faixa 0: Neutro (0.95 <= PC <= 1.05)
# - Faixa 1: Deficitário Moderado (1.05 < PC <= 1.25)
# - Faixa 2: Deficitário Acentuado (PC > 1.25)
# - Faixa 3: Superavitário Moderado (0.75 <= PC < 0.95)
# - Faixa 4: Superavitário Acentuado (PC < 0.75)
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_orcamento_derivado AS
WITH base AS (
    SELECT
        VL_ENT_TOTAL,
        VL_SAI_TOTAL,
        QT_TRANS_TOTAL,
        CAST(VL_ENT_TOTAL - VL_SAI_TOTAL AS DECIMAL(25,2)) AS VL_RES_ORC,
        CASE 
            WHEN QT_TRANS_TOTAL = 0 OR VL_ENT_TOTAL = 0 THEN NULL
            ELSE CAST(ROUND(VL_SAI_TOTAL / VL_ENT_TOTAL, 6) AS DECIMAL(9,6))
        END AS PC_SAI_ENT
    FROM vw_agregacoes_financeiras
),
faixas AS (
    SELECT
        *,
        CASE 
            WHEN PC_SAI_ENT IS NULL THEN NULL
            WHEN PC_SAI_ENT >= 0.950000 AND PC_SAI_ENT <= 1.050000 THEN 0
            WHEN PC_SAI_ENT > 1.050000 AND PC_SAI_ENT <= 1.250000 THEN 1
            WHEN PC_SAI_ENT > 1.250000 THEN 2
            WHEN PC_SAI_ENT >= 0.750000 AND PC_SAI_ENT < 0.950000 THEN 3
            ELSE 4
        END AS CD_FAIXA_ORC
    FROM base
)
SELECT
    VL_RES_ORC,
    PC_SAI_ENT,
    CD_FAIXA_ORC,
    CASE 
        WHEN CD_FAIXA_ORC IS NULL THEN NULL
        WHEN CD_FAIXA_ORC = 0 THEN 0
        WHEN CD_FAIXA_ORC IN (1, 2) THEN 2
        ELSE 1
    END AS CD_RES_ORC,
    CASE 
        WHEN CD_FAIXA_ORC IS NULL THEN NULL
        WHEN CD_FAIXA_ORC = 0 THEN 'Neutro'
        WHEN CD_FAIXA_ORC IN (1, 2) THEN 'Deficitário'
        ELSE 'Superavitário'
    END AS TX_RES_ORC,
    CASE 
        WHEN CD_FAIXA_ORC IS NULL OR CD_FAIXA_ORC = 0 THEN NULL
        WHEN CD_FAIXA_ORC IN (1, 3) THEN 'Moderado'
        ELSE 'Acentuado'
    END AS TX_STS_RES,
    CASE 
        WHEN CD_FAIXA_ORC IS NULL THEN NULL
        WHEN CD_FAIXA_ORC = 0 THEN 'Neutro'
        WHEN CD_FAIXA_ORC = 1 THEN 'Deficitário Moderado'
        WHEN CD_FAIXA_ORC = 2 THEN 'Deficitário Acentuado'
        WHEN CD_FAIXA_ORC = 3 THEN 'Superavitário Moderado'
        WHEN CD_FAIXA_ORC = 4 THEN 'Superavitário Acentuado'
    END AS TX_STS_FINAL
FROM faixas
""")

# 1. Percentuais de Referência Contratuais (Constantes fixas):
# - IND: 75% | ESS: 50% | NAO_ESS: 30% | FUT: 20% | OBR: 30%
# 2. Percentuais Observados sobre a Renda Presumida:
# - PC_SAI_* = VL_SAI_* / VL_REN_PRES (arredondado em 6 casas decimais)
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_percentuais_renda AS
WITH renda_base AS (
    SELECT VL_REN_PRES FROM vw_renda_derivada
),
agregados AS (
    SELECT VL_SAI_IND, VL_SAI_ESS, VL_SAI_NAO_ESS, VL_SAI_FUT, VL_SAI_OBR FROM vw_agregacoes_financeiras
)
SELECT
    -- Constantes de Referência Contratuais
    CAST(0.750000 AS DECIMAL(9,6)) AS PC_REF_IND,
    CAST(0.500000 AS DECIMAL(9,6)) AS PC_REF_ESS,
    CAST(0.300000 AS DECIMAL(9,6)) AS PC_REF_NAO_ESS,
    CAST(0.200000 AS DECIMAL(9,6)) AS PC_REF_FUT,
    CAST(0.300000 AS DECIMAL(9,6)) AS PC_REF_OBR,
    -- Percentuais Observados sobre Renda Presumida
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_IND / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_IND,
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_ESS / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_ESS,
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_NAO_ESS / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_NAO_ESS,
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_FUT / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_FUT,
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_OBR / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_OBR
FROM agregados a
CROSS JOIN renda_base r
""")

# 1. Dimensões de Pontuação:
# - Concentração: pontua o desvio de cada gasto temático em relação aos percentuais de referência
# - Orçamentária: pontua a pressão sobre o orçamento global de acordo com a faixa orçamentária
# - Perfil: pontua a aderência do comportamento financeiro ao macroperfil do cliente
# 2. Regra Especial de IND:
# - O tema 1 (Categorização dos Gastos / IND) recebe estritamente a pontuação de concentração (IND_FIM = CONC_IND)
# - Não soma orçamento nem perfil, mantendo o foco analítico exclusivo no volume de despesas sem categorização
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_pontuacoes AS
WITH ctx AS (
    SELECT 
        (SELECT QT_TRANS_TOTAL FROM vw_agregacoes_financeiras) AS QT_TRANS_TOTAL,
        (SELECT VL_REN_PRES FROM vw_renda_derivada) AS VL_REN_PRES,
        (SELECT CD_FAIXA_ORC FROM vw_orcamento_derivado) AS CD_FAIXA_ORC,
        (SELECT CD_MAC_PRFL_CLI FROM vw_perfil_derivado) AS CD_MAC_PRFL_CLI,
        p.*
    FROM vw_percentuais_renda p
),
conc AS (
    SELECT
        ctx.*,
        -- Pontuação de Concentração por Tema
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR VL_REN_PRES IS NULL THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_IND > 0.750000 THEN 99
            ELSE 0
        END AS NR_PONT_CONC_IND,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR VL_REN_PRES IS NULL THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_ESS < 0.500000 THEN 0
            WHEN PC_SAI_ESS < 0.750000 THEN 1
            ELSE 2
        END AS NR_PONT_CONC_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR VL_REN_PRES IS NULL THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_NAO_ESS < 0.300000 THEN 0
            WHEN PC_SAI_NAO_ESS < 0.450000 THEN 1
            ELSE 2
        END AS NR_PONT_CONC_NAO_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR VL_REN_PRES IS NULL THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_FUT >= 0.300000 THEN 0
            WHEN PC_SAI_FUT >= 0.200000 THEN 1
            ELSE 2
        END AS NR_PONT_CONC_FUT,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR VL_REN_PRES IS NULL THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_OBR < 0.300000 THEN 0
            WHEN PC_SAI_OBR < 0.450000 THEN 1
            ELSE 2
        END AS NR_PONT_CONC_OBR
    FROM ctx
),
orc AS (
    SELECT
        conc.*,
        -- Pontuação Orçamentária por Tema (Matriz de Faixas)
        CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 THEN NULL ELSE 0 END AS NR_PONT_ORC_IND,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 2 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 1) THEN 1
            ELSE 0
        END AS NR_PONT_ORC_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 2 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 1) THEN 1
            ELSE 0
        END AS NR_PONT_ORC_NAO_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 4 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 3) THEN 1
            ELSE 0
        END AS NR_PONT_ORC_FUT,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 2 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 1) THEN 1
            ELSE 0
        END AS NR_PONT_ORC_OBR
    FROM conc
),
prfl AS (
    SELECT
        orc.*,
        -- Pontuação de Perfil por Tema (Matriz de Macroperfis 1-Endividado, 2-Equilibrista, 3-Investidor)
        CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 THEN NULL ELSE 0 END AS NR_PONT_PRFL_IND,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_MAC_PRFL_CLI IS NULL OR CD_MAC_PRFL_CLI NOT IN (1, 2, 3) THEN NULL
            WHEN CD_MAC_PRFL_CLI = 1 THEN 0
            ELSE 1
        END AS NR_PONT_PRFL_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_MAC_PRFL_CLI IS NULL OR CD_MAC_PRFL_CLI NOT IN (1, 2, 3) THEN NULL
            WHEN CD_MAC_PRFL_CLI = 1 THEN 1
            ELSE 0
        END AS NR_PONT_PRFL_NAO_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_MAC_PRFL_CLI IS NULL OR CD_MAC_PRFL_CLI NOT IN (1, 2, 3) THEN NULL
            WHEN CD_MAC_PRFL_CLI = 3 THEN 2
            WHEN CD_MAC_PRFL_CLI = 2 THEN 1
            ELSE 0
        END AS NR_PONT_PRFL_FUT,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_MAC_PRFL_CLI IS NULL OR CD_MAC_PRFL_CLI NOT IN (1, 2, 3) THEN NULL
            WHEN CD_MAC_PRFL_CLI = 1 THEN 2
            ELSE 0
        END AS NR_PONT_PRFL_OBR
    FROM orc
)
SELECT
    *,
    -- REGRA ESPECIAL IND: IND_FIM = CONC_IND (ignora ORC e PRFL)
    NR_PONT_CONC_IND AS NR_PONT_IND_FIM,
    -- Finais para os demais temas (soma CONC + ORC + PRFL, NULL se qualquer parcela for NULL)
    CASE WHEN NR_PONT_CONC_ESS IS NULL OR NR_PONT_ORC_ESS IS NULL OR NR_PONT_PRFL_ESS IS NULL THEN NULL ELSE NR_PONT_CONC_ESS + NR_PONT_ORC_ESS + NR_PONT_PRFL_ESS END AS NR_PONT_ESS_FIM,
    CASE WHEN NR_PONT_CONC_NAO_ESS IS NULL OR NR_PONT_ORC_NAO_ESS IS NULL OR NR_PONT_PRFL_NAO_ESS IS NULL THEN NULL ELSE NR_PONT_CONC_NAO_ESS + NR_PONT_ORC_NAO_ESS + NR_PONT_PRFL_NAO_ESS END AS NR_PONT_NAO_ESS_FIM,
    CASE WHEN NR_PONT_CONC_FUT IS NULL OR NR_PONT_ORC_FUT IS NULL OR NR_PONT_PRFL_FUT IS NULL THEN NULL ELSE NR_PONT_CONC_FUT + NR_PONT_ORC_FUT + NR_PONT_PRFL_FUT END AS NR_PONT_FUT_FIM,
    CASE WHEN NR_PONT_CONC_OBR IS NULL OR NR_PONT_ORC_OBR IS NULL OR NR_PONT_PRFL_OBR IS NULL THEN NULL ELSE NR_PONT_CONC_OBR + NR_PONT_ORC_OBR + NR_PONT_PRFL_OBR END AS NR_PONT_OBR_FIM,
    -- Flag de Pontuação Completa: 'S' se todos os 5 temas finais foram pontuados
    CASE 
        WHEN NR_PONT_CONC_IND IS NOT NULL 
         AND (NR_PONT_CONC_ESS + NR_PONT_ORC_ESS + NR_PONT_PRFL_ESS) IS NOT NULL 
         AND (NR_PONT_CONC_NAO_ESS + NR_PONT_ORC_NAO_ESS + NR_PONT_PRFL_NAO_ESS) IS NOT NULL 
         AND (NR_PONT_CONC_FUT + NR_PONT_ORC_FUT + NR_PONT_PRFL_FUT) IS NOT NULL 
         AND (NR_PONT_CONC_OBR + NR_PONT_ORC_OBR + NR_PONT_PRFL_OBR) IS NOT NULL 
        THEN 'S' 
        ELSE 'N' 
    END AS FL_PONTUACAO_COMPLETA
FROM prfl
""")

# 1. Identificação da Maior Pontuação (GREATEST dos 5 temas finais)
# 2. Contagem de Vencedores (empates):
# - Se houver mais de um tema com a pontuação máxima: CD_TEMA_VENCEDOR = 9 ('Empate')
# - Se houver vencedor único: atribui o código do tema com maior pontuação (1 a 5)
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_tema_vencedor AS
WITH base AS (
    SELECT * FROM vw_pontuacoes
),
max_calculado AS (
    SELECT
        *,
        CASE 
            WHEN FL_PONTUACAO_COMPLETA = 'N' THEN NULL
            ELSE GREATEST(NR_PONT_IND_FIM, NR_PONT_ESS_FIM, NR_PONT_NAO_ESS_FIM, NR_PONT_FUT_FIM, NR_PONT_OBR_FIM)
        END AS NR_PONT_MAX
    FROM base
),
contagem_vencedores AS (
    SELECT
        *,
        CASE 
            WHEN FL_PONTUACAO_COMPLETA = 'N' THEN NULL
            ELSE (
                CASE WHEN NR_PONT_IND_FIM = NR_PONT_MAX THEN 1 ELSE 0 END +
                CASE WHEN NR_PONT_ESS_FIM = NR_PONT_MAX THEN 1 ELSE 0 END +
                CASE WHEN NR_PONT_NAO_ESS_FIM = NR_PONT_MAX THEN 1 ELSE 0 END +
                CASE WHEN NR_PONT_FUT_FIM = NR_PONT_MAX THEN 1 ELSE 0 END +
                CASE WHEN NR_PONT_OBR_FIM = NR_PONT_MAX THEN 1 ELSE 0 END
            )
        END AS QT_TEMAS_PONT_MAX
    FROM max_calculado
)
SELECT
    *,
    CASE 
        WHEN FL_PONTUACAO_COMPLETA = 'N' THEN NULL
        WHEN QT_TEMAS_PONT_MAX > 1 THEN 9
        WHEN NR_PONT_IND_FIM = NR_PONT_MAX THEN 1
        WHEN NR_PONT_ESS_FIM = NR_PONT_MAX THEN 2
        WHEN NR_PONT_NAO_ESS_FIM = NR_PONT_MAX THEN 3
        WHEN NR_PONT_FUT_FIM = NR_PONT_MAX THEN 4
        WHEN NR_PONT_OBR_FIM = NR_PONT_MAX THEN 5
    END AS CD_TEMA_VENCEDOR,
    CASE 
        WHEN FL_PONTUACAO_COMPLETA = 'N' THEN NULL
        WHEN QT_TEMAS_PONT_MAX > 1 THEN 'Empate'
        WHEN NR_PONT_IND_FIM = NR_PONT_MAX THEN 'Categorização dos Gastos'
        WHEN NR_PONT_ESS_FIM = NR_PONT_MAX THEN 'Gestão de Orçamento'
        WHEN NR_PONT_NAO_ESS_FIM = NR_PONT_MAX THEN 'Consumo Planejado'
        WHEN NR_PONT_FUT_FIM = NR_PONT_MAX THEN 'Formação de Reserva'
        WHEN NR_PONT_OBR_FIM = NR_PONT_MAX THEN 'Uso Consciente do Crédito'
    END AS TX_TEMA_VENCEDOR
FROM contagem_vencedores
""")

# Consolidação dos 80 Atributos do Contrato Físico Final
# Junção analítica via CROSS JOIN das views parciais estruturadas
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW vw_resultado_80_colunas AS
SELECT
    -- 1 a 7: Cliente e CPF
    c.CD_CLI AS CD_CLI,
    DATE('{DATA_EXECUCAO.isoformat()}') AS DT_EXEA,
    DATE('{DT_MES_EXEA.isoformat()}') AS DT_MES_EXEA,
    c.TS_INCL_TRAN_REF AS TS_INCL_TRAN_REF,
    c.FL_CPF_UNICO AS FL_CPF_UNICO,
    c.CD_CPF AS CD_CPF,
    cta.FL_CONTA_ELEGIVEL_UNICA AS FL_CONTA_ELEGIVEL_UNICA,
    -- 8 a 10: Ciclo
    j.TS_DD_INC_MM_CLC_BLC_REF AS TS_DD_INC_MM_CLC_BLC_REF,
    j.DD_INC_MM_CLC_BLC AS DD_INC_MM_CLC_BLC,
    j.DD_INC_MM_CLC_BLC_FALLBACK AS DD_INC_MM_CLC_BLC_FALLBACK,
    -- 11 a 12: Renda
    r.DT_REN_PRES_REF AS DT_REN_PRES_REF,
    r.VL_REN_PRES AS VL_REN_PRES,
    -- 13 a 17: Perfil
    p.DT_REF_PRFL AS DT_REF_PRFL,
    p.CD_MAC_PRFL_CLI AS CD_MAC_PRFL_CLI,
    p.NM_MAC_PRFL_CLI AS NM_MAC_PRFL_CLI,
    p.CD_MIC_PRFL_CLI AS CD_MIC_PRFL_CLI,
    p.NM_MIC_PRFL_CLI AS NM_MIC_PRFL_CLI,
    -- 18 a 21: Janela, Moeda e Agro
    j.DT_REF_INI AS DT_REF_INI,
    j.DT_REF_FIM AS DT_REF_FIM,
    f.FL_SOMENTE_BRL AS FL_SOMENTE_BRL,
    f.FL_TEM_MOV_AGRO AS FL_TEM_MOV_AGRO,
    -- 22 a 26: Quantidades e Totais
    a.QT_TRANS_TOTAL AS QT_TRANS_TOTAL,
    a.QT_TRANS_ENT AS QT_TRANS_ENT,
    a.QT_TRANS_SAI AS QT_TRANS_SAI,
    a.VL_ENT_TOTAL AS VL_TRANS_ENT,
    a.VL_SAI_TOTAL AS VL_TRANS_SAI,
    -- 27 a 32: Entradas Temáticas e Total
    a.VL_ENT_REN AS VL_ENT_REN,
    a.VL_ENT_EST AS VL_ENT_EST,
    a.VL_ENT_RESG AS VL_ENT_RESG,
    a.VL_ENT_OUT AS VL_ENT_OUT,
    a.VL_ENT_CRED AS VL_ENT_CRED,
    a.VL_ENT_TOTAL AS VL_ENT_TOTAL,
    -- 33 a 38: Saídas Temáticas e Total
    a.VL_SAI_IND AS VL_SAI_IND,
    a.VL_SAI_ESS AS VL_SAI_ESS,
    a.VL_SAI_NAO_ESS AS VL_SAI_NAO_ESS,
    a.VL_SAI_FUT AS VL_SAI_FUT,
    a.VL_SAI_OBR AS VL_SAI_OBR,
    a.VL_SAI_TOTAL AS VL_SAI_TOTAL,
    -- 39 a 45: Orçamento
    o.VL_RES_ORC AS VL_RES_ORC,
    o.PC_SAI_ENT AS PC_SAI_ENT,
    o.CD_RES_ORC AS CD_RES_ORC,
    o.TX_RES_ORC AS TX_RES_ORC,
    o.CD_FAIXA_ORC AS CD_FAIXA_ORC,
    o.TX_STS_RES AS TX_STS_RES,
    o.TX_STS_FINAL AS TX_STS_FINAL,
    -- 46 a 50: Percentuais Saídas sobre Renda
    v.PC_SAI_IND AS PC_SAI_IND,
    v.PC_SAI_ESS AS PC_SAI_ESS,
    v.PC_SAI_NAO_ESS AS PC_SAI_NAO_ESS,
    v.PC_SAI_FUT AS PC_SAI_FUT,
    v.PC_SAI_OBR AS PC_SAI_OBR,
    -- 51 a 55: Percentuais de Referência
    v.PC_REF_IND AS PC_REF_IND,
    v.PC_REF_ESS AS PC_REF_ESS,
    v.PC_REF_NAO_ESS AS PC_REF_NAO_ESS,
    v.PC_REF_FUT AS PC_REF_FUT,
    v.PC_REF_OBR AS PC_REF_OBR,
    -- 56 a 60: Pontuação de Concentração
    v.NR_PONT_CONC_IND AS NR_PONT_CONC_IND,
    v.NR_PONT_CONC_ESS AS NR_PONT_CONC_ESS,
    v.NR_PONT_CONC_NAO_ESS AS NR_PONT_CONC_NAO_ESS,
    v.NR_PONT_CONC_FUT AS NR_PONT_CONC_FUT,
    v.NR_PONT_CONC_OBR AS NR_PONT_CONC_OBR,
    -- 61 a 65: Pontuação Orçamentária
    v.NR_PONT_ORC_IND AS NR_PONT_ORC_IND,
    v.NR_PONT_ORC_ESS AS NR_PONT_ORC_ESS,
    v.NR_PONT_ORC_NAO_ESS AS NR_PONT_ORC_NAO_ESS,
    v.NR_PONT_ORC_FUT AS NR_PONT_ORC_FUT,
    v.NR_PONT_ORC_OBR AS NR_PONT_ORC_OBR,
    -- 66 a 70: Pontuação de Perfil
    v.NR_PONT_PRFL_IND AS NR_PONT_PRFL_IND,
    v.NR_PONT_PRFL_ESS AS NR_PONT_PRFL_ESS,
    v.NR_PONT_PRFL_NAO_ESS AS NR_PONT_PRFL_NAO_ESS,
    v.NR_PONT_PRFL_FUT AS NR_PONT_PRFL_FUT,
    v.NR_PONT_PRFL_OBR AS NR_PONT_PRFL_OBR,
    -- 71 a 75: Pontuações Finais
    v.NR_PONT_IND_FIM AS NR_PONT_IND_FIM,
    v.NR_PONT_ESS_FIM AS NR_PONT_ESS_FIM,
    v.NR_PONT_NAO_ESS_FIM AS NR_PONT_NAO_ESS_FIM,
    v.NR_PONT_FUT_FIM AS NR_PONT_FUT_FIM,
    v.NR_PONT_OBR_FIM AS NR_PONT_OBR_FIM,
    -- 76 a 80: Status e Tema Vencedor
    v.FL_PONTUACAO_COMPLETA AS FL_PONTUACAO_COMPLETA,
    v.NR_PONT_MAX AS NR_PONT_MAX,
    v.QT_TEMAS_PONT_MAX AS QT_TEMAS_PONT_MAX,
    v.CD_TEMA_VENCEDOR AS CD_TEMA_VENCEDOR,
    v.TX_TEMA_VENCEDOR AS TX_TEMA_VENCEDOR
FROM vw_cliente_derivado c
CROSS JOIN vw_conta_elegivel_resumo cta
CROSS JOIN vw_ciclo_janela j
CROSS JOIN vw_renda_derivada r
CROSS JOIN vw_perfil_derivado p
CROSS JOIN vw_flags_moeda_agro f
CROSS JOIN vw_agregacoes_financeiras a
CROSS JOIN vw_orcamento_derivado o
CROSS JOIN vw_tema_vencedor v
""")

print('[RADAR_INDIVIDUAL] View SQL vw_resultado_80_colunas montada com sucesso.')

# Publica e materializa o resultado oficial para os dois usos seguintes.
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW {VIEW_RESULTADO} AS
SELECT * FROM vw_resultado_80_colunas
""")
df_res_80 = spark.table(VIEW_RESULTADO).persist(StorageLevel.MEMORY_AND_DISK)
linha_final_sql = df_res_80.first()
print(f'[RADAR_INDIVIDUAL] View final publicada: {VIEW_RESULTADO}.')

from pyspark.sql import functions as F


# Deriva Entradas Realizadas somente do universo já efetivo/reconciliado.
# O INNER JOIN exige classificação válida; BRL e natureza C são explícitos.
SQL_ENTRADAS_REALIZADAS = """
CREATE OR REPLACE TEMPORARY VIEW vw_cenario_entradas_realizadas_detalhe AS
SELECT
    m.CD_CLI,
    m.DT_TRAN,
    m.CD_NTZ_CTB_TRAN,
    m.CD_CTGR_TRAN_OGNL,
    m.CD_TIP_MOE_CRR,
    m.VL_TRAN,
    c.CD_GRUPO,
    c.TX_GRUPO,
    c.TX_CATEGORIA,
    c.CD_CLASS_RADAR,
    c.TX_CLASS_RADAR
FROM vw_mov_efetivo m
INNER JOIN vw_categorias c
  ON m.CD_CTGR_TRAN_OGNL = c.CD_CATEGORIA
 AND m.CD_NTZ_CTB_TRAN = c.TIPO
WHERE m.CD_NTZ_CTB_TRAN = 'C'
  AND m.CD_TIP_MOE_CRR = 'BRL'
"""
spark.sql(SQL_ENTRADAS_REALIZADAS)

# Janela indisponível é NULL; zero só representa janela existente sem crédito válido.
SQL_TOTAL_ENTRADAS_REALIZADAS = """
CREATE OR REPLACE TEMPORARY VIEW vw_cenario_entradas_realizadas AS
WITH total AS (
    SELECT CAST(SUM(VL_TRAN) AS DECIMAL(25,2)) AS SOMA
    FROM vw_cenario_entradas_realizadas_detalhe
)
SELECT
    CASE
        WHEN j.DT_REF_INI IS NULL OR j.DT_REF_FIM IS NULL
            THEN CAST(NULL AS DECIMAL(25,2))
        ELSE CAST(
            COALESCE(t.SOMA, CAST(0.00 AS DECIMAL(25,2)))
            AS DECIMAL(25,2)
        )
    END AS ENTRADAS_REALIZADAS
FROM vw_ciclo_janela j
CROSS JOIN total t
"""
spark.sql(SQL_TOTAL_ENTRADAS_REALIZADAS)

SQL_BASES_FINANCEIRAS = """
CREATE OR REPLACE TEMPORARY VIEW vw_cenario_bases_financeiras AS
SELECT
    'RENDA_PRESUMIDA' AS CD_CENARIO,
    CASE
        WHEN j.DT_REF_INI IS NULL OR j.DT_REF_FIM IS NULL
            THEN CAST(NULL AS DECIMAL(25,2))
        ELSE CAST(r.VL_REN_PRES AS DECIMAL(25,2))
    END AS BASE_FINANCEIRA
FROM vw_ciclo_janela j
CROSS JOIN vw_renda_derivada r
UNION ALL
SELECT
    'ENTRADAS_REALIZADAS' AS CD_CENARIO,
    CAST(e.ENTRADAS_REALIZADAS AS DECIMAL(25,2)) AS BASE_FINANCEIRA
FROM vw_cenario_entradas_realizadas e
"""
spark.sql(SQL_BASES_FINANCEIRAS)

# Fonte única das regras aplicadas aos cenários financeiros.
SQL_CADEIA_CENARIO = {
    'orcamento': r"""CREATE OR REPLACE TEMPORARY VIEW vw_orcamento_derivado AS
WITH base AS (
    SELECT
        VL_ENT_TOTAL,
        VL_SAI_TOTAL,
        QT_TRANS_TOTAL,
        CAST(VL_ENT_TOTAL - VL_SAI_TOTAL AS DECIMAL(25,2)) AS VL_RES_ORC,
        CASE 
            WHEN QT_TRANS_TOTAL = 0 OR VL_ENT_TOTAL = 0 THEN NULL
            ELSE CAST(ROUND(VL_SAI_TOTAL / VL_ENT_TOTAL, 6) AS DECIMAL(9,6))
        END AS PC_SAI_ENT
    FROM vw_agregacoes_financeiras
),
faixas AS (
    SELECT
        *,
        CASE 
            WHEN PC_SAI_ENT IS NULL THEN NULL
            WHEN PC_SAI_ENT >= 0.950000 AND PC_SAI_ENT <= 1.050000 THEN 0
            WHEN PC_SAI_ENT > 1.050000 AND PC_SAI_ENT <= 1.250000 THEN 1
            WHEN PC_SAI_ENT > 1.250000 THEN 2
            WHEN PC_SAI_ENT >= 0.750000 AND PC_SAI_ENT < 0.950000 THEN 3
            ELSE 4
        END AS CD_FAIXA_ORC
    FROM base
)
SELECT
    VL_RES_ORC,
    PC_SAI_ENT,
    CD_FAIXA_ORC,
    CASE 
        WHEN CD_FAIXA_ORC IS NULL THEN NULL
        WHEN CD_FAIXA_ORC = 0 THEN 0
        WHEN CD_FAIXA_ORC IN (1, 2) THEN 2
        ELSE 1
    END AS CD_RES_ORC,
    CASE 
        WHEN CD_FAIXA_ORC IS NULL THEN NULL
        WHEN CD_FAIXA_ORC = 0 THEN 'Neutro'
        WHEN CD_FAIXA_ORC IN (1, 2) THEN 'Deficitário'
        ELSE 'Superavitário'
    END AS TX_RES_ORC,
    CASE 
        WHEN CD_FAIXA_ORC IS NULL OR CD_FAIXA_ORC = 0 THEN NULL
        WHEN CD_FAIXA_ORC IN (1, 3) THEN 'Moderado'
        ELSE 'Acentuado'
    END AS TX_STS_RES,
    CASE 
        WHEN CD_FAIXA_ORC IS NULL THEN NULL
        WHEN CD_FAIXA_ORC = 0 THEN 'Neutro'
        WHEN CD_FAIXA_ORC = 1 THEN 'Deficitário Moderado'
        WHEN CD_FAIXA_ORC = 2 THEN 'Deficitário Acentuado'
        WHEN CD_FAIXA_ORC = 3 THEN 'Superavitário Moderado'
        WHEN CD_FAIXA_ORC = 4 THEN 'Superavitário Acentuado'
    END AS TX_STS_FINAL
FROM faixas""",
    'percentuais': r"""CREATE OR REPLACE TEMPORARY VIEW vw_percentuais_renda AS
WITH renda_base AS (
    SELECT VL_REN_PRES FROM vw_renda_derivada
),
agregados AS (
    SELECT VL_SAI_IND, VL_SAI_ESS, VL_SAI_NAO_ESS, VL_SAI_FUT, VL_SAI_OBR FROM vw_agregacoes_financeiras
)
SELECT
    -- Constantes de Referência Contratuais
    CAST(0.750000 AS DECIMAL(9,6)) AS PC_REF_IND,
    CAST(0.500000 AS DECIMAL(9,6)) AS PC_REF_ESS,
    CAST(0.300000 AS DECIMAL(9,6)) AS PC_REF_NAO_ESS,
    CAST(0.200000 AS DECIMAL(9,6)) AS PC_REF_FUT,
    CAST(0.300000 AS DECIMAL(9,6)) AS PC_REF_OBR,
    -- Percentuais Observados sobre Renda Presumida
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_IND / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_IND,
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_ESS / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_ESS,
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_NAO_ESS / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_NAO_ESS,
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_FUT / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_FUT,
    CASE WHEN r.VL_REN_PRES IS NULL OR r.VL_REN_PRES <= 0 THEN NULL ELSE CAST(ROUND(a.VL_SAI_OBR / r.VL_REN_PRES, 6) AS DECIMAL(9,6)) END AS PC_SAI_OBR
FROM agregados a
CROSS JOIN renda_base r""",
    'pontuacoes': r"""CREATE OR REPLACE TEMPORARY VIEW vw_pontuacoes AS
WITH ctx AS (
    SELECT 
        (SELECT QT_TRANS_TOTAL FROM vw_agregacoes_financeiras) AS QT_TRANS_TOTAL,
        (SELECT VL_REN_PRES FROM vw_renda_derivada) AS VL_REN_PRES,
        (SELECT CD_FAIXA_ORC FROM vw_orcamento_derivado) AS CD_FAIXA_ORC,
        (SELECT CD_MAC_PRFL_CLI FROM vw_perfil_derivado) AS CD_MAC_PRFL_CLI,
        p.*
    FROM vw_percentuais_renda p
),
conc AS (
    SELECT
        ctx.*,
        -- Pontuação de Concentração por Tema
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR VL_REN_PRES IS NULL THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_IND > 0.750000 THEN 99
            ELSE 0
        END AS NR_PONT_CONC_IND,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR VL_REN_PRES IS NULL THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_ESS < 0.500000 THEN 0
            WHEN PC_SAI_ESS < 0.750000 THEN 1
            ELSE 2
        END AS NR_PONT_CONC_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR VL_REN_PRES IS NULL THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_NAO_ESS < 0.300000 THEN 0
            WHEN PC_SAI_NAO_ESS < 0.450000 THEN 1
            ELSE 2
        END AS NR_PONT_CONC_NAO_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR VL_REN_PRES IS NULL THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_FUT >= 0.300000 THEN 0
            WHEN PC_SAI_FUT >= 0.200000 THEN 1
            ELSE 2
        END AS NR_PONT_CONC_FUT,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR VL_REN_PRES IS NULL THEN NULL
            WHEN VL_REN_PRES <= 0 THEN 0
            WHEN PC_SAI_OBR < 0.300000 THEN 0
            WHEN PC_SAI_OBR < 0.450000 THEN 1
            ELSE 2
        END AS NR_PONT_CONC_OBR
    FROM ctx
),
orc AS (
    SELECT
        conc.*,
        -- Pontuação Orçamentária por Tema (Matriz de Faixas)
        CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 THEN NULL ELSE 0 END AS NR_PONT_ORC_IND,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 2 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 1) THEN 1
            ELSE 0
        END AS NR_PONT_ORC_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 2 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 1) THEN 1
            ELSE 0
        END AS NR_PONT_ORC_NAO_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 4 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 3) THEN 1
            ELSE 0
        END AS NR_PONT_ORC_FUT,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 2 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 1) THEN 1
            ELSE 0
        END AS NR_PONT_ORC_OBR
    FROM conc
),
prfl AS (
    SELECT
        orc.*,
        -- Pontuação de Perfil por Tema (Matriz de Macroperfis 1-Endividado, 2-Equilibrista, 3-Investidor)
        CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 THEN NULL ELSE 0 END AS NR_PONT_PRFL_IND,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_MAC_PRFL_CLI IS NULL OR CD_MAC_PRFL_CLI NOT IN (1, 2, 3) THEN NULL
            WHEN CD_MAC_PRFL_CLI = 1 THEN 0
            ELSE 1
        END AS NR_PONT_PRFL_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_MAC_PRFL_CLI IS NULL OR CD_MAC_PRFL_CLI NOT IN (1, 2, 3) THEN NULL
            WHEN CD_MAC_PRFL_CLI = 1 THEN 1
            ELSE 0
        END AS NR_PONT_PRFL_NAO_ESS,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_MAC_PRFL_CLI IS NULL OR CD_MAC_PRFL_CLI NOT IN (1, 2, 3) THEN NULL
            WHEN CD_MAC_PRFL_CLI = 3 THEN 2
            WHEN CD_MAC_PRFL_CLI = 2 THEN 1
            ELSE 0
        END AS NR_PONT_PRFL_FUT,
        CASE 
            WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_MAC_PRFL_CLI IS NULL OR CD_MAC_PRFL_CLI NOT IN (1, 2, 3) THEN NULL
            WHEN CD_MAC_PRFL_CLI = 1 THEN 2
            ELSE 0
        END AS NR_PONT_PRFL_OBR
    FROM orc
)
SELECT
    *,
    -- REGRA ESPECIAL IND: IND_FIM = CONC_IND (ignora ORC e PRFL)
    NR_PONT_CONC_IND AS NR_PONT_IND_FIM,
    -- Finais para os demais temas (soma CONC + ORC + PRFL, NULL se qualquer parcela for NULL)
    CASE WHEN NR_PONT_CONC_ESS IS NULL OR NR_PONT_ORC_ESS IS NULL OR NR_PONT_PRFL_ESS IS NULL THEN NULL ELSE NR_PONT_CONC_ESS + NR_PONT_ORC_ESS + NR_PONT_PRFL_ESS END AS NR_PONT_ESS_FIM,
    CASE WHEN NR_PONT_CONC_NAO_ESS IS NULL OR NR_PONT_ORC_NAO_ESS IS NULL OR NR_PONT_PRFL_NAO_ESS IS NULL THEN NULL ELSE NR_PONT_CONC_NAO_ESS + NR_PONT_ORC_NAO_ESS + NR_PONT_PRFL_NAO_ESS END AS NR_PONT_NAO_ESS_FIM,
    CASE WHEN NR_PONT_CONC_FUT IS NULL OR NR_PONT_ORC_FUT IS NULL OR NR_PONT_PRFL_FUT IS NULL THEN NULL ELSE NR_PONT_CONC_FUT + NR_PONT_ORC_FUT + NR_PONT_PRFL_FUT END AS NR_PONT_FUT_FIM,
    CASE WHEN NR_PONT_CONC_OBR IS NULL OR NR_PONT_ORC_OBR IS NULL OR NR_PONT_PRFL_OBR IS NULL THEN NULL ELSE NR_PONT_CONC_OBR + NR_PONT_ORC_OBR + NR_PONT_PRFL_OBR END AS NR_PONT_OBR_FIM,
    -- Flag de Pontuação Completa: 'S' se todos os 5 temas finais foram pontuados
    CASE 
        WHEN NR_PONT_CONC_IND IS NOT NULL 
         AND (NR_PONT_CONC_ESS + NR_PONT_ORC_ESS + NR_PONT_PRFL_ESS) IS NOT NULL 
         AND (NR_PONT_CONC_NAO_ESS + NR_PONT_ORC_NAO_ESS + NR_PONT_PRFL_NAO_ESS) IS NOT NULL 
         AND (NR_PONT_CONC_FUT + NR_PONT_ORC_FUT + NR_PONT_PRFL_FUT) IS NOT NULL 
         AND (NR_PONT_CONC_OBR + NR_PONT_ORC_OBR + NR_PONT_PRFL_OBR) IS NOT NULL 
        THEN 'S' 
        ELSE 'N' 
    END AS FL_PONTUACAO_COMPLETA
FROM prfl""",
    'vencedor': r"""CREATE OR REPLACE TEMPORARY VIEW vw_tema_vencedor AS
WITH base AS (
    SELECT * FROM vw_pontuacoes
),
max_calculado AS (
    SELECT
        *,
        CASE 
            WHEN FL_PONTUACAO_COMPLETA = 'N' THEN NULL
            ELSE GREATEST(NR_PONT_IND_FIM, NR_PONT_ESS_FIM, NR_PONT_NAO_ESS_FIM, NR_PONT_FUT_FIM, NR_PONT_OBR_FIM)
        END AS NR_PONT_MAX
    FROM base
),
contagem_vencedores AS (
    SELECT
        *,
        CASE 
            WHEN FL_PONTUACAO_COMPLETA = 'N' THEN NULL
            ELSE (
                CASE WHEN NR_PONT_IND_FIM = NR_PONT_MAX THEN 1 ELSE 0 END +
                CASE WHEN NR_PONT_ESS_FIM = NR_PONT_MAX THEN 1 ELSE 0 END +
                CASE WHEN NR_PONT_NAO_ESS_FIM = NR_PONT_MAX THEN 1 ELSE 0 END +
                CASE WHEN NR_PONT_FUT_FIM = NR_PONT_MAX THEN 1 ELSE 0 END +
                CASE WHEN NR_PONT_OBR_FIM = NR_PONT_MAX THEN 1 ELSE 0 END
            )
        END AS QT_TEMAS_PONT_MAX
    FROM max_calculado
)
SELECT
    *,
    CASE 
        WHEN FL_PONTUACAO_COMPLETA = 'N' THEN NULL
        WHEN QT_TEMAS_PONT_MAX > 1 THEN 9
        WHEN NR_PONT_IND_FIM = NR_PONT_MAX THEN 1
        WHEN NR_PONT_ESS_FIM = NR_PONT_MAX THEN 2
        WHEN NR_PONT_NAO_ESS_FIM = NR_PONT_MAX THEN 3
        WHEN NR_PONT_FUT_FIM = NR_PONT_MAX THEN 4
        WHEN NR_PONT_OBR_FIM = NR_PONT_MAX THEN 5
    END AS CD_TEMA_VENCEDOR,
    CASE 
        WHEN FL_PONTUACAO_COMPLETA = 'N' THEN NULL
        WHEN QT_TEMAS_PONT_MAX > 1 THEN 'Empate'
        WHEN NR_PONT_IND_FIM = NR_PONT_MAX THEN 'Categorização dos Gastos'
        WHEN NR_PONT_ESS_FIM = NR_PONT_MAX THEN 'Gestão de Orçamento'
        WHEN NR_PONT_NAO_ESS_FIM = NR_PONT_MAX THEN 'Consumo Planejado'
        WHEN NR_PONT_FUT_FIM = NR_PONT_MAX THEN 'Formação de Reserva'
        WHEN NR_PONT_OBR_FIM = NR_PONT_MAX THEN 'Uso Consciente do Crédito'
    END AS TX_TEMA_VENCEDOR
FROM contagem_vencedores""",
}

def _nome_view_cenario(prefixo, nome_original):
    return f'vw_cenario_{prefixo}_{nome_original[3:]}'


def _remapear_sql_cenario(sql_original, prefixo):
    nomes = [
        'vw_agregacoes_financeiras', 'vw_renda_derivada',
        'vw_orcamento_derivado', 'vw_percentuais_renda',
        'vw_pontuacoes', 'vw_tema_vencedor',
    ]
    remapeado = sql_original
    for nome in nomes:
        remapeado = re.sub(
            rf'\b{re.escape(nome)}\b',
            _nome_view_cenario(prefixo, nome),
            remapeado,
        )
    return remapeado


def calcular_cenario_entradas_realizadas(base_financeira):
    prefixo = 'entradas_realizadas'
    base_col = F.lit(base_financeira).cast(DecimalType(25, 2))
    nome_agregacoes = _nome_view_cenario(prefixo, 'vw_agregacoes_financeiras')
    nome_renda = _nome_view_cenario(prefixo, 'vw_renda_derivada')
    spark.table('vw_agregacoes_financeiras').withColumn(
        'VL_ENT_TOTAL', base_col
    ).createOrReplaceTempView(nome_agregacoes)
    spark.table('vw_renda_derivada').withColumn(
        'VL_REN_PRES', base_col
    ).createOrReplaceTempView(nome_renda)

    for etapa in ('orcamento', 'percentuais', 'pontuacoes', 'vencedor'):
        spark.sql(_remapear_sql_cenario(SQL_CADEIA_CENARIO[etapa], prefixo))

    nome_orcamento = _nome_view_cenario(prefixo, 'vw_orcamento_derivado')
    nome_vencedor = _nome_view_cenario(prefixo, 'vw_tema_vencedor')
    df_orcamento = spark.table(nome_orcamento).alias('o')
    df_vencedor = spark.table(nome_vencedor).alias('v')
    calculo = df_orcamento.crossJoin(df_vencedor).select(
        F.to_json(F.struct(*[
            F.col(f'o.{campo}') for campo in df_orcamento.columns
        ])).alias('ORCAMENTO'),
        F.to_json(F.struct(*[
            F.col(f'v.{campo}') for campo in df_vencedor.columns
        ])).alias('RESULTADO'),
    ).first()
    return {
        'orcamento': json.loads(calculo['ORCAMENTO']),
        'resultado': json.loads(calculo['RESULTADO']),
    }


resultado_oficial = linha_final_sql.asDict(recursive=True)
base_entradas_realizadas = spark.table('vw_cenario_bases_financeiras').where(
    F.col('CD_CENARIO') == 'ENTRADAS_REALIZADAS'
).first()['BASE_FINANCEIRA']
bases_cenarios = {
    'RENDA_PRESUMIDA': resultado_oficial.get('VL_REN_PRES'),
    'ENTRADAS_REALIZADAS': base_entradas_realizadas,
}

calculo_entradas = calcular_cenario_entradas_realizadas(base_entradas_realizadas)
CAMPOS_ORCAMENTO_CENARIO = [
    'VL_RES_ORC', 'PC_SAI_ENT', 'CD_RES_ORC', 'TX_RES_ORC',
    'CD_FAIXA_ORC', 'TX_STS_RES', 'TX_STS_FINAL',
]
CAMPOS_MOTOR_CENARIO = [
    'PC_SAI_IND', 'PC_SAI_ESS', 'PC_SAI_NAO_ESS', 'PC_SAI_FUT', 'PC_SAI_OBR',
    'NR_PONT_CONC_IND', 'NR_PONT_CONC_ESS', 'NR_PONT_CONC_NAO_ESS',
    'NR_PONT_CONC_FUT', 'NR_PONT_CONC_OBR',
    'NR_PONT_ORC_ESS', 'NR_PONT_ORC_NAO_ESS',
    'NR_PONT_ORC_FUT', 'NR_PONT_ORC_OBR',
    'NR_PONT_IND_FIM', 'NR_PONT_ESS_FIM', 'NR_PONT_NAO_ESS_FIM',
    'NR_PONT_FUT_FIM', 'NR_PONT_OBR_FIM',
    'FL_PONTUACAO_COMPLETA', 'NR_PONT_MAX', 'QT_TEMAS_PONT_MAX',
    'CD_TEMA_VENCEDOR', 'TX_TEMA_VENCEDOR',
]

resultado_entradas = dict(resultado_oficial)
for campo in CAMPOS_ORCAMENTO_CENARIO:
    resultado_entradas[campo] = calculo_entradas['orcamento'][campo]
for campo in CAMPOS_MOTOR_CENARIO:
    resultado_entradas[campo] = calculo_entradas['resultado'][campo]

resultados_cenarios = {
    'RENDA_PRESUMIDA': resultado_oficial,
    'ENTRADAS_REALIZADAS': resultado_entradas,
}
print(
    '[RADAR_INDIVIDUAL] Cenários preparados: '
    f'Renda Presumida={bases_cenarios["RENDA_PRESUMIDA"]}; '
    f'Entradas Realizadas={bases_cenarios["ENTRADAS_REALIZADAS"]}.'
)

# Bloco exclusivamente explicativo. Nenhuma view criada abaixo alimenta o motor.
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_dashboard_transacoes AS
SELECT
    m.NR_TRAN_INST_PCT,
    m.CD_CLI,
    m.DT_TRAN,
    m.CD_NTZ_CTB_TRAN,
    m.CD_CTGR_TRAN_OGNL,
    m.CD_TIP_MOE_CRR,
    m.VL_TRAN,
    m.IN_JANELA,
    CAST(p.TX_DCR_TRAN_OGNL AS STRING) AS TX_DCR_TRAN_OGNL,
    CAST(p.NR_MCA_PCT_OPB AS STRING) AS NR_MCA_PCT_OPB,
    c.CD_GRUPO,
    c.TX_GRUPO,
    c.CD_IR,
    c.TX_IR,
    COALESCE(c.TX_CATEGORIA, 'Sem Categoria') AS TX_CATEGORIA,
    c.CD_CLASS_RADAR AS CD_CLASS_RADAR,
    c.TX_CLASS_RADAR AS TX_CLASS_RADAR,
    CASE
      WHEN c.CD_CLASS_RADAR IS NOT NULL AND c.TX_CLASS_RADAR IS NOT NULL THEN 'S'
      ELSE 'N'
    END AS IN_CLASSIFICADA_APRESENTACAO,
    COALESCE(c.IN_AGRO, 'N') AS IN_AGRO,
    COALESCE(c.IN_PARTICIPA_CALCULO, 'N') AS IN_PARTICIPA_CALCULO,
    COALESCE(c.IN_PARTICIPA_ORCAMENTO, 'N') AS IN_PARTICIPA_ORCAMENTO,
    r.TIPO_CONSUMO
FROM vw_mov_marcado m
LEFT JOIN vw_q5_mov_contexto_apresentacao p
  ON m.NR_TRAN_INST_PCT = p.NR_TRAN_INST_PCT
LEFT JOIN vw_categorias c
  ON m.CD_CTGR_TRAN_OGNL = c.CD_CATEGORIA
 AND m.CD_NTZ_CTB_TRAN = c.TIPO
LEFT JOIN vw_ids_consumidos_todos r
  ON m.NR_TRAN_INST_PCT = r.NR_TRAN_INST_PCT
""")

spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_dashboard_transacoes_efetivas AS
SELECT *
FROM vw_dashboard_transacoes
WHERE IN_JANELA = 'S'
  AND TIPO_CONSUMO IS NULL
  AND CD_TIP_MOE_CRR = 'BRL'
""")

# Pareamento explicativo dos IDs exatos que o motor já selecionou.
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_dashboard_pares_exatos AS
WITH consumidas AS (
    SELECT
        m.*,
        e.TIPO_CONSUMO,
        ROW_NUMBER() OVER (
            PARTITION BY m.CD_CLI, m.DT_TRAN, m.VL_TRAN,
                         m.CD_TIP_MOE_CRR, m.IN_JANELA, m.CD_NTZ_CTB_TRAN
            ORDER BY m.NR_TRAN_INST_PCT
        ) AS RN_PAR
    FROM vw_mov_marcado m
    INNER JOIN vw_ids_consumidos_exatos e
      ON m.NR_TRAN_INST_PCT = e.NR_TRAN_INST_PCT
),
pares AS (
    SELECT
        c.TIPO_CONSUMO,
        c.IN_JANELA,
        c.NR_TRAN_INST_PCT AS ID_CREDITO,
        d.NR_TRAN_INST_PCT AS ID_DEBITO,
        c.DT_TRAN,
        c.VL_TRAN,
        c.CD_TIP_MOE_CRR
    FROM consumidas c
    INNER JOIN consumidas d
      ON c.CD_CLI = d.CD_CLI
     AND c.DT_TRAN = d.DT_TRAN
     AND c.VL_TRAN = d.VL_TRAN
     AND c.CD_TIP_MOE_CRR = d.CD_TIP_MOE_CRR
     AND c.IN_JANELA = d.IN_JANELA
     AND c.RN_PAR = d.RN_PAR
    WHERE c.CD_NTZ_CTB_TRAN = 'C'
      AND d.CD_NTZ_CTB_TRAN = 'D'
)
SELECT
    x.*,
    CAST(pc.TX_DCR_TRAN_OGNL AS STRING) AS DESC_CREDITO,
    CAST(pc.NR_MCA_PCT_OPB AS STRING) AS BANCO_CREDITO,
    CAST(pd.TX_DCR_TRAN_OGNL AS STRING) AS DESC_DEBITO,
    CAST(pd.NR_MCA_PCT_OPB AS STRING) AS BANCO_DEBITO
FROM pares x
LEFT JOIN vw_q5_mov_contexto_apresentacao pc ON x.ID_CREDITO = pc.NR_TRAN_INST_PCT
LEFT JOIN vw_q5_mov_contexto_apresentacao pd ON x.ID_DEBITO = pd.NR_TRAN_INST_PCT
""")

spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vw_dashboard_pares_borda AS
SELECT
    b.NR_TRAN_DENTRO,
    b.NR_TRAN_FORA,
    b.DT_TRAN_DENTRO,
    b.DT_TRAN_FORA,
    b.DIF_DIAS,
    md.CD_NTZ_CTB_TRAN AS NTZ_DENTRO,
    mf.CD_NTZ_CTB_TRAN AS NTZ_FORA,
    md.VL_TRAN,
    md.CD_TIP_MOE_CRR,
    CAST(pd.TX_DCR_TRAN_OGNL AS STRING) AS DESC_DENTRO,
    CAST(pd.NR_MCA_PCT_OPB AS STRING) AS BANCO_DENTRO,
    CAST(pf.TX_DCR_TRAN_OGNL AS STRING) AS DESC_FORA,
    CAST(pf.NR_MCA_PCT_OPB AS STRING) AS BANCO_FORA
FROM vw_pares_borda_calculados b
INNER JOIN vw_mov_marcado md ON b.NR_TRAN_DENTRO = md.NR_TRAN_INST_PCT
INNER JOIN vw_mov_marcado mf ON b.NR_TRAN_FORA = mf.NR_TRAN_INST_PCT
LEFT JOIN vw_q5_mov_contexto_apresentacao pd ON b.NR_TRAN_DENTRO = pd.NR_TRAN_INST_PCT
LEFT JOIN vw_q5_mov_contexto_apresentacao pf ON b.NR_TRAN_FORA = pf.NR_TRAN_INST_PCT
""")

df_dashboard_pivot = spark.sql("""
SELECT
    CD_NTZ_CTB_TRAN AS ntz,
    CD_CLASS_RADAR AS cd_classe,
    TX_CLASS_RADAR AS tx_classe,
    IN_CLASSIFICADA_APRESENTACAO AS classificada,
    CD_CTGR_TRAN_OGNL AS cd_cat,
    TX_CATEGORIA AS tx_cat,
    IN_PARTICIPA_CALCULO AS part_calc,
    IN_PARTICIPA_ORCAMENTO AS part_orc,
    COUNT(1) AS qt,
    SUM(VL_TRAN) AS vl_mov,
    SUM(CASE WHEN IN_PARTICIPA_CALCULO = 'S' THEN VL_TRAN ELSE 0 END) AS vl_tematico,
    SUM(CASE WHEN IN_PARTICIPA_ORCAMENTO = 'S' THEN VL_TRAN ELSE 0 END) AS vl_orcamentario
FROM vw_dashboard_transacoes_efetivas
GROUP BY CD_NTZ_CTB_TRAN, CD_CLASS_RADAR, TX_CLASS_RADAR,
         IN_CLASSIFICADA_APRESENTACAO,
         CD_CTGR_TRAN_OGNL, TX_CATEGORIA,
         IN_PARTICIPA_CALCULO, IN_PARTICIPA_ORCAMENTO
ORDER BY CD_NTZ_CTB_TRAN, CD_CLASS_RADAR, CD_CTGR_TRAN_OGNL
""")

lista_dashboard_pivot = []
lista_dashboard_transacoes = []
lista_dashboard_exatos = []
lista_dashboard_borda = []
lista_dashboard_externas = []

df_dashboard_transacoes = spark.sql("""
SELECT * FROM vw_dashboard_transacoes_efetivas
""")
df_dashboard_exatos = spark.sql("""
SELECT * FROM vw_dashboard_pares_exatos
""")
df_dashboard_borda = spark.sql("""
SELECT * FROM vw_dashboard_pares_borda
""")
df_dashboard_externas = spark.sql("""
SELECT
    t.*,
    CASE
      WHEN t.TIPO_CONSUMO = 'BORDA_CONTEXTO' THEN 'Contraparte de borda'
      WHEN t.TIPO_CONSUMO = 'EXATO_CONTEXTO' THEN 'Consumida em par exato externo'
      ELSE 'Não utilizada'
    END AS USO_CONTEXTO
FROM vw_dashboard_transacoes t
WHERE t.IN_JANELA = 'N'
""")
df_dashboard_metricas = spark.sql("""
SELECT
    (SELECT COUNT(1) FROM vw_mov_raw) AS QT_OFICIAIS,
    (SELECT COUNT(1) FROM vw_dashboard_pares_exatos WHERE IN_JANELA = 'S') AS QT_PARES_EXATOS,
    (SELECT COUNT(1) FROM vw_pares_borda_calculados) AS QT_PARES_BORDA,
    (SELECT COUNT(1) FROM vw_mov_efetivo) AS QT_EFETIVAS
""")

def empacotar_dashboard(bloco, dataframe):
    campos = [F.col(campo) for campo in dataframe.columns]
    return dataframe.select(
        F.lit(bloco).alias('BLOCO'),
        F.to_json(F.struct(*campos)).alias('PAYLOAD'),
    )

pacotes_dashboard = [
    empacotar_dashboard('PIVOT', df_dashboard_pivot),
    empacotar_dashboard('TRANSACOES', df_dashboard_transacoes),
    empacotar_dashboard('EXATOS', df_dashboard_exatos),
    empacotar_dashboard('BORDA', df_dashboard_borda),
    empacotar_dashboard('EXTERNAS', df_dashboard_externas),
    empacotar_dashboard('METRICAS', df_dashboard_metricas),
]
linhas_dashboard = reduce(lambda a, b: a.unionByName(b), pacotes_dashboard).collect()
dados_dashboard = {}
for linha in linhas_dashboard:
    dados_dashboard.setdefault(linha['BLOCO'], []).append(json.loads(linha['PAYLOAD']))

lista_dashboard_pivot = dados_dashboard.get('PIVOT', [])
lista_dashboard_transacoes = dados_dashboard.get('TRANSACOES', [])
lista_dashboard_exatos = dados_dashboard.get('EXATOS', [])
lista_dashboard_borda = dados_dashboard.get('BORDA', [])
lista_dashboard_externas = dados_dashboard.get('EXTERNAS', [])

def chave_nula(valor):
    return (valor is None, valor)

lista_dashboard_pivot.sort(key=lambda r: (
    chave_nula(r.get('ntz')), chave_nula(r.get('cd_classe')), chave_nula(r.get('cd_cat'))
))
lista_dashboard_transacoes.sort(key=lambda r: (
    chave_nula(r.get('CD_NTZ_CTB_TRAN')), chave_nula(r.get('CD_CLASS_RADAR')),
    chave_nula(r.get('CD_CTGR_TRAN_OGNL')), chave_nula(r.get('DT_TRAN')),
    chave_nula(r.get('NR_TRAN_INST_PCT')),
))
lista_dashboard_exatos.sort(key=lambda r: (
    r.get('IN_JANELA') != 'S', chave_nula(r.get('DT_TRAN')),
    chave_nula(r.get('ID_CREDITO')), chave_nula(r.get('ID_DEBITO')),
))
lista_dashboard_borda.sort(key=lambda r: (
    chave_nula(r.get('DT_TRAN_DENTRO')), chave_nula(r.get('NR_TRAN_DENTRO')),
    chave_nula(r.get('NR_TRAN_FORA')),
))
lista_dashboard_externas.sort(key=lambda r: (
    chave_nula(r.get('DT_TRAN')), chave_nula(r.get('NR_TRAN_INST_PCT')),
))

metricas_dashboard = dados_dashboard['METRICAS'][0]
qt_raw = int(metricas_dashboard['QT_OFICIAIS'])
qt_pares_exatos_oficiais = int(metricas_dashboard['QT_PARES_EXATOS'])
qt_pares_borda = int(metricas_dashboard['QT_PARES_BORDA'])
qt_efetivo = int(metricas_dashboard['QT_EFETIVAS'])
print(
    f'[RADAR_INDIVIDUAL] Reconciliação: oficiais={qt_raw}; '
    f'pares_exatos={qt_pares_exatos_oficiais}; borda={qt_pares_borda}; '
    f'efetivas={qt_efetivo}.'
)

res_dict = dict(resultados_cenarios['RENDA_PRESUMIDA'])

def esc(valor):
    return html.escape('—' if valor is None else str(valor), quote=True)

def fmt_data(valor, curta=False):
    if valor is None:
        return '—'
    if isinstance(valor, str):
        try:
            valor = datetime.date.fromisoformat(valor[:10])
        except ValueError:
            return esc(valor)
    return valor.strftime('%d/%m' if curta else '%d/%m/%Y')

def fmt_moeda(valor, sinal=False):
    if valor is None:
        return '—'
    numero = Decimal(str(valor))
    prefixo = ''
    if sinal:
        prefixo = '+ ' if numero > 0 else ('− ' if numero < 0 else '')
    numero = abs(numero) if sinal else numero
    texto = f'{numero:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')
    return f'{prefixo}R$ {texto}'

def fmt_percentual(valor):
    if valor is None:
        return '—'
    texto = f'{Decimal(str(valor)) * Decimal("100"):.2f}'.replace('.', ',')
    return f'{texto}%'

def fmt_inteiro(valor):
    return '—' if valor is None else str(int(valor))

def nome_natureza(codigo):
    return {'C': 'Crédito', 'D': 'Débito'}.get(codigo, 'Natureza não reconhecida')

def somar_metricas(linhas, campo):
    valores = [r[campo] for r in linhas if r[campo] is not None]
    return sum((Decimal(str(v)) for v in valores), Decimal('0')) if valores else None

def metricas_html(linhas):
    return (
        f'<b>{sum(int(r["qt"]) for r in linhas)}</b> tx · '
        f'Mov. <span data-private="money">{fmt_moeda(somar_metricas(linhas, "vl_mov"))}</span> · '
        f'Tema <span data-private="money">{fmt_moeda(somar_metricas(linhas, "vl_tematico"))}</span> · '
        f'Orç. <span data-private="money">{fmt_moeda(somar_metricas(linhas, "vl_orcamentario"))}</span>'
    )

def rotulo_tratamento(transacao):
    tema = transacao.get('IN_PARTICIPA_CALCULO') == 'S'
    orcamento = transacao.get('IN_PARTICIPA_ORCAMENTO') == 'S'
    classificada = transacao.get('IN_CLASSIFICADA_APRESENTACAO') == 'S'
    if tema and orcamento:
        return 'Tema + Orçamento'
    if tema:
        return 'Tema / Fora do orçamento'
    if orcamento:
        return 'Fora do tema / Orçamento'
    if classificada:
        return 'Classificada / Fora tema e orçamento'
    return 'Sem classificação / Fora tema e orçamento'

def rotulo_classe(cd_classe, tx_classe, classificada):
    if classificada != 'S' or cd_classe is None or tx_classe is None:
        return 'Sem classificação'
    return str(tx_classe)

def tabela_transacoes_html(transacoes):
    if not transacoes:
        corpo = '<tr><td colspan="7">Nenhuma transação nesta categoria.</td></tr>'
    else:
        corpo = ''.join(
            '<tr>'
            f'<td>{fmt_data(t["DT_TRAN"])}</td>'
            f'<td>{esc(nome_natureza(t["CD_NTZ_CTB_TRAN"]))}</td>'
            f'<td class="desc">{esc(t["TX_DCR_TRAN_OGNL"])}</td>'
            f'<td><code>{esc(t["NR_MCA_PCT_OPB"])}</code></td>'
            f'<td data-private="money">{fmt_moeda(t["VL_TRAN"])}</td>'
            f'<td>{esc(t["CD_TIP_MOE_CRR"])}</td>'
            f'<td>{esc(rotulo_tratamento(t))}</td>'
            '</tr>'
            for t in transacoes
        )
    return (
        '<div class="tx-wrap"><table class="tx-table"><thead><tr>'
        '<th>Data</th><th>Natureza</th>'
        '<th>Descrição original<br/><small>TX_DCR_TRAN_OGNL</small></th>'
        '<th>Banco / Marca<br/><small>NR_MCA_PCT_OPB</small></th>'
        '<th>Valor</th><th>Moeda</th><th>Tratamento</th>'
        f'</tr></thead><tbody>{corpo}</tbody></table></div>'
    )

def render_composicao():
    blocos_natureza = []
    for ntz, titulo in [('C', 'Entradas'), ('D', 'Saídas')]:
        linhas_ntz = [r for r in lista_dashboard_pivot if r['ntz'] == ntz]
        if not linhas_ntz:
            continue
        blocos_classe = []
        classes = []
        for r in linhas_ntz:
            chave = (r['cd_classe'], r['tx_classe'], r['classificada'])
            if chave not in classes:
                classes.append(chave)
        for cd_classe, tx_classe, classificada in classes:
            linhas_classe = [
                r for r in linhas_ntz
                if (r['cd_classe'], r['tx_classe'], r['classificada'])
                == (cd_classe, tx_classe, classificada)
            ]
            blocos_categoria = []
            for cat in linhas_classe:
                transacoes = [
                    t for t in lista_dashboard_transacoes
                    if t['CD_NTZ_CTB_TRAN'] == ntz
                    and t['CD_CLASS_RADAR'] == cd_classe
                    and t['IN_CLASSIFICADA_APRESENTACAO'] == classificada
                    and t['CD_CTGR_TRAN_OGNL'] == cat['cd_cat']
                ]
                codigo = '—' if cat['cd_cat'] is None else str(cat['cd_cat'])
                blocos_categoria.append(
                    '<details class="level category" data-depth="category">'
                    f'<summary><span>{esc(codigo)} — {esc(cat["tx_cat"])}</span>'
                    f'<span class="metrics">{metricas_html([cat])}</span></summary>'
                    '<div class="detail-body"><div class="mini-grid">'
                    f'<div><span>Valor movimentado</span><strong data-private="money">{fmt_moeda(cat["vl_mov"])}</strong></div>'
                    f'<div><span>Valor temático</span><strong data-private="money">{fmt_moeda(cat["vl_tematico"])}</strong></div>'
                    f'<div><span>Valor orçamentário</span><strong data-private="money">{fmt_moeda(cat["vl_orcamentario"])}</strong></div>'
                    f'</div>{tabela_transacoes_html(transacoes)}</div></details>'
                )
            blocos_classe.append(
                '<details class="level class" data-depth="class">'
                f'<summary><span>{esc(rotulo_classe(cd_classe, tx_classe, classificada))}</span>'
                f'<span class="metrics">{metricas_html(linhas_classe)}</span></summary>'
                f'<div class="detail-body">{"".join(blocos_categoria)}</div></details>'
            )
        total_orc = res_dict.get('VL_ENT_TOTAL' if ntz == 'C' else 'VL_SAI_TOTAL')
        blocos_natureza.append(
            '<details class="level nature" data-depth="nature">'
            f'<summary><span>{titulo}</span><span class="metrics">'
            f'{sum(int(r["qt"]) for r in linhas_ntz)} tx · Orçamento <span data-private="money">{fmt_moeda(total_orc)}</span></span></summary>'
            f'<div class="detail-body">{"".join(blocos_classe)}</div></details>'
        )
    if not blocos_natureza:
        return '<div class="result-note">Nenhuma transação efetiva em BRL para compor.</div>'
    return ''.join(blocos_natureza)

def lado_reconciliacao(titulo, data, ntz, valor, moeda, descricao, banco, fora=False):
    classe = 'side out' if fora else 'side'
    return (
        f'<div class="{classe}"><div class="tx-title">{esc(nome_natureza(ntz))} · {esc(titulo)}</div>'
        f'<div>{fmt_data(data)}</div><div class="tx-value">{fmt_moeda(valor)}</div><div>{esc(moeda)}</div>'
        '<dl><dt>Descrição original<span class="field-tag">TX_DCR_TRAN_OGNL</span></dt>'
        f'<dd>{esc(descricao)}</dd><dt>Banco / Marca<span class="field-tag">NR_MCA_PCT_OPB</span></dt>'
        f'<dd><code>{esc(banco)}</code></dd></dl></div>'
    )

def render_eventos_reconciliacao():
    eventos = []
    for par in lista_dashboard_exatos:
        if par['IN_JANELA'] != 'S':
            continue
        eventos.append(
            '<details class="event"><summary>'
            f'Par exato · {fmt_moeda(par["VL_TRAN"])} · {fmt_data(par["DT_TRAN"])}</summary>'
            '<div class="event-body"><div class="pair">'
            + lado_reconciliacao('Dentro do ciclo', par['DT_TRAN'], 'D', par['VL_TRAN'], par['CD_TIP_MOE_CRR'], par['DESC_DEBITO'], par['BANCO_DEBITO'])
            + '<div class="link">↔</div>'
            + lado_reconciliacao('Dentro do ciclo', par['DT_TRAN'], 'C', par['VL_TRAN'], par['CD_TIP_MOE_CRR'], par['DESC_CREDITO'], par['BANCO_CREDITO'])
            + '</div><div class="checks">✓ mesma data · ✓ mesmo valor · ✓ mesma moeda · ✓ naturezas opostas</div>'
            '<div class="result-note">Resultado: as duas movimentações foram anuladas e não participaram dos cálculos.</div>'
            '</div></details>'
        )
    for par in lista_dashboard_borda:
        eventos.append(
            '<details class="event"><summary>'
            f'Reconciliação de borda · {fmt_moeda(par["VL_TRAN"])} · diferença de {par["DIF_DIAS"]} dias</summary>'
            '<div class="event-body"><div class="pair">'
            + lado_reconciliacao('Fora do ciclo', par['DT_TRAN_FORA'], par['NTZ_FORA'], par['VL_TRAN'], par['CD_TIP_MOE_CRR'], par['DESC_FORA'], par['BANCO_FORA'], True)
            + f'<div class="link">{par["DIF_DIAS"]} dias<br/>↔</div>'
            + lado_reconciliacao('Dentro do ciclo', par['DT_TRAN_DENTRO'], par['NTZ_DENTRO'], par['VL_TRAN'], par['CD_TIP_MOE_CRR'], par['DESC_DENTRO'], par['BANCO_DENTRO'])
            + '</div><div class="checks">✓ mesmo valor · ✓ mesma moeda · ✓ naturezas opostas · ✓ diferença dentro de 5 dias</div>'
            '<div class="result-note">A movimentação oficial foi anulada. A contraparte externa foi usada somente como evidência e não entrou nos cálculos financeiros.</div>'
            '</div></details>'
        )
    return ''.join(eventos) or '<div class="result-note">Nenhum evento de reconciliação removeu movimentações oficiais.</div>'

def render_contexto_externo():
    linhas = ''.join(
        '<tr>'
        f'<td>{fmt_data(t["DT_TRAN"])}</td><td>{esc(nome_natureza(t["CD_NTZ_CTB_TRAN"]))}</td>'
        f'<td class="desc">{esc(t["TX_DCR_TRAN_OGNL"])}</td><td><code>{esc(t["NR_MCA_PCT_OPB"])}</code></td>'
        f'<td>{fmt_moeda(t["VL_TRAN"])}</td><td>{esc(t["CD_TIP_MOE_CRR"])}</td><td>{esc(t["USO_CONTEXTO"])}</td>'
        '</tr>'
        for t in lista_dashboard_externas
    )
    if not linhas:
        linhas = '<tr><td colspan="7">Nenhuma movimentação fora do ciclo.</td></tr>'
    return (
        f'<details class="event"><summary>Contexto externo analisado · {len(lista_dashboard_externas)} movimentações</summary>'
        '<div class="event-body"><div class="ext-table"><table><thead><tr>'
        '<th>Data</th><th>Natureza</th><th>Descrição original<br/><small>TX_DCR_TRAN_OGNL</small></th>'
        '<th>Banco / Marca<br/><small>NR_MCA_PCT_OPB</small></th><th>Valor</th><th>Moeda</th><th>Uso no contexto</th>'
        f'</tr></thead><tbody>{linhas}</tbody></table></div>'
        '<div class="result-note" style="margin-top:12px">Nenhuma movimentação externa participou dos cálculos financeiros do ciclo.</div>'
        '</div></details>'
    )

temas = [
    (1, 'Categorização dos Gastos', 'VL_SAI_IND', 'PC_SAI_IND', 'PC_REF_IND', 'NR_PONT_CONC_IND', None, None, 'NR_PONT_IND_FIM'),
    (2, 'Gestão de Orçamento', 'VL_SAI_ESS', 'PC_SAI_ESS', 'PC_REF_ESS', 'NR_PONT_CONC_ESS', 'NR_PONT_ORC_ESS', 'NR_PONT_PRFL_ESS', 'NR_PONT_ESS_FIM'),
    (3, 'Consumo Planejado', 'VL_SAI_NAO_ESS', 'PC_SAI_NAO_ESS', 'PC_REF_NAO_ESS', 'NR_PONT_CONC_NAO_ESS', 'NR_PONT_ORC_NAO_ESS', 'NR_PONT_PRFL_NAO_ESS', 'NR_PONT_NAO_ESS_FIM'),
    (4, 'Formação de Reserva', 'VL_SAI_FUT', 'PC_SAI_FUT', 'PC_REF_FUT', 'NR_PONT_CONC_FUT', 'NR_PONT_ORC_FUT', 'NR_PONT_PRFL_FUT', 'NR_PONT_FUT_FIM'),
    (5, 'Uso Consciente do Crédito', 'VL_SAI_OBR', 'PC_SAI_OBR', 'PC_REF_OBR', 'NR_PONT_CONC_OBR', 'NR_PONT_ORC_OBR', 'NR_PONT_PRFL_OBR', 'NR_PONT_OBR_FIM'),
]

def render_linhas_pontuacao(resultado):
    linhas = []
    for codigo, nome, vl, pc, ref, conc, orc, prfl, final in temas:
        vencedor = (
            resultado.get('CD_TEMA_VENCEDOR') == codigo
            or (
                resultado.get('CD_TEMA_VENCEDOR') == 9
                and resultado.get(final) is not None
                and resultado.get(final) == resultado.get('NR_PONT_MAX')
            )
        )
        classe = ' class="winner-row"' if vencedor else ''
        valor_final = fmt_inteiro(resultado.get(final))
        celula_final = f'<span class="score-final-badge">{valor_final}</span>' if vencedor else valor_final
        regra_final = 'Final = Concentração' if codigo == 1 else 'Final = Concentração + Orçamento + Perfil'
        linhas.append(
            f'<tr{classe}><td class="theme-cell">{esc(nome)}<small>{esc(regra_final)}</small></td>'
            f'<td class="num" data-private="money">{fmt_moeda(resultado.get(vl))}</td>'
            f'<td class="num">{fmt_percentual(resultado.get(pc))}</td>'
            f'<td class="num">{fmt_percentual(resultado.get(ref))}</td>'
            f'<td class="num">{fmt_inteiro(resultado.get(conc))}</td>'
            f'<td class="num">{fmt_inteiro(resultado.get(orc)) if orc else "—"}</td>'
            f'<td class="num">{fmt_inteiro(resultado.get(prfl)) if prfl else "—"}</td>'
            f'<td class="final-cell">{celula_final}</td></tr>'
        )
    return ''.join(linhas)

def tag_execucao(rotulo, valor, positivo='S'):
    classe = ' ok' if valor == positivo else ''
    prefixo = '✓ ' if classe else ''
    exibido = {'S': 'Sim', 'N': 'Não', None: '—'}.get(valor, str(valor))
    return f'<span class="run-tag{classe}">{prefixo}{esc(rotulo)}: {esc(exibido)}</span>'

dt_contexto_ini = dt_ini_j - timedelta(days=DIAS_CONTEXTO_RECONCILIACAO) if dt_ini_j else None
dt_contexto_fim = dt_fim_j + timedelta(days=DIAS_CONTEXTO_RECONCILIACAO) if dt_fim_j else None
fallback_usado = (
    '—' if res_dict.get('DD_INC_MM_CLC_BLC_FALLBACK') is None
    else ('Sim' if res_dict.get('DD_INC_MM_CLC_BLC') is None else 'Não')
)
def texto_payload(valor):
    return '—' if valor is None else str(valor)

def montar_payload_cenario(chave, resultado):
    presumida = chave == 'RENDA_PRESUMIDA'
    base_financeira = bases_cenarios[chave]
    pc_saida_entrada_cenario = resultado.get('PC_SAI_ENT')
    largura_cenario = (
        0 if pc_saida_entrada_cenario is None
        else max(0, min(100, float(pc_saida_entrada_cenario) * 100))
    )
    rotulo = 'Renda Presumida' if presumida else 'Entradas Realizadas'
    referencia_rotulo = 'Referência da renda' if presumida else 'Origem da base'
    referencia_valor = (
        fmt_data(resultado.get('DT_REN_PRES_REF'))
        if presumida
        else f'Entradas efetivas do ciclo · {fmt_data(resultado.get("DT_REF_INI"), True)} → {fmt_data(resultado.get("DT_REF_FIM"), True)}'
    )
    return {
        'base_kicker': 'Base financeira',
        'base_badge': f'{rotulo} ativa',
        'valor_base': fmt_moeda(base_financeira),
        'referencia_rotulo': referencia_rotulo,
        'referencia_valor': referencia_valor,
        'entrada_rotulo': 'Base financeira',
        'entrada_valor': fmt_moeda(base_financeira),
        'saldo_status': texto_payload(resultado.get('TX_STS_FINAL')),
        'saldo_valor': fmt_moeda(resultado.get('VL_RES_ORC'), True),
        'razao_valor': fmt_percentual(pc_saida_entrada_cenario),
        'largura_medidor': f'{largura_cenario:.2f}%',
        'pontuacao_html': render_linhas_pontuacao(resultado),
        'tag_base_html': f'<span class="run-tag base-active">Base ativa: {esc(rotulo)}</span>',
        'tag_pontuacao_html': tag_execucao('Pontuação completa', resultado.get('FL_PONTUACAO_COMPLETA')),
    }

payload_cenarios = {
    chave: montar_payload_cenario(chave, resultados_cenarios[chave])
    for chave in ('RENDA_PRESUMIDA', 'ENTRADAS_REALIZADAS')
}
payload_padrao = payload_cenarios['RENDA_PRESUMIDA']
payload_cenarios_json = (
    json.dumps(payload_cenarios, ensure_ascii=False, separators=(',', ':'))
    .replace('&', '\\u0026')
    .replace('<', '\\u003c')
    .replace('>', '\\u003e')
)

CSS_COMPLEMENTO = r"""
[data-radar-root="individual"] .base-switch{display:grid;grid-template-columns:1fr 1fr;gap:4px;margin-top:12px;padding:4px;background:#f0f3f8;border:1px solid var(--line);border-radius:12px}
[data-radar-root="individual"] .base-switch-btn{border:0;background:transparent;color:var(--muted);border-radius:9px;padding:7px 6px;font-size:9px;font-weight:850;line-height:1.15;cursor:pointer}
[data-radar-root="individual"] .base-switch-btn.active{background:#fff;color:var(--bb-blue-deep);box-shadow:0 2px 8px rgba(27,36,74,.12)}
[data-radar-root="individual"] .base-switch-btn:focus-visible{outline:3px solid rgba(70,94,255,.2);outline-offset:1px}
[data-radar-root="individual"] .scenario-active-tag{display:inline-flex;align-items:center;border:1px solid #cce7da;background:var(--good-bg);color:var(--good);border-radius:999px;padding:5px 8px;font-size:9px;font-weight:850}
[data-radar-root="individual"] .run-tag.base-active{background:#eef8f3;border-color:#cce7da;color:#176848}
"""

CSS_APROVADO = '[data-radar-root="individual"]{\n  --bb-blue:#465eff;--bb-blue-deep:#252d84;--bb-yellow:#fcfc30;--ink:#13162b;--muted:#667085;\n  --bg:#f4f6fb;--card:#fff;--line:#e3e7ef;--soft:#f8f9fc;--good:#087a55;--good-bg:#e8f7f0;\n  --warn:#8c5a00;--warn-bg:#fff5d9;--danger:#b42318;--danger-bg:#fff0ee;--shadow:0 14px 40px rgba(27,36,74,.08);\n  --radius:22px;--radius-sm:14px;--max:1280px\n}\n[data-radar-root="individual"],[data-radar-root="individual"] *{box-sizing:border-box}\n[data-radar-root="individual"]{scroll-behavior:smooth}\n[data-radar-root="individual"]{margin:0;background:var(--bg);color:var(--ink);font-family:Inter,ui-sans-serif,system-ui,-apple-system,"Segoe UI",Arial,sans-serif;line-height:1.45}\n[data-radar-root="individual"] button,[data-radar-root="individual"] input{font:inherit}\n[data-radar-root="individual"] a{color:inherit}\n[data-radar-root="individual"] .shell{max-width:var(--max);margin:auto;padding:0 24px 84px}\n[data-radar-root="individual"] .skip{position:absolute;left:-9999px;top:auto}\n[data-radar-root="individual"] .skip:focus{left:16px;top:16px;z-index:9999;background:#fff;padding:10px 14px;border-radius:10px}\n[data-radar-root="individual"] .topbar{position:sticky;top:0;z-index:50;background:rgba(244,246,251,.88);backdrop-filter:blur(18px);border-bottom:1px solid rgba(227,231,239,.8)}\n[data-radar-root="individual"] .topbar-inner{max-width:var(--max);margin:auto;padding:12px 24px;display:flex;gap:14px;align-items:center;justify-content:space-between}\n[data-radar-root="individual"] .brand{display:flex;align-items:center;gap:10px;font-weight:850;white-space:nowrap}\n[data-radar-root="individual"] .brand-mark{width:28px;height:28px;border-radius:9px;background:var(--bb-yellow);border:7px solid var(--bb-blue);transform:rotate(45deg);box-shadow:inset 0 0 0 2px #fff}\n[data-radar-root="individual"] .nav{display:flex;gap:4px;overflow:auto;scrollbar-width:none}\n[data-radar-root="individual"] .nav::-webkit-scrollbar{display:none}\n[data-radar-root="individual"] .nav a{text-decoration:none;color:#525b75;padding:8px 10px;border-radius:10px;font-size:12px;font-weight:720;white-space:nowrap}\n[data-radar-root="individual"] .nav a:hover,[data-radar-root="individual"] .nav a.active{background:#fff;color:var(--bb-blue-deep);box-shadow:0 1px 4px rgba(0,0,0,.06)}\n[data-radar-root="individual"] .privacy{border:1px solid var(--line);background:#fff;border-radius:999px;padding:8px 11px;font-weight:750;font-size:12px;cursor:pointer;white-space:nowrap}\n[data-radar-root="individual"] .hero{margin-top:24px;background:linear-gradient(135deg,var(--bb-blue-deep),#3342b6 52%,var(--bb-blue));color:#fff;border-radius:28px;padding:26px 28px;box-shadow:0 20px 60px rgba(37,45,132,.18);position:relative;overflow:hidden}\n[data-radar-root="individual"] .hero:after{content:"";position:absolute;width:320px;height:320px;border-radius:50%;background:var(--bb-yellow);right:-165px;top:-175px;opacity:.96}\n[data-radar-root="individual"] .hero-grid{display:grid;grid-template-columns:1.4fr .8fr;gap:28px;position:relative;z-index:1}\n[data-radar-root="individual"] .eyebrow{font-size:10px;font-weight:850;letter-spacing:.12em;text-transform:uppercase;opacity:.72}\n[data-radar-root="individual"] .hero h1{margin:7px 0 14px;font-size:clamp(30px,4vw,48px);line-height:1.02;letter-spacing:-.04em}\n[data-radar-root="individual"] .hero p{margin:0;color:rgba(255,255,255,.76);max-width:760px}\n[data-radar-root="individual"] .hero-status{align-self:end;background:rgba(255,255,255,.1);border:1px solid rgba(255,255,255,.18);padding:18px;border-radius:18px}\n[data-radar-root="individual"] .hero-status small{display:block;color:rgba(255,255,255,.7)}\n[data-radar-root="individual"] .hero-status strong{font-size:21px;display:block;margin:3px 0}\n[data-radar-root="individual"] .hero-status span{font-size:12px}\n[data-radar-root="individual"] .hero-chips{display:flex;gap:7px;flex-wrap:wrap;margin-top:0}\n[data-radar-root="individual"] .chip{border:1px solid rgba(255,255,255,.2);background:rgba(255,255,255,.1);padding:6px 9px;border-radius:999px;font-size:10px;font-weight:720}\n[data-radar-root="individual"] .context-strip{display:flex;align-items:stretch;gap:0;background:#fff;border:1px solid var(--line);border-radius:18px;overflow:hidden}\n[data-radar-root="individual"] .context-item{flex:1;min-width:0;padding:14px 16px;border-right:1px solid var(--line)}\n[data-radar-root="individual"] .context-item:last-child{border-right:0}\n[data-radar-root="individual"] .context-label{font-size:9px;color:var(--muted);font-weight:850;text-transform:uppercase;letter-spacing:.07em}\n[data-radar-root="individual"] .context-value{font-size:14px;font-weight:820;margin-top:4px;overflow-wrap:anywhere}\n[data-radar-root="individual"] .context-meta{font-size:9px;color:var(--muted);margin-top:2px}\n[data-radar-root="individual"] .money-grid{display:grid;grid-template-columns:1fr auto 1fr auto 1.15fr;gap:0;align-items:stretch;background:#fff;border:1px solid var(--line);border-radius:20px;overflow:hidden}\n[data-radar-root="individual"] .money-card{background:#fff;padding:20px 22px;border-right:1px solid var(--line)}\n[data-radar-root="individual"] .money-card:last-child{border-right:0}\n[data-radar-root="individual"] .money-card .label{font-size:10px;color:var(--muted);text-transform:uppercase;font-weight:850;letter-spacing:.08em}\n[data-radar-root="individual"] .money-card .value{font-size:clamp(24px,3vw,36px);font-weight:900;margin:5px 0;letter-spacing:-.04em}\n[data-radar-root="individual"] .money-card .meta{font-size:11px;color:var(--muted)}\n[data-radar-root="individual"] .operator{align-self:center;font-size:22px;color:#a7afc3;font-weight:300;padding:0 10px}\n[data-radar-root="individual"] .money-card.balance-card{background:#f6fbf8}\n[data-radar-root="individual"] .balance-card .value{color:var(--good)}\n[data-radar-root="individual"] .status-pill{display:inline-flex;margin-top:10px;background:var(--good-bg);color:var(--good);padding:7px 10px;border-radius:999px;font-size:11px;font-weight:850}\n[data-radar-root="individual"] .ratio-bar{margin-top:12px;height:8px;background:#e9edf4;border-radius:999px;overflow:hidden}\n[data-radar-root="individual"] .ratio-bar span{display:block;height:100%;width:72.39%;background:var(--bb-blue)}\n[data-radar-root="individual"] .trace-box{display:grid;grid-template-columns:1.25fr .75fr;gap:12px;margin-top:12px}\n[data-radar-root="individual"] .trace-main,[data-radar-root="individual"] .trace-side{background:#fff;border:1px solid var(--line);border-radius:20px;padding:18px}\n[data-radar-root="individual"] .trace-main h3,[data-radar-root="individual"] .trace-side h3{margin:0 0 7px;font-size:15px}\n[data-radar-root="individual"] .trace-main p,[data-radar-root="individual"] .trace-side p{margin:0;color:var(--muted);font-size:12px}\n[data-radar-root="individual"] .formula{display:flex;gap:8px;align-items:center;flex-wrap:wrap;margin-top:15px}\n[data-radar-root="individual"] .formula span{background:var(--soft);border:1px solid var(--line);border-radius:11px;padding:8px 10px;font-size:11px;font-weight:800}\n[data-radar-root="individual"] .formula b{color:var(--muted)}\n[data-radar-root="individual"] .controls{display:flex;gap:8px;flex-wrap:wrap;align-items:center}\n[data-radar-root="individual"] .control-btn{border:1px solid var(--line);background:#fff;border-radius:11px;padding:8px 11px;font-size:11px;font-weight:750;cursor:pointer}\n[data-radar-root="individual"] .control-btn:hover{border-color:#bbc3d4}\n[data-radar-root="individual"] .search{border:1px solid var(--line);background:#fff;border-radius:11px;padding:8px 11px;font-size:11px;min-width:220px;outline:none}\n[data-radar-root="individual"] .search:focus{border-color:var(--bb-blue);box-shadow:0 0 0 3px rgba(70,94,255,.12)}\n[data-radar-root="individual"] .explorer{background:#fff;border:1px solid var(--line);border-radius:22px;padding:10px;box-shadow:var(--shadow)}\n[data-radar-root="individual"] .level{background:#fff;border:1px solid var(--line);border-radius:14px;margin:8px 0;overflow:hidden}\n[data-radar-root="individual"] .level summary{cursor:pointer;list-style:none;padding:14px 15px;display:flex;justify-content:space-between;gap:16px;align-items:center}\n[data-radar-root="individual"] .level summary::-webkit-details-marker{display:none}\n[data-radar-root="individual"] .level summary:before{content:"+";display:grid;place-items:center;flex:0 0 24px;height:24px;border-radius:8px;background:#eef0ff;color:var(--bb-blue-deep);font-weight:900}\n[data-radar-root="individual"] .level[open]>summary:before{content:"−"}\n[data-radar-root="individual"] .level>summary{font-weight:750}\n[data-radar-root="individual"] .nature>summary{background:#f4f5ff;font-size:15px}\n[data-radar-root="individual"] .class>summary{background:#fafbff}\n[data-radar-root="individual"] .category>summary{font-weight:650}\n[data-radar-root="individual"] .metrics{font-size:11px;color:var(--muted);font-weight:500;text-align:right;margin-left:auto}\n[data-radar-root="individual"] .detail-body{padding:0 13px 13px 26px}\n[data-radar-root="individual"] .mini-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:8px;margin:8px 0 12px}\n[data-radar-root="individual"] .mini-grid div{background:#f8f9fc;border-radius:11px;padding:10px}\n[data-radar-root="individual"] .mini-grid span{font-size:9px;text-transform:uppercase;color:var(--muted);display:block}\n[data-radar-root="individual"] .mini-grid strong{font-size:13px}\n[data-radar-root="individual"] .tx-wrap,[data-radar-root="individual"] .ext-table{overflow-x:auto;border-radius:12px;border:1px solid var(--line);background:#fff}\n[data-radar-root="individual"] table{width:100%;border-collapse:collapse}\n[data-radar-root="individual"] th,[data-radar-root="individual"] td{padding:10px;border-bottom:1px solid var(--line);font-size:11px;text-align:left;vertical-align:top}\n[data-radar-root="individual"] th{font-size:9px;text-transform:uppercase;letter-spacing:.04em;color:var(--muted);background:#f8f9fc;position:sticky;top:0}\n[data-radar-root="individual"] th small{display:block;font-size:8px;letter-spacing:0;text-transform:none;color:#98a2b3}\n[data-radar-root="individual"] td.desc{min-width:220px;max-width:360px;white-space:normal}\n[data-radar-root="individual"] td code{font-size:10px;white-space:nowrap}\n[data-radar-root="individual"] .sim-note{font-size:10px;color:var(--muted);font-style:italic}\n[data-radar-root="individual"] .score-layout{display:grid;grid-template-columns:1.35fr .65fr;gap:12px}\n[data-radar-root="individual"] .score-list{background:#fff;border:1px solid var(--line);border-radius:18px;overflow:hidden}\n[data-radar-root="individual"] .score-item{background:#fff;padding:15px 16px;border-bottom:1px solid var(--line)}\n[data-radar-root="individual"] .score-item:last-of-type{border-bottom:0}\n[data-radar-root="individual"] .score-head{display:flex;justify-content:space-between;gap:14px;align-items:start}\n[data-radar-root="individual"] .score-head strong{display:block;font-size:13px}\n[data-radar-root="individual"] .score-head span{display:block;font-size:10px;color:var(--muted);margin-top:2px}\n[data-radar-root="individual"] .score-head>b{font-size:21px}\n[data-radar-root="individual"] .score-track{height:7px;background:#edf0f5;border-radius:999px;overflow:hidden;margin:11px 0 8px}\n[data-radar-root="individual"] .score-track span{display:block;height:100%;background:var(--bb-blue);min-width:0}\n[data-radar-root="individual"] .score-memory{display:flex;gap:12px;flex-wrap:wrap;color:var(--muted);font-size:10px}\n[data-radar-root="individual"] .score-memory b{color:var(--ink)}\n[data-radar-root="individual"] .winner-card{background:var(--bb-yellow);border:1px solid #e2df4f;border-radius:18px;padding:20px;position:sticky;top:84px;align-self:start}\n[data-radar-root="individual"] .winner-card .kicker{color:#333}\n[data-radar-root="individual"] .winner-card h3{font-size:25px;margin:6px 0;letter-spacing:-.03em}\n[data-radar-root="individual"] .winner-card .big{font-size:34px;font-weight:900}\n[data-radar-root="individual"] .winner-card p{font-size:11px;margin:8px 0 0;max-width:300px}\n[data-radar-root="individual"] .rule-note{background:#fff8df;border:1px solid #f1df9b;color:#725000;padding:10px 12px;border-radius:11px;font-size:10px;margin-top:10px}\n[data-radar-root="individual"] .recon{background:#fff;border:1px solid var(--line);border-radius:22px;padding:16px;box-shadow:var(--shadow)}\n[data-radar-root="individual"] .funnel{display:flex;align-items:center;justify-content:center;gap:9px;flex-wrap:wrap;margin:4px 0 16px}\n[data-radar-root="individual"] .step{padding:10px 13px;border-radius:11px;background:#f5f7fb;text-align:center;font-size:10px}\n[data-radar-root="individual"] .step b{font-size:18px;display:block}\n[data-radar-root="individual"] .minus{color:var(--danger)}\n[data-radar-root="individual"] .event{border:1px solid var(--line);border-radius:14px;margin:9px 0;overflow:hidden}\n[data-radar-root="individual"] .event summary{cursor:pointer;padding:13px 15px;font-weight:750;background:#fafbfe}\n[data-radar-root="individual"] .event-body{padding:13px 15px}\n[data-radar-root="individual"] .pair{display:grid;grid-template-columns:1fr 58px 1fr;gap:9px;align-items:stretch}\n[data-radar-root="individual"] .side{background:#f7f9fc;border-radius:12px;padding:13px}\n[data-radar-root="individual"] .side.out{background:var(--warn-bg)}\n[data-radar-root="individual"] .link{text-align:center;color:var(--muted);font-weight:850;align-self:center}\n[data-radar-root="individual"] .side .tx-title{font-size:13px;font-weight:850;margin-bottom:6px}\n[data-radar-root="individual"] .side .tx-value{font-size:18px;font-weight:850;margin:4px 0}\n[data-radar-root="individual"] .side dl{display:grid;grid-template-columns:120px 1fr;gap:5px 8px;margin:11px 0 0;font-size:10px}\n[data-radar-root="individual"] .side dt{color:var(--muted)}\n[data-radar-root="individual"] .side dd{margin:0;font-weight:650;word-break:break-word}\n[data-radar-root="individual"] .field-tag{font-size:7px;color:#8b98aa;display:block}\n[data-radar-root="individual"] .checks{margin:11px 0;font-size:10px;color:var(--muted)}\n[data-radar-root="individual"] .result-note{background:var(--good-bg);color:#155e3c;padding:10px;border-radius:10px;font-size:10px}\n[data-radar-root="individual"] .tech{background:#fff;border:1px solid var(--line);border-radius:18px;padding:0;overflow:hidden}\n[data-radar-root="individual"] .tech>summary{cursor:pointer;font-weight:820;padding:15px 17px;background:#fbfcfe}\n[data-radar-root="individual"] .tech .scroll{max-height:620px;overflow:auto;margin:0;border-top:1px solid var(--line)}\n[data-radar-root="individual"] .privacy-on [data-sensitive="true"]{filter:blur(5px);user-select:none}\n[data-radar-root="individual"] .privacy-on .tech tbody tr[data-private="true"] td:last-child{filter:blur(5px);user-select:none}\n[data-radar-root="individual"] mark{background:var(--bb-yellow);color:inherit;padding:0 .08em}\n[data-radar-root="individual"] .section{margin-top:24px}\n[data-radar-root="individual"] .section-head{display:flex;align-items:center;justify-content:space-between;gap:14px;margin-bottom:10px}\n[data-radar-root="individual"] .section-head h2{font-size:18px;letter-spacing:-.02em;margin:0}\n[data-radar-root="individual"] .run-header{margin-top:24px;background:#fff;border:1px solid var(--line);border-radius:20px;padding:17px 19px;display:flex;align-items:center;justify-content:space-between;gap:18px;box-shadow:0 8px 28px rgba(27,36,74,.05)}\n[data-radar-root="individual"] .run-title{display:flex;align-items:baseline;gap:10px;white-space:nowrap}\n[data-radar-root="individual"] .run-title .eyebrow{color:var(--muted);opacity:1}\n[data-radar-root="individual"] .run-title h1{font-size:22px;line-height:1;margin:0;letter-spacing:-.035em}\n[data-radar-root="individual"] .run-tags{display:flex;justify-content:flex-end;gap:7px;flex-wrap:wrap}\n[data-radar-root="individual"] .run-tag{display:inline-flex;align-items:center;min-height:28px;padding:5px 9px;border-radius:999px;border:1px solid var(--line);background:var(--soft);font-size:10px;font-weight:780;color:#4c556f;white-space:nowrap}\n[data-radar-root="individual"] .run-tag.ok{background:#f2f8f5;border-color:#d6eadf;color:#176848}\n[data-radar-root="individual"] .context-grid{display:grid;grid-template-columns:repeat(12,minmax(0,1fr));gap:10px}\n[data-radar-root="individual"] .context-card{position:relative;background:#fff;border:1px solid var(--line);border-radius:18px;padding:16px 17px;min-height:138px;overflow:hidden;box-shadow:0 5px 18px rgba(27,36,74,.035)}\n[data-radar-root="individual"] .context-card:before{content:"";position:absolute;left:0;top:0;bottom:0;width:3px;background:#dfe3ff}\n[data-radar-root="individual"] .context-card:hover{border-color:#d4d9e6;box-shadow:0 10px 28px rgba(27,36,74,.065)}\n[data-radar-root="individual"] .identity-card{grid-column:span 4;min-height:174px}\n[data-radar-root="individual"] .identity-card:before{background:var(--bb-blue)}\n[data-radar-root="individual"] .cycle-card{grid-column:span 8;min-height:174px}\n[data-radar-root="individual"] .cycle-card:before{background:#8090ff}\n[data-radar-root="individual"] .income-card{grid-column:span 3}\n[data-radar-root="individual"] .income-card:before{background:#22a06b}\n[data-radar-root="individual"] .profile-card{grid-column:span 3}\n[data-radar-root="individual"] .profile-card:before{background:#8c7ae6}\n[data-radar-root="individual"] .coverage-card{grid-column:span 3}\n[data-radar-root="individual"] .coverage-card:before{background:#19a47b}\n[data-radar-root="individual"] .window-card{grid-column:span 3}\n[data-radar-root="individual"] .window-card:before{background:#5a84d6}\n[data-radar-root="individual"] .context-top{display:flex;align-items:center;justify-content:space-between;gap:10px}\n[data-radar-root="individual"] .context-kicker{font-size:11px;font-weight:880;letter-spacing:.08em;text-transform:uppercase;color:var(--muted)}\n[data-radar-root="individual"] .context-dot{width:8px;height:8px;border-radius:50%;background:var(--bb-yellow);box-shadow:0 0 0 4px #fffbd3}\n[data-radar-root="individual"] .context-badge{font-size:11px;font-weight:820;padding:4px 7px;border-radius:999px;background:#f0f2ff;color:var(--bb-blue-deep);white-space:nowrap}\n[data-radar-root="individual"] .context-badge.subtle{background:#f5f6f9;color:var(--muted)}\n[data-radar-root="individual"] .context-badge.good{background:var(--good-bg);color:var(--good)}\n[data-radar-root="individual"] .context-main{font-size:23px;font-weight:900;letter-spacing:-.035em;margin:12px 0 12px;line-height:1.05}\n[data-radar-root="individual"] .context-main small{font-size:11px;font-weight:720;letter-spacing:0;color:var(--muted);margin-left:3px}\n[data-radar-root="individual"] .date-range span{color:#a2a9ba;font-weight:500;margin:0 4px}\n[data-radar-root="individual"] .context-labels{display:flex;gap:6px;flex-wrap:wrap;margin-top:10px}\n[data-radar-root="individual"] .micro-label{display:inline-flex;align-items:center;gap:5px;padding:6px 8px;border:1px solid #edf0f5;border-radius:8px;background:#fafbfe;font-size:11px;color:#596276;font-weight:720}\n[data-radar-root="individual"] .micro-label b{color:var(--ink);font-weight:840}\n[data-radar-root="individual"] .context-data{display:grid;grid-template-columns:1fr 1fr;gap:8px;margin-top:12px}\n[data-radar-root="individual"] .context-data.single{grid-template-columns:1fr}\n[data-radar-root="individual"] .data-cell{padding-top:9px;border-top:1px solid #eef0f4}\n[data-radar-root="individual"] .data-cell span{display:block;font-size:10px;text-transform:uppercase;letter-spacing:.06em;color:#98a2b3;font-weight:820}\n[data-radar-root="individual"] .data-cell strong{display:block;margin-top:3px;font-size:11px;line-height:1.25;overflow-wrap:anywhere}\n[data-radar-root="individual"] .context-foot{margin-top:8px;color:#8a93a7;font-size:10px;white-space:normal}\n[data-radar-root="individual"] .section-head>div:first-child{min-width:0}\n[data-radar-root="individual"] .section-label{display:block;font-size:9px;font-weight:900;letter-spacing:.11em;text-transform:uppercase;color:#98a2b3;margin-bottom:2px}\n[data-radar-root="individual"] .context-card,[data-radar-root="individual"] .finance-card,[data-radar-root="individual"] .score-card-radar{padding:0}\n[data-radar-root="individual"] .context-card>summary,[data-radar-root="individual"] .finance-card>summary,[data-radar-root="individual"] .score-card-radar>summary{list-style:none;cursor:pointer;padding:15px 16px;display:flex;align-items:center;justify-content:space-between;gap:12px;min-height:76px}\n[data-radar-root="individual"] .context-card>summary::-webkit-details-marker,[data-radar-root="individual"] .finance-card>summary::-webkit-details-marker,[data-radar-root="individual"] .score-card-radar>summary::-webkit-details-marker{display:none}\n[data-radar-root="individual"] .context-card>summary:after,[data-radar-root="individual"] .finance-card>summary:after,[data-radar-root="individual"] .score-card-radar>summary:after{content:"+";display:grid;place-items:center;flex:0 0 25px;height:25px;border-radius:8px;background:#eef0ff;color:var(--bb-blue-deep);font-weight:900}\n[data-radar-root="individual"] .context-card[open]>summary:after,[data-radar-root="individual"] .finance-card[open]>summary:after,[data-radar-root="individual"] .score-card-radar[open]>summary:after{content:"−"}\n[data-radar-root="individual"] .card-summary-main{min-width:0;flex:1}\n[data-radar-root="individual"] .card-summary-row{display:flex;align-items:center;gap:8px;flex-wrap:wrap}\n[data-radar-root="individual"] .card-summary-value{font-size:20px;font-weight:900;letter-spacing:-.035em;line-height:1.1;margin-top:5px;overflow-wrap:anywhere}\n[data-radar-root="individual"] .card-summary-value.good{color:var(--good)}\n[data-radar-root="individual"] .card-body{border-top:1px solid #edf0f4;padding:12px 16px 15px}\n[data-radar-root="individual"] .context-card .context-labels,[data-radar-root="individual"] .context-card .context-data,[data-radar-root="individual"] .context-card .context-foot{margin-top:0}\n[data-radar-root="individual"] .context-card .context-data{margin-top:10px}\n[data-radar-root="individual"] .context-card .context-foot{padding-top:9px}\n[data-radar-root="individual"] .finance-card .card-labels{margin-top:0}\n[data-radar-root="individual"] .finance-card .meter{margin-top:4px}\n[data-radar-root="individual"] .score-card-radar .score-data{margin-top:0}\n[data-radar-root="individual"] .score-card-radar .score-parts{margin-top:10px}\n[data-radar-root="individual"] .score-card-radar.featured>summary{background:linear-gradient(135deg,#fffef0,#fff)}\n[data-radar-root="individual"] .finance-grid{display:grid;grid-template-columns:repeat(12,minmax(0,1fr));gap:10px}\n[data-radar-root="individual"] .finance-card,[data-radar-root="individual"] .score-card-radar{position:relative;background:#fff;border:1px solid var(--line);border-radius:18px;padding:16px 17px;overflow:hidden;box-shadow:0 5px 18px rgba(27,36,74,.035);transition:.18s ease}\n[data-radar-root="individual"] .finance-card:before,[data-radar-root="individual"] .score-card-radar:before{content:"";position:absolute;left:0;top:0;bottom:0;width:3px;background:#dfe3ff}\n[data-radar-root="individual"] .finance-card:hover,[data-radar-root="individual"] .score-card-radar:hover{border-color:#d4d9e6;box-shadow:0 10px 28px rgba(27,36,74,.065);transform:translateY(-1px)}\n[data-radar-root="individual"] .finance-in{grid-column:span 3}\n[data-radar-root="individual"] .finance-out{grid-column:span 3}\n[data-radar-root="individual"] .finance-balance{grid-column:span 4;background:#f8fcfa}\n[data-radar-root="individual"] .finance-ratio{grid-column:span 2}\n[data-radar-root="individual"] .finance-balance:before{background:#8fd8bd}\n[data-radar-root="individual"] .finance-ratio:before{background:#b9c1ff}\n[data-radar-root="individual"] .card-top{display:flex;align-items:center;justify-content:space-between;gap:10px}\n[data-radar-root="individual"] .card-kicker{font-size:9px;text-transform:uppercase;letter-spacing:.08em;color:var(--muted);font-weight:900}\n[data-radar-root="individual"] .card-badge{padding:4px 7px;border-radius:999px;background:#f1f3ff;color:var(--bb-blue-deep);font-size:8px;font-weight:850;white-space:nowrap}\n[data-radar-root="individual"] .card-badge.good{background:var(--good-bg);color:var(--good)}\n[data-radar-root="individual"] .card-value{font-size:clamp(24px,3vw,34px);font-weight:900;letter-spacing:-.04em;line-height:1.05;margin:14px 0 12px}\n[data-radar-root="individual"] .finance-balance .card-value{color:var(--good)}\n[data-radar-root="individual"] .card-labels{display:flex;gap:6px;flex-wrap:wrap}\n[data-radar-root="individual"] .card-label{display:inline-flex;gap:4px;align-items:center;background:#f7f8fb;border:1px solid #eceff4;border-radius:8px;padding:5px 7px;font-size:9px;color:var(--muted)}\n[data-radar-root="individual"] .card-label b{color:var(--ink)}\n[data-radar-root="individual"] .meter{height:7px;background:#edf0f5;border-radius:999px;overflow:hidden;margin:12px 0 7px}\n[data-radar-root="individual"] .meter span{display:block;height:100%;background:var(--bb-blue);width:72.39%}\n[data-radar-root="individual"] .score-grid-radar{display:grid;grid-template-columns:repeat(12,minmax(0,1fr));gap:10px}\n[data-radar-root="individual"] .score-card-radar{grid-column:span 3;min-height:190px}\n[data-radar-root="individual"] .score-card-radar.featured{grid-column:span 6;background:linear-gradient(135deg,#fffef0,#fff);border-color:#e8e584}\n[data-radar-root="individual"] .score-card-radar.featured:before{background:var(--bb-yellow);width:5px}\n[data-radar-root="individual"] .score-card-radar.wide{grid-column:span 3}\n[data-radar-root="individual"] .score-title{font-size:13px;font-weight:850;line-height:1.2;max-width:220px}\n[data-radar-root="individual"] .score-final{font-size:36px;font-weight:950;letter-spacing:-.05em;line-height:1}\n[data-radar-root="individual"] .score-final small{font-size:9px;color:var(--muted);font-weight:850;letter-spacing:.04em;text-transform:uppercase;display:block;margin-bottom:3px}\n[data-radar-root="individual"] .score-data{display:grid;grid-template-columns:repeat(3,1fr);gap:6px;margin-top:14px}\n[data-radar-root="individual"] .score-data div{border-top:1px solid #edf0f4;padding-top:8px}\n[data-radar-root="individual"] .score-data span{display:block;font-size:8px;text-transform:uppercase;letter-spacing:.05em;color:#98a2b3;font-weight:850}\n[data-radar-root="individual"] .score-data b{display:block;font-size:11px;margin-top:2px}\n[data-radar-root="individual"] .score-parts{display:flex;gap:6px;flex-wrap:wrap;margin-top:12px}\n[data-radar-root="individual"] .score-part{background:#f7f8fb;border:1px solid #eceff4;border-radius:8px;padding:5px 7px;font-size:9px;color:var(--muted)}\n[data-radar-root="individual"] .score-part b{color:var(--ink)}\n[data-radar-root="individual"] .score-rule{margin-top:10px;font-size:9px;color:#725000;background:#fff8df;border:1px solid #f1df9b;border-radius:9px;padding:7px 8px}\n@media(max-width:1050px){[data-radar-root="individual"] .finance-in,[data-radar-root="individual"] .finance-out{grid-column:span 3}\n[data-radar-root="individual"] .finance-balance,[data-radar-root="individual"] .finance-ratio{grid-column:span 6}\n[data-radar-root="individual"] .score-card-radar,[data-radar-root="individual"] .score-card-radar.featured,[data-radar-root="individual"] .score-card-radar.wide{grid-column:span 6}\n[data-radar-root="individual"] .context-grid{grid-template-columns:repeat(6,minmax(0,1fr))}\n[data-radar-root="individual"] .identity-card,[data-radar-root="individual"] .cycle-card{grid-column:span 3}\n[data-radar-root="individual"] .income-card,[data-radar-root="individual"] .profile-card,[data-radar-root="individual"] .coverage-card,[data-radar-root="individual"] .window-card{grid-column:span 3}\n[data-radar-root="individual"] .run-header{align-items:flex-start}\n[data-radar-root="individual"] .run-tags{justify-content:flex-start}\n[data-radar-root="individual"] .money-grid{grid-template-columns:1fr 30px 1fr}\n[data-radar-root="individual"] .money-grid .operator:nth-of-type(2){display:none}\n[data-radar-root="individual"] .money-grid .balance-card{grid-column:1/-1}\n[data-radar-root="individual"] .score-layout{grid-template-columns:1fr}\n[data-radar-root="individual"] .winner-card{position:static}\n[data-radar-root="individual"] .hero-grid{grid-template-columns:1fr}}\n@media(max-width:760px){[data-radar-root="individual"] .finance-in,[data-radar-root="individual"] .finance-out,[data-radar-root="individual"] .finance-balance,[data-radar-root="individual"] .finance-ratio,[data-radar-root="individual"] .score-card-radar,[data-radar-root="individual"] .score-card-radar.featured,[data-radar-root="individual"] .score-card-radar.wide{grid-column:1/-1}\n[data-radar-root="individual"] .score-data{grid-template-columns:repeat(3,1fr)}\n[data-radar-root="individual"] .shell{padding:0 13px 60px}\n[data-radar-root="individual"] .topbar-inner{padding:10px 13px}\n[data-radar-root="individual"] .nav{display:none}\n[data-radar-root="individual"] .brand span:last-child{display:none}\n[data-radar-root="individual"] .run-header{margin-top:14px;display:block}\n[data-radar-root="individual"] .run-tags{margin-top:12px}\n[data-radar-root="individual"] .context-grid{grid-template-columns:repeat(2,minmax(0,1fr))}\n[data-radar-root="individual"] .identity-card,[data-radar-root="individual"] .cycle-card,[data-radar-root="individual"] .income-card,[data-radar-root="individual"] .profile-card,[data-radar-root="individual"] .coverage-card,[data-radar-root="individual"] .window-card{grid-column:span 1}\n[data-radar-root="individual"] .section-head{align-items:start;flex-direction:column}\n[data-radar-root="individual"] .pair{grid-template-columns:1fr}\n[data-radar-root="individual"] .link{transform:rotate(90deg)}\n[data-radar-root="individual"] .metrics{display:none}}\n@media(max-width:480px){[data-radar-root="individual"] .context-grid{grid-template-columns:1fr}\n[data-radar-root="individual"] .identity-card,[data-radar-root="individual"] .cycle-card,[data-radar-root="individual"] .income-card,[data-radar-root="individual"] .profile-card,[data-radar-root="individual"] .coverage-card,[data-radar-root="individual"] .window-card{grid-column:span 1}\n[data-radar-root="individual"] .run-title{display:block}\n[data-radar-root="individual"] .run-title .eyebrow{margin-bottom:5px}\n[data-radar-root="individual"] .run-tags{gap:5px}\n[data-radar-root="individual"] .run-tag{font-size:9px;padding:5px 8px}\n[data-radar-root="individual"] .money-grid{grid-template-columns:1fr}\n[data-radar-root="individual"] .operator{transform:rotate(90deg);text-align:center}\n[data-radar-root="individual"] .mini-grid{grid-template-columns:1fr}\n[data-radar-root="individual"] .context-card{min-height:auto}\n[data-radar-root="individual"] .context-main{font-size:22px}\n[data-radar-root="individual"] .score-memory{display:grid;grid-template-columns:1fr 1fr}}\n@media(prefers-reduced-motion:reduce){[data-radar-root="individual"]{scroll-behavior:auto}\n[data-radar-root="individual"],[data-radar-root="individual"] *{transition:none!important;animation:none!important}}\n[data-radar-root="individual"] .context-card,[data-radar-root="individual"] .finance-card,[data-radar-root="individual"] .score-card-radar{padding:16px 17px}\n[data-radar-root="individual"] .card-static-head{display:block;min-height:auto;padding:0}\n[data-radar-root="individual"] .card-static-head .card-summary-main{width:100%}\n[data-radar-root="individual"] .context-card .card-body,[data-radar-root="individual"] .finance-card .card-body,[data-radar-root="individual"] .score-card-radar .card-body{padding:11px 0 0;margin-top:11px;border-top:1px solid #edf0f4}\n[data-radar-root="individual"] .context-card .context-labels{margin-top:0}\n[data-radar-root="individual"] .context-card .context-data{margin-top:10px}\n[data-radar-root="individual"] .context-card .context-foot{padding-top:0;margin-top:8px}\n[data-radar-root="individual"] .finance-card .card-labels{margin-top:0}\n[data-radar-root="individual"] .finance-card .meter{margin-top:4px}\n[data-radar-root="individual"] .score-card-radar .score-data{margin-top:0}\n[data-radar-root="individual"] .score-card-radar .score-parts{margin-top:10px}\n[data-radar-root="individual"] .score-card-radar.featured .card-static-head{background:transparent}\n[data-radar-root="individual"] .score-table-shell{background:#fff;border:1px solid var(--line);border-radius:20px;overflow:hidden;box-shadow:0 8px 26px rgba(27,36,74,.045)}\n[data-radar-root="individual"] .score-table-shell>summary{list-style:none;cursor:pointer;display:flex;align-items:center;justify-content:space-between;gap:16px;padding:16px 18px;background:#fff;font-weight:850}\n[data-radar-root="individual"] .score-table-shell>summary::-webkit-details-marker{display:none}\n[data-radar-root="individual"] .score-table-shell>summary:after{content:"+";display:grid;place-items:center;flex:0 0 28px;height:28px;border-radius:9px;background:#eef0ff;color:var(--bb-blue-deep);font-weight:950}\n[data-radar-root="individual"] .score-table-shell[open]>summary:after{content:"−"}\n[data-radar-root="individual"] .score-summary-main{display:flex;gap:9px;align-items:center;flex-wrap:wrap}\n[data-radar-root="individual"] .score-summary-main strong{font-size:13px}\n[data-radar-root="individual"] .score-summary-meta{font-size:10px;color:var(--muted);font-weight:650}\n[data-radar-root="individual"] .score-winner-tag{display:inline-flex;align-items:center;gap:5px;background:#fffbd9;border:1px solid #ebe47f;color:#655f00;border-radius:999px;padding:5px 8px;font-size:9px;font-weight:850}\n[data-radar-root="individual"] .score-table-body{border-top:1px solid var(--line);padding:14px}\n[data-radar-root="individual"] .score-table-wrap{overflow-x:auto;border:1px solid var(--line);border-radius:14px;background:#fff}\n[data-radar-root="individual"] .score-table{min-width:760px}\n[data-radar-root="individual"] .score-table th,[data-radar-root="individual"] .score-table td{padding:11px 12px}\n[data-radar-root="individual"] .score-table thead th{background:#f8f9fc;top:0;z-index:1}\n[data-radar-root="individual"] .score-table tbody tr:last-child td{border-bottom:0}\n[data-radar-root="individual"] .score-table .theme-cell{min-width:220px;font-weight:800}\n[data-radar-root="individual"] .score-table .theme-cell small{display:block;margin-top:3px;color:var(--muted);font-size:9px;font-weight:650}\n[data-radar-root="individual"] .score-table .num{text-align:right;white-space:nowrap;font-variant-numeric:tabular-nums}\n[data-radar-root="individual"] .score-table .final-cell{text-align:center;font-size:18px;font-weight:950;color:var(--bb-blue-deep)}\n[data-radar-root="individual"] .score-table .winner-row td{background:#fffef0}\n[data-radar-root="individual"] .score-table .winner-row td:first-child{box-shadow:inset 4px 0 0 var(--bb-yellow)}\n[data-radar-root="individual"] .score-inline-badge{display:inline-flex;margin-left:6px;background:#fff8c7;color:#645d00;border:1px solid #ebe47f;border-radius:999px;padding:3px 6px;font-size:8px;font-weight:900;vertical-align:middle;white-space:nowrap}\n[data-radar-root="individual"] .score-subhead{text-align:right!important}\n[data-radar-root="individual"] .score-subhead:first-child{text-align:left!important}\n@media(max-width:760px){[data-radar-root="individual"] .score-table-body{padding:10px}\n[data-radar-root="individual"] .score-table-shell>summary{padding:14px}\n[data-radar-root="individual"] .score-summary-meta{width:100%}}\n[data-radar-root="individual"] .topbar-inner{justify-content:space-between}\n[data-radar-root="individual"] .score-table-simple{min-width:0}\n[data-radar-root="individual"] .score-table-simple th:last-child,[data-radar-root="individual"] .score-table-simple td:last-child{width:140px;text-align:center}\n[data-radar-root="individual"] .score-table-simple .theme-cell{min-width:0}\n[data-radar-root="individual"] .score-table-simple .final-cell{font-size:20px}\n[data-radar-root="individual"] .score-table-static{padding:14px;overflow:hidden}\n[data-radar-root="individual"] .score-table-static .score-table-wrap{height:100%;margin:0}\n[data-radar-root="individual"] .score-table-complete{min-width:980px;width:100%}\n[data-radar-root="individual"] .score-table-complete th,[data-radar-root="individual"] .score-table-complete td{padding:12px 11px}\n[data-radar-root="individual"] .score-table-complete th{font-size:9px}\n[data-radar-root="individual"] .score-table-complete .theme-cell{min-width:230px}\n[data-radar-root="individual"] .score-table-complete .final-cell{font-size:18px;font-weight:900;text-align:center}\n[data-radar-root="individual"] .score-table-complete .winner-row td{background:#fffdf0}\n[data-radar-root="individual"] .score-table-complete .winner-row td:first-child{font-weight:900}\n[data-radar-root="individual"] .score-final-badge{display:inline-flex;align-items:center;justify-content:center;width:38px;height:38px;border-radius:999px;background:var(--bb-yellow);color:#173b67;font-weight:950;line-height:1;box-shadow:inset 0 0 0 1px #e7de58}\n@media(max-width:760px){[data-radar-root="individual"] .score-table-static{padding:10px}\n[data-radar-root="individual"] .score-table-complete{min-width:920px}}\n[data-radar-root="individual"] .top-actions{display:flex;align-items:center;gap:8px;white-space:nowrap}\n[data-radar-root="individual"] .export-btn{border:1px solid var(--bb-blue);background:var(--bb-blue);color:#fff;border-radius:999px;padding:8px 12px;font-weight:800;font-size:12px;cursor:pointer;white-space:nowrap}\n[data-radar-root="individual"] .export-btn:hover{background:var(--bb-blue-deep);border-color:var(--bb-blue-deep)}\n[data-radar-root="individual"] .export-btn:focus-visible,[data-radar-root="individual"] .privacy:focus-visible{outline:3px solid rgba(70,94,255,.22);outline-offset:2px}\n[data-radar-root="individual"] .export-toast{position:fixed;right:22px;bottom:22px;z-index:9999;max-width:360px;background:#171b2f;color:#fff;border-radius:12px;padding:11px 14px;box-shadow:0 12px 36px rgba(0,0,0,.22);font-size:11px;line-height:1.4;opacity:0;transform:translateY(8px);pointer-events:none;transition:.18s ease}\n[data-radar-root="individual"] .export-toast.show{opacity:1;transform:translateY(0)}\n@media(max-width:520px){[data-radar-root="individual"] .top-actions{gap:5px}\n[data-radar-root="individual"] .privacy,[data-radar-root="individual"] .export-btn{font-size:10px;padding:7px 9px}\n[data-radar-root="individual"] .export-toast{left:12px;right:12px;bottom:12px;max-width:none}}'
CSS_REFINADO_V1 = r"""
[data-radar-root="individual"]{
  --max:1320px;--radius:18px;--radius-sm:12px;
  --ink:#111827;--muted:#667085;--bg:#f5f7fb;--card:#fff;
  --line:#e6eaf1;--soft:#f8f9fc;--good:#087a55;--good-bg:#eaf8f1;
  --bb-blue:#4d5870;--bb-blue-deep:#252b3a;--bb-yellow:#e6eaf1;
  --danger:#b42318;--warn:#8c5a00;--warn-bg:#fff7e8;
  --shadow:0 12px 32px rgba(16,24,40,.055)
}
[data-radar-root="individual"]{
  background:radial-gradient(circle at 12% -10%,rgba(102,112,133,.06),transparent 32rem),linear-gradient(180deg,#fafbfe 0,#f5f7fb 28rem);
  color:var(--ink);letter-spacing:-.005em
}
[data-radar-root="individual"] .shell{max-width:var(--max);padding:30px 28px 80px}
[data-radar-root="individual"] .topbar{display:none!important}
[data-radar-root="individual"] .run-header{
  margin-top:0;display:grid;grid-template-columns:minmax(0,1fr) auto;
  grid-template-areas:"title actions" "tags tags";align-items:center;
  gap:20px 24px;padding:28px 30px 24px;border:1px solid rgba(223,228,238,.92);
  border-radius:24px;background:rgba(255,255,255,.88);
  box-shadow:0 20px 55px rgba(22,34,84,.07);backdrop-filter:blur(14px)
}
[data-radar-root="individual"] .run-title{grid-area:title;display:block;min-width:0;white-space:normal}
[data-radar-root="individual"] .run-title .eyebrow{margin:0;color:#7b8498;font-size:10px;letter-spacing:.14em}
[data-radar-root="individual"] .run-title h1{margin:5px 0 0;font-size:32px;line-height:1.02;letter-spacing:-.055em}
[data-radar-root="individual"] .top-actions{grid-area:actions;display:flex;gap:8px;align-items:center;justify-content:flex-end}
[data-radar-root="individual"] .privacy,[data-radar-root="individual"] .export-btn{
  min-height:38px;padding:9px 14px;border:1px solid #dfe4ed;border-radius:11px;
  background:#fff;color:var(--ink);font-size:11px;font-weight:800;
  box-shadow:0 1px 2px rgba(16,24,40,.03);cursor:pointer;white-space:nowrap
}
[data-radar-root="individual"] .privacy:hover{background:#f8f9fc;border-color:#cbd2df}
[data-radar-root="individual"] .export-btn{background:var(--ink);border-color:var(--ink);color:#fff}
[data-radar-root="individual"] .export-btn:hover{background:#252b3a;border-color:#252b3a}
[data-radar-root="individual"] .privacy:focus-visible,[data-radar-root="individual"] .export-btn:focus-visible,
[data-radar-root="individual"] .control-btn:focus-visible,[data-radar-root="individual"] .base-switch-btn:focus-visible{
  outline:3px solid rgba(102,112,133,.2);outline-offset:2px
}
[data-radar-root="individual"] .run-tags{
  grid-area:tags;justify-content:flex-start;gap:7px;max-width:none;
  padding-top:18px;border-top:1px solid #edf0f5
}
[data-radar-root="individual"] .run-tag{min-height:30px;padding:6px 10px;border-color:#e7eaf0;background:#fafbfc;color:#596276;font-size:9px}
[data-radar-root="individual"] .run-tag.base-active,[data-radar-root="individual"] .run-tag.ok,
[data-radar-root="individual"] .scenario-active-tag{background:#edf8f3;border-color:#d2eadf;color:#176848}
[data-radar-root="individual"] .section{margin-top:34px;scroll-margin-top:24px}
[data-radar-root="individual"] .section-head{margin-bottom:14px;align-items:flex-end}
[data-radar-root="individual"] .section-label{margin-bottom:3px;color:#98a2b3;font-size:9px;letter-spacing:.14em}
[data-radar-root="individual"] .section-head h2{font-size:20px;letter-spacing:-.035em}
[data-radar-root="individual"] .context-grid,[data-radar-root="individual"] .finance-grid{gap:14px}
[data-radar-root="individual"] .context-card,[data-radar-root="individual"] .finance-card{
  padding:19px 20px;border:1px solid #e6eaf1;border-radius:18px;
  background:rgba(255,255,255,.96);box-shadow:0 6px 18px rgba(16,24,40,.035)
}
[data-radar-root="individual"] .context-card:before,[data-radar-root="individual"] .finance-card:before{width:0!important}
[data-radar-root="individual"] .context-card:hover,[data-radar-root="individual"] .finance-card:hover{
  transform:none;border-color:#d8dee9;box-shadow:0 12px 28px rgba(16,24,40,.055)
}
[data-radar-root="individual"] .context-grid .identity-card{grid-column:span 4}
[data-radar-root="individual"] .context-grid .cycle-card{grid-column:span 5}
[data-radar-root="individual"] .context-grid .income-card{grid-column:span 3}
[data-radar-root="individual"] .context-grid .profile-card{grid-column:span 6}
[data-radar-root="individual"] .context-grid .coverage-card{grid-column:span 6}
[data-radar-root="individual"] .context-grid .identity-card,[data-radar-root="individual"] .context-grid .cycle-card,
[data-radar-root="individual"] .context-grid .income-card{min-height:184px}
[data-radar-root="individual"] .context-grid .profile-card,[data-radar-root="individual"] .context-grid .coverage-card{min-height:146px}
[data-radar-root="individual"] .context-kicker,[data-radar-root="individual"] .card-kicker{font-size:9px;letter-spacing:.11em;color:#7b8498}
[data-radar-root="individual"] .context-dot{width:8px;height:8px;background:#98a2b3;box-shadow:0 0 0 4px #f1f3f6}
[data-radar-root="individual"] .context-badge,[data-radar-root="individual"] .card-badge{
  padding:5px 8px;background:#f4f6fb;color:#4d5870;border:1px solid #eaedf3
}
[data-radar-root="individual"] .context-badge.good,[data-radar-root="individual"] .card-badge.good{background:var(--good-bg);color:var(--good);border-color:#d6eee3}
[data-radar-root="individual"] .card-summary-value{font-size:27px;letter-spacing:-.05em;margin:14px 0 15px}
[data-radar-root="individual"] .context-labels{gap:6px}
[data-radar-root="individual"] .micro-label{background:#f8f9fc;border-color:#eceff4;border-radius:9px;padding:6px 9px}
[data-radar-root="individual"] .data-cell{border-top-color:#edf0f4;padding-top:9px}
[data-radar-root="individual"] .base-switch{margin:12px 0 10px;padding:3px;border:0;background:#f1f3f7;border-radius:10px}
[data-radar-root="individual"] .base-switch-btn{min-height:34px;border-radius:8px}
[data-radar-root="individual"] .base-switch-btn.active{color:var(--ink);box-shadow:0 1px 5px rgba(16,24,40,.10)}
[data-radar-root="individual"] .finance-grid .finance-card{grid-column:span 3;min-height:166px}
[data-radar-root="individual"] .finance-grid .finance-balance{background:linear-gradient(180deg,#f9fdfb,#f3faf7)}
[data-radar-root="individual"] .meter{height:6px;background:#edf0f4;margin:13px 0 9px}
[data-radar-root="individual"] .meter span{background:#667085}
[data-radar-root="individual"] .explorer,[data-radar-root="individual"] .score-table-shell,
[data-radar-root="individual"] .recon-events{
  padding:14px;border:1px solid #e6eaf1;border-radius:18px;background:#fff;
  box-shadow:0 8px 24px rgba(16,24,40,.04)
}
[data-radar-root="individual"] .control-btn,[data-radar-root="individual"] .search{
  min-height:38px;border-radius:11px;border-color:#dfe4ed;background:#fff;padding:9px 12px
}
[data-radar-root="individual"] .search:focus{border-color:#98a2b3;box-shadow:0 0 0 3px rgba(102,112,133,.12)}
[data-radar-root="individual"] .level,[data-radar-root="individual"] .event{border-color:#e8ebf1;border-radius:12px;margin:7px 0}
[data-radar-root="individual"] .level summary,[data-radar-root="individual"] .event summary{min-height:50px;background:#fbfcfd;padding:12px 14px}
[data-radar-root="individual"] .level summary:before{background:#f0f2f7;color:#4d5870}
[data-radar-root="individual"] .nature>summary,[data-radar-root="individual"] .class>summary{background:#fbfcfd}
[data-radar-root="individual"] .table-wrap,[data-radar-root="individual"] .score-table-wrap{border-color:#e5e9f0;border-radius:13px}
[data-radar-root="individual"] th{background:#f8f9fb;color:#7b8498}
[data-radar-root="individual"] th,[data-radar-root="individual"] td{border-bottom-color:#edf0f4;padding:12px 13px}
[data-radar-root="individual"] .score-table{min-width:980px}
[data-radar-root="individual"] .score-table .final-cell{color:var(--ink)}
[data-radar-root="individual"] .score-table .winner-row td{background:#fafbfc}
[data-radar-root="individual"] .score-table .winner-row td:first-child{box-shadow:inset 4px 0 0 #98a2b3}
[data-radar-root="individual"] .score-final-badge{background:#eef0f4;color:var(--ink);box-shadow:inset 0 0 0 1px #d8dde6}
[data-radar-root="individual"] .recon{
  display:grid;grid-template-columns:minmax(300px,4fr) minmax(0,8fr);
  gap:14px;align-items:start;padding:0;border:0;border-radius:0;background:transparent;box-shadow:none
}
[data-radar-root="individual"] .recon>.window-card{grid-column:auto;min-height:178px;margin:0}
[data-radar-root="individual"] .funnel{gap:10px;padding:10px 0 4px;margin:0 0 8px}
[data-radar-root="individual"] .step{padding:11px 12px;border-radius:11px;background:#f7f8fb}
[data-radar-root="individual"] .step b{font-size:17px}
[data-radar-root="individual"] .event summary{display:block;font-size:11px}
[data-radar-root="individual"] .privacy-pending [data-private]{visibility:hidden}
[data-radar-root="individual"] [data-private],[data-radar-root="individual"] .masked-value{filter:none!important;user-select:auto!important}
[data-radar-root="individual"] .masked-value{font-variant-ligatures:none;letter-spacing:.04em}
@media(max-width:1050px){
  [data-radar-root="individual"] .shell{max-width:960px;padding-top:22px}
  [data-radar-root="individual"] .run-header{padding:24px}
  [data-radar-root="individual"] .context-grid .identity-card,[data-radar-root="individual"] .context-grid .cycle-card,
  [data-radar-root="individual"] .context-grid .income-card,[data-radar-root="individual"] .context-grid .profile-card,
  [data-radar-root="individual"] .context-grid .coverage-card{grid-column:span 6}
  [data-radar-root="individual"] .finance-grid .finance-card{grid-column:span 6}
  [data-radar-root="individual"] .recon{grid-template-columns:1fr}
}
@media(max-width:780px){
  [data-radar-root="individual"] .shell{padding:14px 13px 54px}
  [data-radar-root="individual"] .run-header{grid-template-columns:1fr;grid-template-areas:"title" "actions" "tags";gap:16px;padding:20px;border-radius:20px}
  [data-radar-root="individual"] .run-title h1{font-size:28px}
  [data-radar-root="individual"] .top-actions{justify-content:flex-start;width:100%}
  [data-radar-root="individual"] .privacy,[data-radar-root="individual"] .export-btn{flex:1}
  [data-radar-root="individual"] .run-tags{padding-top:15px}
  [data-radar-root="individual"] .section{margin-top:28px}
  [data-radar-root="individual"] .section-head{align-items:flex-start;flex-direction:column}
  [data-radar-root="individual"] .context-grid,[data-radar-root="individual"] .finance-grid{grid-template-columns:1fr}
  [data-radar-root="individual"] .context-grid .identity-card,[data-radar-root="individual"] .context-grid .cycle-card,
  [data-radar-root="individual"] .context-grid .income-card,[data-radar-root="individual"] .context-grid .profile-card,
  [data-radar-root="individual"] .context-grid .coverage-card,[data-radar-root="individual"] .finance-grid .finance-card{
    grid-column:1/-1;min-height:auto
  }
  [data-radar-root="individual"] .controls{width:100%}
  [data-radar-root="individual"] .search{width:100%;min-width:0}
}
@media(max-width:480px){
  [data-radar-root="individual"] .run-header{padding:18px 16px}
  [data-radar-root="individual"] .run-title h1{font-size:26px}
  [data-radar-root="individual"] .top-actions{display:grid;grid-template-columns:1fr 1fr}
  [data-radar-root="individual"] .privacy,[data-radar-root="individual"] .export-btn{width:100%}
  [data-radar-root="individual"] .context-data,[data-radar-root="individual"] .mini-grid{grid-template-columns:1fr}
  [data-radar-root="individual"] .card-summary-value{font-size:24px}
  [data-radar-root="individual"] .score-table{min-width:820px}
}
"""
CSS_DASHBOARD = re.sub(r'\s+', ' ', f'{CSS_APROVADO}{CSS_COMPLEMENTO}{CSS_REFINADO_V1}').strip().replace('filter:blur(5px);', 'filter:none;')

radar_root_id = f'radar-financeiro-individual-{CD_CLI}-{DATA_EXECUCAO.strftime("%Y%m%d")}'

html_dashboard = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="utf-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1"/>
<title>Radar Financeiro</title>
<style>{CSS_DASHBOARD}</style>

</head>
<body>
<section class="privacy-pending" data-radar-root="individual" id="{radar_root_id}" tabindex="-1">
<a class="skip" href="#{radar_root_id}">Ir para o conteúdo</a>

<main class="shell" data-role="content">
<section aria-label="Radar Financeiro" class="run-header"><div class="run-title"><div class="eyebrow">Análise do ciclo</div><h1>Radar Financeiro</h1></div>
<div class="top-actions"><button aria-pressed="false" class="privacy" data-role="privacy-button" type="button">Mostrar dados</button><button class="export-btn" data-role="export-button" type="button">Exportar HTML</button></div>
<div aria-label="Condições da execução" class="run-tags">
{tag_execucao('CPF único', res_dict.get('FL_CPF_UNICO'))}
{tag_execucao('Conta única', res_dict.get('FL_CONTA_ELEGIVEL_UNICA'))}
{tag_execucao('Somente BRL', res_dict.get('FL_SOMENTE_BRL'))}
{tag_execucao('Agro', res_dict.get('FL_TEM_MOV_AGRO'))}
<span data-radar-slot-html="tag_base_html">{payload_padrao['tag_base_html']}</span>
<span data-radar-slot-html="tag_pontuacao_html">{payload_padrao['tag_pontuacao_html']}</span>
</div></section>

<section class="section" data-section="contexto"><div class="section-head"><div><span class="section-label">Contexto</span><h2>Contexto</h2></div></div><div class="context-grid">
<article class="context-card identity-card"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="context-kicker">Cliente</span><span aria-hidden="true" class="context-dot"></span></div><div class="card-summary-value" data-private="identifier">{esc(res_dict.get('CD_CLI'))}</div></div></div>
<div class="card-body"><div class="context-labels"><span class="micro-label"><code>NR_AG_TITR</code> <b data-private="identifier">{esc(res_cta['NR_AG_TITR'])}</b></span><span class="micro-label"><code>CD_CT_TITR</code> <b data-private="identifier">{esc(res_cta['CD_CT_TITR'])}</b></span></div>
<div class="context-labels"><span class="micro-label"><code>CD_UOR_CC</code> <b data-private="identifier">{esc(cta_norm_row['CD_UOR_CC_NORM'])}</b></span><span class="micro-label"><code>NR_CC</code> <b data-private="identifier">{esc(cta_norm_row['NR_CC_NORM'])}</b></span></div></div></article>
<article class="context-card cycle-card"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="context-kicker">Ciclo</span><span class="context-badge">Dia {fmt_inteiro(res_dict.get('DD_INC_MM_CLC_BLC_FALLBACK'))}</span></div><div class="card-summary-value date-range">{fmt_data(res_dict.get('DT_REF_INI'), True)} <span>→</span> {fmt_data(res_dict.get('DT_REF_FIM'), True)}</div></div></div>
<div class="card-body"><div class="context-labels"><span class="micro-label">Início <b>{fmt_data(res_dict.get('DT_REF_INI'))}</b></span><span class="micro-label">Fim <b>{fmt_data(res_dict.get('DT_REF_FIM'))}</b></span><span class="micro-label">Fallback <b>{fallback_usado}</b></span></div><div class="context-data single"><div class="data-cell"><span>Referência</span><strong>{fmt_data(res_dict.get('TS_DD_INC_MM_CLC_BLC_REF'))}</strong></div></div></div></article>
<article class="context-card income-card" data-radar-component="base-financeira"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="context-kicker" data-radar-slot="base_kicker">{payload_padrao['base_kicker']}</span><span class="context-badge good">BRL</span></div><div aria-live="polite" class="card-summary-value" data-private="money" data-radar-slot="valor_base">{payload_padrao['valor_base']}</div></div></div><div aria-label="Selecionar base financeira" class="base-switch" role="group"><button aria-pressed="true" class="base-switch-btn active" data-radar-mode="RENDA_PRESUMIDA" type="button">Renda Presumida</button><button aria-pressed="false" class="base-switch-btn" data-radar-mode="ENTRADAS_REALIZADAS" type="button">Entradas Realizadas</button></div><div class="card-body"><div class="context-data single"><div class="data-cell"><span data-radar-slot="referencia_rotulo">{payload_padrao['referencia_rotulo']}</span><strong data-radar-slot="referencia_valor">{payload_padrao['referencia_valor']}</strong></div></div><div class="context-foot"><span class="scenario-active-tag" data-radar-slot="base_badge">{payload_padrao['base_badge']}</span></div></div></article>
<article class="context-card profile-card"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="context-kicker">Perfil</span><span class="context-badge subtle">Ref. {fmt_data(res_dict.get('DT_REF_PRFL'), True)}</span></div><div class="card-summary-value">{esc(res_dict.get('NM_MAC_PRFL_CLI'))}</div></div></div><div class="card-body"><div class="context-labels"><span class="micro-label">Microperfil <b>{esc(res_dict.get('NM_MIC_PRFL_CLI'))}</b></span></div><div class="context-foot">Referência: {fmt_data(res_dict.get('DT_REF_PRFL'))}</div></div></article>
<article class="context-card coverage-card"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="context-kicker">Base analisada</span><span class="context-badge good">{('Somente BRL' if res_dict.get('FL_SOMENTE_BRL') == 'S' else 'Moedas verificadas')}</span></div><div class="card-summary-value">{fmt_inteiro(res_dict.get('QT_TRANS_TOTAL'))} <small>transações</small></div></div></div><div class="card-body"><div class="context-labels"><span class="micro-label">Entradas <b>{fmt_inteiro(res_dict.get('QT_TRANS_ENT'))}</b></span><span class="micro-label">Saídas <b>{fmt_inteiro(res_dict.get('QT_TRANS_SAI'))}</b></span></div><div class="context-foot">Moeda utilizada nos cálculos: BRL</div></div></article>

</div></section>

<section class="section" data-section="resultado"><div class="section-head"><div><span class="section-label">Resultado</span><h2>Resumo financeiro</h2></div></div><div class="finance-grid">
<article class="finance-card finance-in"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="card-kicker" data-radar-slot="entrada_rotulo">{payload_padrao['entrada_rotulo']}</span><span class="card-badge">{fmt_inteiro(res_dict.get('QT_TRANS_ENT'))} transações</span></div><div class="card-summary-value" data-private="money" data-radar-slot="entrada_valor">{payload_padrao['entrada_valor']}</div></div></div><div class="card-body"><div class="card-labels"><span class="card-label">Base aplicada a toda a cadeia deste cenário</span></div></div></article>
<article class="finance-card finance-out"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="card-kicker">Saídas</span><span class="card-badge">{fmt_inteiro(res_dict.get('QT_TRANS_SAI'))} transações</span></div><div class="card-summary-value" data-private="money">{fmt_moeda(res_dict.get('VL_TRANS_SAI'))}</div></div></div><div class="card-body"><div class="card-labels"><span class="card-label">Consideradas <b>{fmt_inteiro(res_dict.get('QT_TRANS_SAI'))}</b></span></div></div></article>
<article class="finance-card finance-balance"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="card-kicker">Saldo do ciclo</span><span class="card-badge good" data-radar-slot="saldo_status">{payload_padrao['saldo_status']}</span></div><div class="card-summary-value good" data-private="money" data-radar-slot="saldo_valor">{payload_padrao['saldo_valor']}</div></div></div><div class="card-body"><div class="card-labels"><span class="card-label">Situação <b>pré-calculada pelo motor</b></span></div></div></article>
<article class="finance-card finance-ratio"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="card-kicker">Saídas / Base</span></div><div class="card-summary-value" data-radar-slot="razao_valor">{payload_padrao['razao_valor']}</div></div></div><div class="card-body"><div aria-label="Relação Saídas sobre Base Financeira" class="meter"><span data-radar-meter="true" style="width:{payload_padrao['largura_medidor']}"></span></div><div class="card-labels"><span class="card-label">Relação do ciclo</span></div></div></article>
</div></section>

<section class="section" data-section="composicao"><div class="section-head"><div><span class="section-label">Explicabilidade</span><h2>Composição</h2></div><div class="controls"><input aria-label="Buscar na composição" class="search" data-role="composition-search" placeholder="Buscar classe ou categoria" type="search"/><button class="control-btn" data-action="expand-composition" type="button">Expandir tudo</button><button class="control-btn" data-action="collapse-composition" type="button">Recolher</button></div></div><div class="explorer" data-role="composition-explorer">{render_composicao()}</div></section>

<section class="section" data-section="pontuacao"><div class="section-head"><div><span class="section-label">Motor</span><h2>Pontuação</h2></div><span class="scenario-active-tag" data-radar-slot="base_badge">{payload_padrao['base_badge']}</span></div><div aria-label="Tabela completa de pontuação" class="score-table-shell score-table-static"><div class="score-table-wrap"><table class="score-table score-table-complete"><thead><tr><th>Tema</th><th class="num">Valor</th><th class="num">% base</th><th class="num">Referência</th><th class="num">Conc.</th><th class="num">Orç.</th><th class="num">Perfil</th><th class="score-subhead">Final</th></tr></thead><tbody data-radar-slot-html="pontuacao_html">{payload_padrao['pontuacao_html']}</tbody></table></div></div></section>


<section class="section" data-section="reconciliacao"><div class="section-head"><div><span class="section-label">Controle</span><h2>Reconciliação</h2></div><div class="controls"><button class="control-btn" data-action="expand-reconciliation" type="button">Expandir eventos</button><button class="control-btn" data-action="collapse-reconciliation" type="button">Recolher</button></div></div><div class="recon"><article class="context-card window-card"><div class="card-static-head"><div class="card-summary-main"><div class="card-summary-row"><span class="context-kicker">Janela de contexto</span><span class="context-badge">±{DIAS_CONTEXTO_RECONCILIACAO} dias</span></div><div class="card-summary-value date-range">{fmt_data(dt_contexto_ini, True)} <span>→</span> {fmt_data(dt_contexto_fim, True)}</div></div></div><div class="card-body"><div class="context-data"><div class="data-cell"><span>Ciclo oficial</span><strong>{fmt_data(dt_ini_j, True)} → {fmt_data(dt_fim_j, True)}</strong></div><div class="data-cell"><span>Contexto</span><strong>{fmt_data(dt_contexto_ini, True)} → {fmt_data(dt_contexto_fim, True)}</strong></div></div></div></article><div class="recon-events"><div class="funnel"><div class="step"><b>{qt_raw}</b>oficiais</div><div>→</div><div class="step minus"><b>−{2 * qt_pares_exatos_oficiais}</b>pares exatos removidos</div><div>→</div><div class="step minus"><b>−{qt_pares_borda}</b>borda removida</div><div>→</div><div class="step"><b>{qt_efetivo}</b>efetivas</div></div>{render_eventos_reconciliacao()}{render_contexto_externo()}</div></div></section>

</main>
<div aria-live="polite" class="export-toast" data-role="export-toast" role="status">HTML já salvo no workdir: radar_financeiro_{res_dict.get('CD_CLI')}_{DATA_EXECUCAO.strftime('%Y%m%d')}.html</div>
<script>
(function(){{
  const root=document.getElementById('{radar_root_id}');
  if(!root||root.dataset.radarBound==='1')return;
  root.dataset.radarBound='1';
  const one=selector=>root.querySelector(selector);
  const all=selector=>root.querySelectorAll(selector);
  function bind(element,eventName,handler){{
    if(!element||element.dataset.radarEventBound==='1')return;
    element.dataset.radarEventBound='1';
    element.addEventListener(eventName,handler);
  }}
  const scenarioPayloads={payload_cenarios_json};
  Object.values(scenarioPayloads).forEach(payload=>Object.freeze(payload));
  Object.freeze(scenarioPayloads);
  let dataVisible=false;
  let privacyReady=false;
  let privacyTargets=[];
  const originalValues=new WeakMap();
  const privacyButton=one('[data-role="privacy-button"]');
  function getPrivateTargets(scope=root){{
    if(!scope)return[];
    return Array.from(scope.querySelectorAll('[data-private]'))
      .filter(element=>!element.closest('[data-section="reconciliacao"]'));
  }}
  function capturePrivateTargets(forceScenarioSlots=false){{
    privacyTargets=getPrivateTargets(root);
    privacyTargets.forEach(element=>{{
      const scenarioSlot=element.matches('[data-radar-slot][data-private]');
      if(!originalValues.has(element)||(forceScenarioSlots&&scenarioSlot)){{
        originalValues.set(element,element.innerHTML);
      }}
    }});
  }}
  function renderPrivacy(){{
    privacyTargets.forEach(element=>{{
      const original=originalValues.get(element);
      if(original===undefined)return;
      element.innerHTML=dataVisible?original:'****';
      element.classList.toggle('masked-value',!dataVisible);
    }});
    root.classList.remove('privacy-pending');
    privacyButton.textContent=dataVisible?'Ocultar dados':'Mostrar dados';
    privacyButton.setAttribute('aria-pressed',String(dataVisible));
  }}
  function applyScenario(mode){{
    const payload=scenarioPayloads[mode];
    if(!payload)return;
    all('[data-radar-slot]').forEach(element=>{{
      const name=element.dataset.radarSlot;
      element.textContent=String(payload[name]??'—');
    }});
    all('[data-radar-slot-html]').forEach(element=>{{
      const name=element.dataset.radarSlotHtml;
      element.innerHTML=payload[name]||'';
    }});
    const meter=one('[data-radar-meter]');
    if(meter)meter.style.width=payload.largura_medidor;
    root.dataset.radarScenario=mode;
    all('[data-radar-mode]').forEach(button=>{{
      const active=button.dataset.radarMode===mode;
      button.classList.toggle('active',active);
      button.setAttribute('aria-pressed',String(active));
    }});
    if(privacyReady){{capturePrivateTargets(true);renderPrivacy();}}
  }}
  all('[data-radar-mode]').forEach(button=>bind(button,'click',()=>applyScenario(button.dataset.radarMode)));
  applyScenario('RENDA_PRESUMIDA');
  capturePrivateTargets();
  privacyReady=true;
  renderPrivacy();
  bind(privacyButton,'click',()=>{{dataVisible=!dataVisible;renderPrivacy();}});
  all('details').forEach(d=>d.open=false);
  function setOpen(container,open){{if(container)container.querySelectorAll('details').forEach(d=>d.open=open);}}
  const composition=one('[data-role="composition-explorer"]');
  const reconciliation=one('[data-section="reconciliacao"]');
  bind(one('[data-action="expand-composition"]'),'click',()=>setOpen(composition,true));
  bind(one('[data-action="collapse-composition"]'),'click',()=>setOpen(composition,false));
  bind(one('[data-action="expand-reconciliation"]'),'click',()=>setOpen(reconciliation,true));
  bind(one('[data-action="collapse-reconciliation"]'),'click',()=>setOpen(reconciliation,false));
  const compSearch=one('[data-role="composition-search"]');
  bind(compSearch,'input',()=>{{const q=compSearch.value.trim().toLowerCase();composition.querySelectorAll('details.category').forEach(d=>{{const hit=!q||d.textContent.toLowerCase().includes(q);d.style.display=hit?'':'none';if(q&&hit){{d.open=true;let p=d.parentElement.closest('details');while(p&&root.contains(p)){{p.open=true;p=p.parentElement.closest('details');}}}}}});}});
  const exportToast=one('[data-role="export-toast"]');
  function showExportToast(message){{if(!exportToast)return;exportToast.textContent=message;exportToast.classList.add('show');window.clearTimeout(showExportToast._timer);showExportToast._timer=window.setTimeout(()=>exportToast.classList.remove('show'),5000);}}
  const filename='radar_financeiro_{res_dict.get('CD_CLI')}_{DATA_EXECUCAO.strftime('%Y%m%d')}.html';
  const exportButton=one('[data-role="export-button"]');
  bind(exportButton,'click',()=>{{
    if(root.parentElement===document.body){{
      const clone=document.documentElement.cloneNode(true);
      const cloneRoot=clone.querySelector('[data-radar-root="individual"]');
      const cloneTargets=getPrivateTargets(cloneRoot);
      cloneTargets.forEach((element,index)=>{{
        const original=originalValues.get(privacyTargets[index]);
        if(original!==undefined){{element.innerHTML=original;element.classList.remove('masked-value');}}
      }});
      cloneRoot.classList.add('privacy-pending');
      const clonePrivacy=cloneRoot.querySelector('[data-role="privacy-button"]');
      if(clonePrivacy){{clonePrivacy.textContent='Mostrar dados';clonePrivacy.setAttribute('aria-pressed','false');}}
      clone.querySelectorAll('[data-radar-bound],[data-radar-event-bound]').forEach(element=>{{
        element.removeAttribute('data-radar-bound');
        element.removeAttribute('data-radar-event-bound');
      }});
      const source='<!DOCTYPE html>\\n'+clone.outerHTML;
      const url=URL.createObjectURL(new Blob([source],{{type:'text/html;charset=utf-8'}}));
      const link=document.createElement('a');link.href=url;link.download=filename;link.click();
      window.setTimeout(()=>URL.revokeObjectURL(url),1000);
      showExportToast('Cópia exportada: '+filename);
    }}else{{showExportToast('HTML salvo no workdir: '+filename);}}
  }});
}})();
</script>
</section>
</body></html>"""

html_dashboard = re.sub(r'>\s+<', '><', html_dashboard).strip()
html_bytes = html_dashboard.encode('utf-8')
tamanho_html = len(html_bytes)
if tamanho_html > LIMITE_PAYLOAD_BYTES:
    raise RuntimeError(f'Payload HTML de {tamanho_html} bytes excede o limite de 2 MiB.')

metadados_dashboard = {
    'tamanho_bytes': tamanho_html,
    'cd_cli': CD_CLI,
    'data_execucao': DATA_EXECUCAO.strftime('%Y%m%d'),
    'filename': f'radar_financeiro_{CD_CLI}_{DATA_EXECUCAO.strftime("%Y%m%d")}.html',
    'root_id': radar_root_id,
}
print(f'[RADAR_INDIVIDUAL] HTML gerado: {tamanho_html} bytes.')

print('[RADAR_INDIVIDUAL] Tabela final oficial (80 atributos):')
spark.table(VIEW_RESULTADO).show(truncate=False, vertical=True)

if 'df_q5_contexto_apresentacao' in globals() and df_q5_contexto_apresentacao.is_cached:
    df_q5_contexto_apresentacao.unpersist(blocking=False)
df_res_80.unpersist(blocking=False)

tempo_total = time.perf_counter() - inicio_execucao
print(f'[RADAR_INDIVIDUAL] Processamento concluído em {tempo_total:.3f}s.')


## Dashboard


In [ ]:
from IPython.display import HTML, Javascript, display
from pathlib import Path
import re

html_dashboard_local = spark.get_from_spark("html_dashboard")
metadados_local = spark.get_from_spark("metadados_dashboard")

destino_html = Path.cwd().resolve() / metadados_local['filename']
destino_html.write_text(html_dashboard_local, encoding='utf-8')

print(
    f"[RADAR_INDIVIDUAL] HTML salvo: {destino_html} "
    f"({metadados_local['tamanho_bytes']} bytes)."
)
display(HTML(html_dashboard_local))
for script_dashboard in re.findall(r'<script>(.*?)</script>', html_dashboard_local, flags=re.S | re.I):
    display(Javascript(script_dashboard))
